In [ ]:
# =============================================================================
# MODEL 3B - STEP 1
# CLEAN 4-CLASS CT PROJECT INITIALIZATION
# =============================================================================

from pathlib import Path
import os
import shutil
import json
import random
import numpy as np

print("=" * 80)
print("MODEL 3B - 4 CLASS CT")
print("PROJECT INITIALIZATION")
print("=" * 80)

# -----------------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# -----------------------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")

# -----------------------------------------------------------------------------
# 2. PROJECT DIRECTORIES
# -----------------------------------------------------------------------------

ROOT = Path("/content/model3b")

RAW_DIR = ROOT / "raw"
PROCESSED_DIR = ROOT / "processed"
MANIFEST_DIR = ROOT / "manifests"
AUDIT_DIR = ROOT / "audit"
MODEL_DIR = ROOT / "model"
RESULTS_DIR = ROOT / "results"

for d in [
    ROOT,
    RAW_DIR,
    PROCESSED_DIR,
    MANIFEST_DIR,
    AUDIT_DIR,
    MODEL_DIR,
    RESULTS_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

# Persistent Drive location
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

DRIVE_PROCESSED = DRIVE_ROOT / "processed"
DRIVE_MANIFESTS = DRIVE_ROOT / "manifests"
DRIVE_MODEL = DRIVE_ROOT / "model"
DRIVE_RESULTS = DRIVE_ROOT / "results"

for d in [
    DRIVE_ROOT,
    DRIVE_PROCESSED,
    DRIVE_MANIFESTS,
    DRIVE_MODEL,
    DRIVE_RESULTS
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

# -----------------------------------------------------------------------------
# 3. FIXED CLASS DEFINITION
# -----------------------------------------------------------------------------

CLASSES = {
    0: "Meningioma",
    1: "Pituitary",
    2: "Brain_Metastasis",
    3: "Schwannoma"
}

CLASS_TO_ID = {
    v: k for k, v in CLASSES.items()
}

# -----------------------------------------------------------------------------
# 4. REPRODUCIBILITY
# -----------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# -----------------------------------------------------------------------------
# 5. PROJECT CONFIG
# -----------------------------------------------------------------------------

CONFIG = {
    "project": "Model 3B",
    "version": "4-class-CT",
    "modality": "CT",
    "classes": CLASSES,
    "seed": SEED,
    "target_image_size": [224, 224],
    "target_spacing_mm": [1.0, 1.0, 1.0],
    "segmentation": False,
    "augmentation": "training_only",
    "patient_level_split": True,
    "drive_limit_gb": 15
}

with open(
    MANIFEST_DIR / "model3b_config.json",
    "w"
) as f:
    json.dump(
        CONFIG,
        f,
        indent=2
    )

# -----------------------------------------------------------------------------
# 6. DISPLAY
# -----------------------------------------------------------------------------

print("\nProject root:")
print(ROOT)

print("\nDrive root:")
print(DRIVE_ROOT)

print("\nClasses:")

for k, v in CLASSES.items():
    print(f"  {k} -> {v}")

print("\nDirectory structure:")

for d in [
    RAW_DIR,
    PROCESSED_DIR,
    MANIFEST_DIR,
    AUDIT_DIR,
    MODEL_DIR,
    RESULTS_DIR
]:
    print(" ", d)

print("\n" + "=" * 80)
print("STEP 1 COMPLETE")
print("=" * 80)

MODEL 3B - 4 CLASS CT
PROJECT INITIALIZATION
Mounted at /content/drive

Project root:
/content/model3b

Drive root:
/content/drive/MyDrive/Model3B

Classes:
  0 -> Meningioma
  1 -> Pituitary
  2 -> Brain_Metastasis
  3 -> Schwannoma

Directory structure:
  /content/model3b/raw
  /content/model3b/processed
  /content/model3b/manifests
  /content/model3b/audit
  /content/model3b/model
  /content/model3b/results

STEP 1 COMPLETE


In [ ]:
# =============================================================================
# MODEL 3B - STEP 2
# CT PROCESSING ENVIRONMENT
# =============================================================================

!pip -q install nibabel SimpleITK pandas numpy scipy scikit-image \
    scikit-learn matplotlib seaborn tqdm

import nibabel as nib
import SimpleITK as sitk

print("=" * 80)
print("MODEL 3B - CT ENVIRONMENT READY")
print("=" * 80)

print("Nibabel:", nib.__version__)
print("SimpleITK:", sitk.Version_VersionString())

print("\nRequired libraries installed.")
print("=" * 80)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 24.4 MB/s eta 0:00:00
MODEL 3B - CT ENVIRONMENT READY
Nibabel: 5.4.2
SimpleITK: 2.5.6

Required libraries installed.


In [ ]:
# =============================================================================
# MODEL 3B - STEP 3
# SELECTIVE CT ACQUISITION
# MENINGIOMA + PITUITARY
# =============================================================================

import os
import re
import io
import json
import struct
import hashlib
import requests
import zipfile
import zlib
from pathlib import Path
from tqdm.auto import tqdm

# =============================================================================
# CONFIG
# =============================================================================

SOURCES = {
    "Meningioma": {
        "url":
        "https://zenodo.org/api/records/17486320/files/meningioma.zip/content",

        "output":
        RAW_DIR / "Meningioma"
    },

    "Pituitary": {
        "url":
        "https://zenodo.org/api/records/17486320/files/pituitary-tumor.zip/content",

        "output":
        RAW_DIR / "Pituitary"
    }
}

for source in SOURCES.values():
    source["output"].mkdir(
        parents=True,
        exist_ok=True
    )


# =============================================================================
# HTTP RANGE HELPERS
# =============================================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Model3B-CT-Acquisition/1.0"
})


def get_range(url, start, end):

    r = session.get(
        url,
        headers={
            "Range": f"bytes={start}-{end}"
        },
        timeout=120
    )

    r.raise_for_status()

    if r.status_code != 206:
        raise RuntimeError(
            f"Server did not return HTTP 206: {r.status_code}"
        )

    return r.content


def get_file_size(url):

    r = session.get(
        url,
        headers={
            "Range": "bytes=0-0"
        },
        timeout=60
    )

    r.raise_for_status()

    cr = r.headers.get(
        "Content-Range",
        ""
    )

    m = re.search(
        r"/(\d+)$",
        cr
    )

    if not m:
        raise RuntimeError(
            f"Could not determine remote file size.\n"
            f"Content-Range: {cr}"
        )

    return int(m.group(1))


# =============================================================================
# ZIP CENTRAL DIRECTORY
# =============================================================================

def find_signature(data, signature):

    return data.rfind(signature)


def read_zip_directory(url):

    size = get_file_size(url)

    print(
        f"Archive size: "
        f"{size / 1024**3:.3f} GB"
    )

    tail_size = min(
        8 * 1024 * 1024,
        size
    )

    tail_start = size - tail_size

    tail = get_range(
        url,
        tail_start,
        size - 1
    )

    # -------------------------------------------------------------------------
    # EOCD
    # -------------------------------------------------------------------------

    eocd_sig = b"PK\x05\x06"

    eocd_pos = tail.rfind(
        eocd_sig
    )

    if eocd_pos < 0:

        raise RuntimeError(
            "EOCD not found."
        )

    eocd = tail[
        eocd_pos:eocd_pos + 22
    ]

    (
        signature,
        disk,
        cd_disk,
        disk_entries,
        total_entries,
        cd_size_32,
        cd_offset_32,
        comment_len
    ) = struct.unpack(
        "<4sHHHHIIH",
        eocd
    )

    eocd_absolute = (
        tail_start +
        eocd_pos
    )

    # -------------------------------------------------------------------------
    # ZIP64
    # -------------------------------------------------------------------------

    if (
        total_entries == 0xFFFF
        or
        cd_size_32 == 0xFFFFFFFF
        or
        cd_offset_32 == 0xFFFFFFFF
    ):

        zip64_locator_sig = b"PK\x06\x07"

        locator_pos = tail.rfind(
            zip64_locator_sig,
            0,
            eocd_pos
        )

        if locator_pos < 0:

            raise RuntimeError(
                "ZIP64 locator not found."
            )

        locator = tail[
            locator_pos:
            locator_pos + 20
        ]

        (
            _,
            zip64_disk,
            zip64_offset,
            total_disks
        ) = struct.unpack(
            "<4sIQI",
            locator
        )

        zip64_abs = zip64_offset

        zip64_record = get_range(
            url,
            zip64_abs,
            zip64_abs + 55
        )

        (
            sig,
            record_size,
            version_made,
            version_needed,
            disk_number,
            cd_disk_number,
            entries_disk,
            total_entries,
            cd_size,
            cd_offset
        ) = struct.unpack(
            "<4sQHHIIQQQQ",
            zip64_record[:56]
        )

    else:

        total_entries = total_entries
        cd_size = cd_size_32
        cd_offset = cd_offset_32

    print(
        f"Total entries: {total_entries}"
    )

    print(
        f"Central directory: "
        f"{cd_size / 1024**2:.3f} MB"
    )

    print(
        f"Central directory offset: "
        f"{cd_offset}"
    )

    # -------------------------------------------------------------------------
    # DOWNLOAD CENTRAL DIRECTORY ONLY
    # -------------------------------------------------------------------------

    cd = get_range(
        url,
        cd_offset,
        cd_offset + cd_size - 1
    )

    print(
        f"Central directory downloaded: "
        f"{len(cd) / 1024**2:.3f} MB"
    )

    return size, cd


# =============================================================================
# CENTRAL DIRECTORY PARSER
# =============================================================================

def parse_central_directory(cd):

    entries = []

    pos = 0

    signature = b"PK\x01\x02"

    while pos < len(cd):

        idx = cd.find(
            signature,
            pos
        )

        if idx < 0:
            break

        if idx + 46 > len(cd):
            break

        fixed = cd[
            idx:
            idx + 46
        ]

        fields = struct.unpack(
            "<4s6H3I5H2I",
            fixed
        )

        (
            sig,
            version_made,
            version_needed,
            flag,
            compression,
            mod_time,
            mod_date,
            crc32,
            compressed_size,
            uncompressed_size,
            filename_len,
            extra_len,
            comment_len,
            disk_start,
            internal_attr,
            external_attr,
            local_offset
        ) = fields

        name_start = idx + 46

        name_end = (
            name_start +
            filename_len
        )

        extra_end = (
            name_end +
            extra_len
        )

        name = cd[
            name_start:name_end
        ].decode(
            "utf-8",
            errors="replace"
        )

        extra = cd[
            name_end:extra_end
        ]

        # ---------------------------------------------------------------------
        # ZIP64 EXTRA FIELD
        # ---------------------------------------------------------------------

        if (
            compressed_size == 0xFFFFFFFF
            or
            uncompressed_size == 0xFFFFFFFF
            or
            local_offset == 0xFFFFFFFF
        ):

            ep = 0

            while ep + 4 <= len(extra):

                header_id, data_size = struct.unpack(
                    "<HH",
                    extra[ep:ep + 4]
                )

                data = extra[
                    ep + 4:
                    ep + 4 + data_size
                ]

                if header_id == 0x0001:

                    dp = 0

                    if uncompressed_size == 0xFFFFFFFF:

                        uncompressed_size = struct.unpack(
                            "<Q",
                            data[dp:dp + 8]
                        )[0]

                        dp += 8

                    if compressed_size == 0xFFFFFFFF:

                        compressed_size = struct.unpack(
                            "<Q",
                            data[dp:dp + 8]
                        )[0]

                        dp += 8

                    if local_offset == 0xFFFFFFFF:

                        local_offset = struct.unpack(
                            "<Q",
                            data[dp:dp + 8]
                        )[0]

                        dp += 8

                    break

                ep += 4 + data_size

        entries.append({
            "name": name,
            "compression": compression,
            "compressed_size": compressed_size,
            "uncompressed_size": uncompressed_size,
            "local_offset": local_offset,
            "crc32": crc32,
            "flag": flag
        })

        pos = (
            idx +
            46 +
            filename_len +
            extra_len +
            comment_len
        )

    return entries


# =============================================================================
# SELECT CT-BRAIN FILES
# =============================================================================

def select_ct(entries, class_name):

    selected = []

    for e in entries:

        name = e["name"]

        normalized = name.lower()

        if not normalized.endswith(
            ".nii.gz"
        ):
            continue

        if "ct-brain.nii.gz" not in normalized:
            continue

        # Exclude CT-BONE and unrelated paths
        if "ct-bone" in normalized:
            continue

        if "mri" in normalized:
            continue

        # Patient ID
        m = re.search(
            r"(sub-\d+)",
            name,
            re.IGNORECASE
        )

        if not m:
            continue

        patient = m.group(1).lower()

        selected.append({
            **e,
            "patient_id": patient,
            "class_name": class_name,
            "class_id": CLASS_TO_ID[class_name],
            "archive_path": name
        })

    # Remove duplicate patients
    unique = {}

    for e in selected:

        patient = e["patient_id"]

        if patient not in unique:

            unique[patient] = e

    return list(
        unique.values()
    )


# =============================================================================
# EXTRACT ONE ZIP MEMBER WITHOUT DOWNLOADING THE WHOLE ZIP
# =============================================================================

def extract_member(
    url,
    entry,
    output_path
):

    local_offset = entry[
        "local_offset"
    ]

    # Local ZIP header
    local_header = get_range(
        url,
        local_offset,
        local_offset + 30 - 1
    )

    (
        sig,
        version,
        flag,
        compression,
        mod_time,
        mod_date,
        crc32,
        compressed_size,
        uncompressed_size,
        filename_len,
        extra_len
    ) = struct.unpack(
        "<4s5H3I2H",
        local_header
    )

    if sig != b"PK\x03\x04":

        raise RuntimeError(
            "Invalid local ZIP header."
        )

    data_start = (
        local_offset +
        30 +
        filename_len +
        extra_len
    )

    data_end = (
        data_start +
        entry["compressed_size"] -
        1
    )

    # -------------------------------------------------------------------------
    # DOWNLOAD ONLY COMPRESSED MEMBER
    # -------------------------------------------------------------------------

    compressed = get_range(
        url,
        data_start,
        data_end
    )

    # -------------------------------------------------------------------------
    # DECOMPRESS ZIP MEMBER
    # -------------------------------------------------------------------------

    if entry["compression"] == 0:

        raw = compressed

    elif entry["compression"] == 8:

        raw = zlib.decompress(
            compressed,
            -15
        )

    else:

        raise RuntimeError(
            f"Unsupported ZIP compression: "
            f"{entry['compression']}"
        )

    # -------------------------------------------------------------------------
    # VALIDATION
    # -------------------------------------------------------------------------

    if len(raw) != entry[
        "uncompressed_size"
    ]:

        raise RuntimeError(
            f"Size mismatch: "
            f"{len(raw)} != "
            f"{entry['uncompressed_size']}"
        )

    actual_crc = (
        zlib.crc32(raw) &
        0xffffffff
    )

    if actual_crc != entry["crc32"]:

        raise RuntimeError(
            "CRC32 mismatch."
        )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_path,
        "wb"
    ) as f:

        f.write(raw)

    # SHA256
    sha = hashlib.sha256(
        raw
    ).hexdigest()

    return sha


# =============================================================================
# ACQUIRE SOURCE
# =============================================================================

def acquire_source(
    class_name,
    source
):

    print("\n")
    print("=" * 80)
    print(
        f"ACQUIRING {class_name.upper()}"
    )
    print("=" * 80)

    url = source["url"]

    output_root = source["output"]

    archive_size, cd = read_zip_directory(
        url
    )

    entries = parse_central_directory(
        cd
    )

    print(
        f"Parsed ZIP entries: "
        f"{len(entries)}"
    )

    selected = select_ct(
        entries,
        class_name
    )

    print(
        f"\nCT-BRAIN volumes: "
        f"{len(selected)}"
    )

    patients = sorted(
        {
            e["patient_id"]
            for e in selected
        }
    )

    print(
        f"Unique patients: "
        f"{len(patients)}"
    )

    if not selected:

        raise RuntimeError(
            f"No CT-BRAIN files found "
            f"for {class_name}"
        )

    manifest_rows = []

    total_mb = sum(
        e["compressed_size"]
        for e in selected
    ) / 1024**2

    print(
        f"Compressed acquisition: "
        f"{total_mb:.2f} MB"
    )

    # -------------------------------------------------------------------------
    # EXTRACTION
    # -------------------------------------------------------------------------

    for i, entry in enumerate(
        selected,
        1
    ):

        patient = entry[
            "patient_id"
        ]

        output_path = (
            output_root /
            patient /
            "CT-BRAIN.nii.gz"
        )

        expected = entry[
            "uncompressed_size"
        ]

        # Resume support
        if (
            output_path.exists()
            and
            output_path.stat().st_size
            == expected
        ):

            print(
                f"[{i}/{len(selected)}] "
                f"{patient} "
                f"→ already complete"
            )

            with open(
                output_path,
                "rb"
            ) as f:

                sha = hashlib.sha256(
                    f.read()
                ).hexdigest()

        else:

            print(
                f"[{i}/{len(selected)}] "
                f"{patient} "
                f"→ {expected / 1024**2:.2f} MB"
            )

            sha = extract_member(
                url,
                entry,
                output_path
            )

            print(
                "    ✅ verified"
            )

        manifest_rows.append({
            "class_id":
                entry["class_id"],

            "class_name":
                class_name,

            "patient_id":
                patient,

            "archive_path":
                entry["archive_path"],

            "compressed_size":
                entry["compressed_size"],

            "uncompressed_size":
                entry["uncompressed_size"],

            "compression_method":
                entry["compression"],

            "local_header_offset":
                entry["local_offset"],

            "sha256":
                sha,

            "local_path":
                str(output_path)
        })

    # -------------------------------------------------------------------------
    # MANIFEST
    # -------------------------------------------------------------------------

    manifest = pd.DataFrame(
        manifest_rows
    )

    manifest_path = (
        MANIFEST_DIR /
        f"{class_name.lower()}_acquisition_manifest.csv"
    )

    manifest.to_csv(
        manifest_path,
        index=False
    )

    print(
        "\nManifest:"
    )

    print(
        manifest_path
    )

    print(
        f"\n✅ {class_name} acquisition complete"
    )

    return manifest


# =============================================================================
# RUN
# =============================================================================

all_manifests = []

for class_name, source in SOURCES.items():

    m = acquire_source(
        class_name,
        source
    )

    all_manifests.append(
        m
    )


# =============================================================================
# COMBINED MANIFEST
# =============================================================================

combined = pd.concat(
    all_manifests,
    ignore_index=True
)

combined_path = (
    MANIFEST_DIR /
    "model3b_step1_ct_acquisition_manifest.csv"
)

combined.to_csv(
    combined_path,
    index=False
)

print("\n")
print("=" * 80)
print("MODEL 3B - STEP 3 COMPLETE")
print("=" * 80)

print(
    "\nTotal CT volumes:",
    len(combined)
)

print(
    "Unique patients:",
    combined["patient_id"].nunique()
)

print("\nBy class:")

print(
    combined.groupby(
        "class_name"
    )["patient_id"]
    .nunique()
)

print(
    "\nCombined manifest:"
)

print(
    combined_path
)

print(
    "\nRaw CT remains in:"
)

print(
    RAW_DIR
)

print("=" * 80)



ACQUIRING MENINGIOMA
Archive size: 4.208 GB
Total entries: 204
Central directory: 0.020 MB
Central directory offset: 4517966148
Central directory downloaded: 0.020 MB
Parsed ZIP entries: 204

CT-BRAIN volumes: 20
Unique patients: 20
Compressed acquisition: 653.45 MB
[1/20] sub-13 → 26.56 MB
    ✅ verified
[2/20] sub-14 → 29.07 MB
    ✅ verified
[3/20] sub-15 → 33.84 MB
    ✅ verified
[4/20] sub-12 → 73.64 MB
    ✅ verified
[5/20] sub-08 → 29.51 MB


KeyboardInterrupt: 

In [ ]:
# =============================================================================
# MODEL 3B - RUNTIME REINITIALIZATION
# =============================================================================

from pathlib import Path
import os
import json
import random
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# DIRECTORIES
# -----------------------------------------------------------------------------

ROOT = Path("/content/model3b")

RAW_DIR = ROOT / "raw"
PROCESSED_DIR = ROOT / "processed"
MANIFEST_DIR = ROOT / "manifests"
AUDIT_DIR = ROOT / "audit"
MODEL_DIR = ROOT / "model"
RESULTS_DIR = ROOT / "results"

for d in [
    ROOT,
    RAW_DIR,
    PROCESSED_DIR,
    MANIFEST_DIR,
    AUDIT_DIR,
    MODEL_DIR,
    RESULTS_DIR
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

# -----------------------------------------------------------------------------
# DRIVE
# -----------------------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

DRIVE_PROCESSED = DRIVE_ROOT / "processed"
DRIVE_MANIFESTS = DRIVE_ROOT / "manifests"
DRIVE_MODEL = DRIVE_ROOT / "model"
DRIVE_RESULTS = DRIVE_ROOT / "results"

for d in [
    DRIVE_ROOT,
    DRIVE_PROCESSED,
    DRIVE_MANIFESTS,
    DRIVE_MODEL,
    DRIVE_RESULTS
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

# -----------------------------------------------------------------------------
# CLASSES
# -----------------------------------------------------------------------------

CLASSES = {
    0: "Meningioma",
    1: "Pituitary",
    2: "Brain_Metastasis",
    3: "Schwannoma"
}

CLASS_TO_ID = {
    v: k
    for k, v in CLASSES.items()
}

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# -----------------------------------------------------------------------------
# CHECK CURRENT FILESYSTEM
# -----------------------------------------------------------------------------

print("=" * 80)
print("MODEL 3B - RUNTIME REINITIALIZED")
print("=" * 80)

print("\nRaw directory:")
print(RAW_DIR)

print("\nExisting CT volumes:")

for cls in CLASSES.values():

    folder = RAW_DIR / cls

    if folder.exists():

        files = list(
            folder.glob(
                "sub-*/CT-BRAIN.nii.gz"
            )
        )

    else:
        files = []

    print(
        f"{cls:20s}: {len(files)}"
    )

print("\nDrive:")
print(DRIVE_ROOT)

print("\n✅ Variables restored.")
print("=" * 80)

Mounted at /content/drive
MODEL 3B - RUNTIME REINITIALIZED

Raw directory:
/content/model3b/raw

Existing CT volumes:
Meningioma          : 0
Pituitary           : 0
Brain_Metastasis    : 0
Schwannoma          : 0

Drive:
/content/drive/MyDrive/Model3B

✅ Variables restored.


In [ ]:
# =============================================================================
# MODEL 3B - STEP 3
# SELECTIVE CT ACQUISITION
# MENINGIOMA + PITUITARY
# ZIP64 + HTTP RANGE + RESUME SAFE
# =============================================================================

import os
import re
import io
import json
import struct
import hashlib
import requests
import zipfile
import zlib
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

# =============================================================================
# CONFIGURATION
# =============================================================================

ROOT = Path("/content/model3b")

RAW_DIR = ROOT / "raw"
MANIFEST_DIR = ROOT / "manifests"
AUDIT_DIR = ROOT / "audit"

RAW_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_TO_ID = {
    "Meningioma": 0,
    "Pituitary": 1,
    "Brain_Metastasis": 2,
    "Schwannoma": 3
}

SOURCES = {
    "Meningioma": {
        "url":
        "https://zenodo.org/api/records/17486320/files/meningioma.zip/content",
        "output":
        RAW_DIR / "Meningioma"
    },

    "Pituitary": {
        "url":
        "https://zenodo.org/api/records/17486320/files/pituitary-tumor.zip/content",
        "output":
        RAW_DIR / "Pituitary"
    }
}

for source in SOURCES.values():
    source["output"].mkdir(
        parents=True,
        exist_ok=True
    )

# =============================================================================
# HTTP SESSION
# =============================================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Model3B-CT-Acquisition/1.0"
})


# =============================================================================
# HTTP RANGE
# =============================================================================

def get_range(url, start, end):

    r = session.get(
        url,
        headers={
            "Range": f"bytes={start}-{end}"
        },
        timeout=120
    )

    r.raise_for_status()

    if r.status_code != 206:
        raise RuntimeError(
            f"Server did not return HTTP 206: {r.status_code}"
        )

    return r.content


# =============================================================================
# REMOTE FILE SIZE
# =============================================================================

def get_file_size(url):

    r = session.get(
        url,
        headers={
            "Range": "bytes=0-0"
        },
        timeout=60
    )

    r.raise_for_status()

    cr = r.headers.get(
        "Content-Range",
        ""
    )

    m = re.search(
        r"/(\d+)$",
        cr
    )

    if not m:

        raise RuntimeError(
            f"Could not determine remote file size.\n"
            f"Content-Range: {cr}"
        )

    return int(m.group(1))


# =============================================================================
# ZIP CENTRAL DIRECTORY
# =============================================================================

def read_zip_directory(url):

    size = get_file_size(url)

    print(
        f"Archive size: "
        f"{size / 1024**3:.3f} GB"
    )

    tail_size = min(
        8 * 1024 * 1024,
        size
    )

    tail_start = size - tail_size

    tail = get_range(
        url,
        tail_start,
        size - 1
    )

    print(
        f"Downloaded ZIP tail: "
        f"{len(tail) / 1024**2:.2f} MB"
    )

    # -------------------------------------------------------------------------
    # EOCD
    # -------------------------------------------------------------------------

    eocd_sig = b"PK\x05\x06"

    eocd_pos = tail.rfind(
        eocd_sig
    )

    if eocd_pos < 0:

        raise RuntimeError(
            "EOCD not found."
        )

    eocd = tail[
        eocd_pos:
        eocd_pos + 22
    ]

    (
        signature,
        disk,
        cd_disk,
        disk_entries,
        total_entries,
        cd_size_32,
        cd_offset_32,
        comment_len
    ) = struct.unpack(
        "<4sHHHHIIH",
        eocd
    )

    eocd_absolute = (
        tail_start +
        eocd_pos
    )

    print(
        f"EOCD absolute offset: "
        f"{eocd_absolute}"
    )

    # -------------------------------------------------------------------------
    # ZIP64
    # -------------------------------------------------------------------------

    if (
        total_entries == 0xFFFF
        or
        cd_size_32 == 0xFFFFFFFF
        or
        cd_offset_32 == 0xFFFFFFFF
    ):

        zip64_locator_sig = b"PK\x06\x07"

        locator_pos = tail.rfind(
            zip64_locator_sig,
            0,
            eocd_pos
        )

        if locator_pos < 0:

            raise RuntimeError(
                "ZIP64 locator not found."
            )

        locator = tail[
            locator_pos:
            locator_pos + 20
        ]

        (
            _,
            zip64_disk,
            zip64_offset,
            total_disks
        ) = struct.unpack(
            "<4sIQI",
            locator
        )

        zip64_abs = zip64_offset

        zip64_record = get_range(
            url,
            zip64_abs,
            zip64_abs + 55
        )

        (
            sig,
            record_size,
            version_made,
            version_needed,
            disk_number,
            cd_disk_number,
            entries_disk,
            total_entries,
            cd_size,
            cd_offset
        ) = struct.unpack(
            "<4sQHHIIQQQQ",
            zip64_record[:56]
        )

    else:

        cd_size = cd_size_32
        cd_offset = cd_offset_32

    print(
        f"Total entries: "
        f"{total_entries}"
    )

    print(
        f"Central directory: "
        f"{cd_size / 1024**2:.3f} MB"
    )

    print(
        f"Central directory offset: "
        f"{cd_offset}"
    )

    # -------------------------------------------------------------------------
    # DOWNLOAD CENTRAL DIRECTORY ONLY
    # -------------------------------------------------------------------------

    cd = get_range(
        url,
        cd_offset,
        cd_offset + cd_size - 1
    )

    print(
        f"Central directory downloaded: "
        f"{len(cd) / 1024**2:.3f} MB"
    )

    return size, cd


# =============================================================================
# CENTRAL DIRECTORY PARSER
# =============================================================================

def parse_central_directory(cd):

    entries = []

    pos = 0

    signature = b"PK\x01\x02"

    while pos < len(cd):

        idx = cd.find(
            signature,
            pos
        )

        if idx < 0:
            break

        if idx + 46 > len(cd):
            break

        fixed = cd[
            idx:
            idx + 46
        ]

        fields = struct.unpack(
            "<4s6H3I5H2I",
            fixed
        )

        (
            sig,
            version_made,
            version_needed,
            flag,
            compression,
            mod_time,
            mod_date,
            crc32,
            compressed_size,
            uncompressed_size,
            filename_len,
            extra_len,
            comment_len,
            disk_start,
            internal_attr,
            external_attr,
            local_offset
        ) = fields

        name_start = idx + 46

        name_end = (
            name_start +
            filename_len
        )

        extra_end = (
            name_end +
            extra_len
        )

        name = cd[
            name_start:
            name_end
        ].decode(
            "utf-8",
            errors="replace"
        )

        extra = cd[
            name_end:
            extra_end
        ]

        # ---------------------------------------------------------------------
        # ZIP64 EXTRA FIELD
        # ---------------------------------------------------------------------

        if (
            compressed_size == 0xFFFFFFFF
            or
            uncompressed_size == 0xFFFFFFFF
            or
            local_offset == 0xFFFFFFFF
        ):

            ep = 0

            while ep + 4 <= len(extra):

                header_id, data_size = struct.unpack(
                    "<HH",
                    extra[
                        ep:
                        ep + 4
                    ]
                )

                data = extra[
                    ep + 4:
                    ep + 4 + data_size
                ]

                if header_id == 0x0001:

                    dp = 0

                    if uncompressed_size == 0xFFFFFFFF:

                        if dp + 8 > len(data):
                            raise RuntimeError(
                                "Invalid ZIP64 uncompressed size field."
                            )

                        uncompressed_size = struct.unpack(
                            "<Q",
                            data[
                                dp:
                                dp + 8
                            ]
                        )[0]

                        dp += 8

                    if compressed_size == 0xFFFFFFFF:

                        if dp + 8 > len(data):
                            raise RuntimeError(
                                "Invalid ZIP64 compressed size field."
                            )

                        compressed_size = struct.unpack(
                            "<Q",
                            data[
                                dp:
                                dp + 8
                            ]
                        )[0]

                        dp += 8

                    if local_offset == 0xFFFFFFFF:

                        if dp + 8 > len(data):
                            raise RuntimeError(
                                "Invalid ZIP64 local offset field."
                            )

                        local_offset = struct.unpack(
                            "<Q",
                            data[
                                dp:
                                dp + 8
                            ]
                        )[0]

                        dp += 8

                    break

                ep += 4 + data_size

        entries.append({
            "name": name,
            "compression": compression,
            "compressed_size": compressed_size,
            "uncompressed_size": uncompressed_size,
            "local_offset": local_offset,
            "crc32": crc32,
            "flag": flag
        })

        pos = (
            idx +
            46 +
            filename_len +
            extra_len +
            comment_len
        )

    return entries


# =============================================================================
# SELECT EXACT CT-BRAIN FILES
# =============================================================================

def select_ct(entries, class_name):

    selected = []

    for e in entries:

        name = e["name"]

        normalized = name.lower()

        if not normalized.endswith(
            ".nii.gz"
        ):
            continue

        if "ct-brain.nii.gz" not in normalized:
            continue

        # Exclude CT-BONE
        if "ct-bone" in normalized:
            continue

        # Exclude MRI
        if "mri" in normalized:
            continue

        # Patient ID
        m = re.search(
            r"(sub-\d+)",
            name,
            re.IGNORECASE
        )

        if not m:
            continue

        patient = m.group(1).lower()

        selected.append({
            **e,
            "patient_id": patient,
            "class_name": class_name,
            "class_id": CLASS_TO_ID[class_name],
            "archive_path": name
        })

    # -------------------------------------------------------------------------
    # REMOVE DUPLICATE PATIENTS
    # -------------------------------------------------------------------------

    unique = {}

    for e in selected:

        patient = e["patient_id"]

        if patient not in unique:
            unique[patient] = e

    return list(
        unique.values()
    )


# =============================================================================
# EXTRACT ONE ZIP MEMBER
# =============================================================================

def extract_member(
    url,
    entry,
    output_path
):

    local_offset = entry[
        "local_offset"
    ]

    # -------------------------------------------------------------------------
    # LOCAL ZIP HEADER
    # -------------------------------------------------------------------------

    local_header = get_range(
        url,
        local_offset,
        local_offset + 30 - 1
    )

    (
        sig,
        version,
        flag,
        compression,
        mod_time,
        mod_date,
        crc32,
        compressed_size,
        uncompressed_size,
        filename_len,
        extra_len
    ) = struct.unpack(
        "<4s5H3I2H",
        local_header
    )

    if sig != b"PK\x03\x04":

        raise RuntimeError(
            "Invalid local ZIP header."
        )

    data_start = (
        local_offset +
        30 +
        filename_len +
        extra_len
    )

    data_end = (
        data_start +
        entry["compressed_size"] -
        1
    )

    # -------------------------------------------------------------------------
    # DOWNLOAD ONLY MEMBER
    # -------------------------------------------------------------------------

    compressed = get_range(
        url,
        data_start,
        data_end
    )

    # -------------------------------------------------------------------------
    # DECOMPRESS
    # -------------------------------------------------------------------------

    if entry["compression"] == 0:

        raw = compressed

    elif entry["compression"] == 8:

        raw = zlib.decompress(
            compressed,
            -15
        )

    else:

        raise RuntimeError(
            f"Unsupported ZIP compression: "
            f"{entry['compression']}"
        )

    # -------------------------------------------------------------------------
    # SIZE VALIDATION
    # -------------------------------------------------------------------------

    if len(raw) != entry[
        "uncompressed_size"
    ]:

        raise RuntimeError(
            f"Size mismatch: "
            f"{len(raw)} != "
            f"{entry['uncompressed_size']}"
        )

    # -------------------------------------------------------------------------
    # CRC32
    # -------------------------------------------------------------------------

    actual_crc = (
        zlib.crc32(raw) &
        0xffffffff
    )

    if actual_crc != entry["crc32"]:

        raise RuntimeError(
            "CRC32 mismatch."
        )

    # -------------------------------------------------------------------------
    # WRITE
    # -------------------------------------------------------------------------

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_path,
        "wb"
    ) as f:

        f.write(raw)

    # -------------------------------------------------------------------------
    # SHA256
    # -------------------------------------------------------------------------

    sha = hashlib.sha256(
        raw
    ).hexdigest()

    return sha


# =============================================================================
# ACQUIRE ONE SOURCE
# =============================================================================

def acquire_source(
    class_name,
    source
):

    print("\n")
    print("=" * 80)
    print(
        f"ACQUIRING {class_name.upper()}"
    )
    print("=" * 80)

    url = source["url"]

    output_root = source["output"]

    # -------------------------------------------------------------------------
    # READ ZIP DIRECTORY
    # -------------------------------------------------------------------------

    archive_size, cd = read_zip_directory(
        url
    )

    # -------------------------------------------------------------------------
    # PARSE
    # -------------------------------------------------------------------------

    entries = parse_central_directory(
        cd
    )

    print(
        f"Parsed ZIP entries: "
        f"{len(entries)}"
    )

    # -------------------------------------------------------------------------
    # SELECT CT
    # -------------------------------------------------------------------------

    selected = select_ct(
        entries,
        class_name
    )

    print(
        f"\nCT-BRAIN volumes: "
        f"{len(selected)}"
    )

    patients = sorted(
        {
            e["patient_id"]
            for e in selected
        }
    )

    print(
        f"Unique patients: "
        f"{len(patients)}"
    )

    if not selected:

        raise RuntimeError(
            f"No CT-BRAIN files found "
            f"for {class_name}"
        )

    # -------------------------------------------------------------------------
    # DUPLICATE CHECK
    # -------------------------------------------------------------------------

    if len(selected) != len(patients):

        raise RuntimeError(
            f"Duplicate patient CT volumes detected "
            f"for {class_name}."
        )

    manifest_rows = []

    total_mb = sum(
        e["compressed_size"]
        for e in selected
    ) / 1024**2

    print(
        f"Compressed acquisition: "
        f"{total_mb:.2f} MB"
    )

    # -------------------------------------------------------------------------
    # EXTRACTION
    # -------------------------------------------------------------------------

    for i, entry in enumerate(
        selected,
        1
    ):

        patient = entry[
            "patient_id"
        ]

        output_path = (
            output_root /
            patient /
            "CT-BRAIN.nii.gz"
        )

        expected = entry[
            "uncompressed_size"
        ]

        # ---------------------------------------------------------------------
        # RESUME SUPPORT
        # ---------------------------------------------------------------------

        if (
            output_path.exists()
            and
            output_path.stat().st_size
            == expected
        ):

            print(
                f"[{i}/{len(selected)}] "
                f"{patient} "
                f"→ already complete"
            )

            with open(
                output_path,
                "rb"
            ) as f:

                sha = hashlib.sha256(
                    f.read()
                ).hexdigest()

        else:

            print(
                f"[{i}/{len(selected)}] "
                f"{patient} "
                f"→ "
                f"{expected / 1024**2:.2f} MB"
            )

            sha = extract_member(
                url,
                entry,
                output_path
            )

            print(
                "    ✅ verified"
            )

        manifest_rows.append({
            "class_id":
                entry["class_id"],

            "class_name":
                class_name,

            "patient_id":
                patient,

            "archive_path":
                entry["archive_path"],

            "compressed_size":
                entry["compressed_size"],

            "uncompressed_size":
                entry["uncompressed_size"],

            "compression_method":
                entry["compression"],

            "local_header_offset":
                entry["local_offset"],

            "sha256":
                sha,

            "local_path":
                str(output_path)
        })

    # -------------------------------------------------------------------------
    # MANIFEST
    # -------------------------------------------------------------------------

    manifest = pd.DataFrame(
        manifest_rows
    )

    manifest_path = (
        MANIFEST_DIR /
        f"{class_name.lower()}_acquisition_manifest.csv"
    )

    manifest.to_csv(
        manifest_path,
        index=False
    )

    print(
        "\nManifest:"
    )

    print(
        manifest_path
    )

    print(
        f"\n✅ {class_name} acquisition complete"
    )

    return manifest


# =============================================================================
# RUN BOTH SOURCES
# =============================================================================

all_manifests = []

for class_name, source in SOURCES.items():

    m = acquire_source(
        class_name,
        source
    )

    all_manifests.append(
        m
    )


# =============================================================================
# COMBINED MANIFEST
# =============================================================================

combined = pd.concat(
    all_manifests,
    ignore_index=True
)

combined_path = (
    MANIFEST_DIR /
    "model3b_step1_ct_acquisition_manifest.csv"
)

combined.to_csv(
    combined_path,
    index=False
)


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n")
print("=" * 80)
print("MODEL 3B - STEP 3 COMPLETE")
print("=" * 80)

print(
    "\nTotal CT volumes:",
    len(combined)
)

print(
    "Unique patients:",
    combined["patient_id"].nunique()
)

print("\nBy class:")

print(
    combined.groupby(
        "class_name"
    )["patient_id"]
    .nunique()
)

print(
    "\nCombined manifest:"
)

print(
    combined_path
)

print(
    "\nRaw CT remains in:"
)

print(
    RAW_DIR
)

print("=" * 80)



ACQUIRING MENINGIOMA
Archive size: 4.208 GB
Downloaded ZIP tail: 8.00 MB
EOCD absolute offset: 4517987471
Total entries: 204
Central directory: 0.020 MB
Central directory offset: 4517966148
Central directory downloaded: 0.020 MB
Parsed ZIP entries: 204

CT-BRAIN volumes: 20
Unique patients: 20
Compressed acquisition: 653.45 MB
[1/20] sub-13 → already complete
[2/20] sub-14 → already complete
[3/20] sub-15 → already complete
[4/20] sub-12 → already complete
[5/20] sub-08 → 29.51 MB
    ✅ verified
[6/20] sub-01 → 27.20 MB
    ✅ verified
[7/20] sub-06 → 35.32 MB
    ✅ verified
[8/20] sub-07 → 27.30 MB
    ✅ verified
[9/20] sub-09 → 27.62 MB
    ✅ verified
[10/20] sub-17 → 26.73 MB
    ✅ verified
[11/20] sub-10 → 33.60 MB
    ✅ verified
[12/20] sub-19 → 28.33 MB
    ✅ verified
[13/20] sub-20 → 39.68 MB
    ✅ verified
[14/20] sub-18 → 29.02 MB
    ✅ verified
[15/20] sub-11 → 34.58 MB
    ✅ verified
[16/20] sub-16 → 30.48 MB
    ✅ verified
[17/20] sub-05 → 35.73 MB
    ✅ verified
[18/20] s

In [ ]:
# =============================================================================
# MODEL 3B - STEP 4
# CT VOLUME + MODALITY AUDIT
# =============================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path

print("=" * 80)
print("MODEL 3B - STEP 4")
print("CT VOLUME / MODALITY AUDIT")
print("=" * 80)

# =============================================================================
# CONFIG
# =============================================================================

ROOT = Path("/content/model3b")
RAW_DIR = ROOT / "raw"
MANIFEST_DIR = ROOT / "manifests"
AUDIT_DIR = ROOT / "audit"

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLASSES = {
    0: "Meningioma",
    1: "Pituitary",
    2: "Brain_Metastasis",
    3: "Schwannoma"
}

# =============================================================================
# DISCOVER CT VOLUMES
# =============================================================================

volume_paths = []

for class_id, class_name in CLASSES.items():

    class_dir = RAW_DIR / class_name

    if not class_dir.exists():
        continue

    for path in sorted(
        class_dir.glob(
            "sub-*/CT-BRAIN.nii.gz"
        )
    ):

        volume_paths.append({
            "class_id": class_id,
            "class_name": class_name,
            "patient_id": path.parent.name,
            "path": path
        })

print(
    f"\nCT volumes found: {len(volume_paths)}"
)

if len(volume_paths) == 0:
    raise RuntimeError(
        "No CT volumes found."
    )

# =============================================================================
# AUDIT
# =============================================================================

rows = []

invalid = []
failed = []

for i, item in enumerate(
    volume_paths,
    1
):

    path = item["path"]

    print(
        f"[{i}/{len(volume_paths)}] "
        f"{item['class_name']} / "
        f"{item['patient_id']}"
    )

    try:

        img = nib.load(
            str(path)
        )

        header = img.header

        shape = tuple(
            int(x)
            for x in img.shape
        )

        zooms = tuple(
            float(x)
            for x in header.get_zooms()
        )

        dtype = str(
            img.get_data_dtype()
        )

        # -------------------------------------------------------------
        # 3D CHECK
        # -------------------------------------------------------------

        is_3d = (
            len(shape) == 3
        )

        # -------------------------------------------------------------
        # FINITE CHECK
        # -------------------------------------------------------------

        data = img.get_fdata(
            dtype=np.float32
        )

        finite = bool(
            np.isfinite(data).all()
        )

        # -------------------------------------------------------------
        # INTENSITY STATISTICS
        # -------------------------------------------------------------

        finite_values = data[
            np.isfinite(data)
        ]

        if len(finite_values) > 0:

            vmin = float(
                np.min(finite_values)
            )

            vmax = float(
                np.max(finite_values)
            )

            mean = float(
                np.mean(finite_values)
            )

            std = float(
                np.std(finite_values)
            )

            p01 = float(
                np.percentile(
                    finite_values,
                    1
                )
            )

            p99 = float(
                np.percentile(
                    finite_values,
                    99
                )
            )

        else:

            vmin = np.nan
            vmax = np.nan
            mean = np.nan
            std = np.nan
            p01 = np.nan
            p99 = np.nan

        # -------------------------------------------------------------
        # VOXEL SPACING
        # -------------------------------------------------------------

        spacing_x = zooms[0] if len(zooms) > 0 else np.nan
        spacing_y = zooms[1] if len(zooms) > 1 else np.nan
        spacing_z = zooms[2] if len(zooms) > 2 else np.nan

        # -------------------------------------------------------------
        # FILE HASH
        # -------------------------------------------------------------

        sha = hashlib.sha256()

        with open(
            path,
            "rb"
        ) as f:

            for chunk in iter(
                lambda: f.read(1024 * 1024),
                b""
            ):

                sha.update(chunk)

        file_sha256 = sha.hexdigest()

        # -------------------------------------------------------------
        # STATUS
        # -------------------------------------------------------------

        valid = (
            is_3d
            and finite
            and all(
                np.isfinite(zooms)
            )
            and all(
                x > 0
                for x in zooms
            )
        )

        if not valid:
            invalid.append(
                str(path)
            )

        rows.append({

            "class_id":
                item["class_id"],

            "class_name":
                item["class_name"],

            "patient_id":
                item["patient_id"],

            "path":
                str(path),

            "shape":
                str(shape),

            "x":
                shape[0] if len(shape) > 0 else np.nan,

            "y":
                shape[1] if len(shape) > 1 else np.nan,

            "z":
                shape[2] if len(shape) > 2 else np.nan,

            "spacing_x":
                spacing_x,

            "spacing_y":
                spacing_y,

            "spacing_z":
                spacing_z,

            "dtype":
                dtype,

            "min":
                vmin,

            "max":
                vmax,

            "mean":
                mean,

            "std":
                std,

            "p01":
                p01,

            "p99":
                p99,

            "is_3d":
                is_3d,

            "finite":
                finite,

            "valid":
                valid,

            "file_size_mb":
                path.stat().st_size / 1024**2,

            "sha256":
                file_sha256
        })

        del data
        del finite_values

    except Exception as e:

        failed.append({
            "path": str(path),
            "error": str(e)
        })

# =============================================================================
# DATAFRAME
# =============================================================================

audit_df = pd.DataFrame(
    rows
)

# =============================================================================
# SAVE AUDIT
# =============================================================================

audit_csv = (
    AUDIT_DIR /
    "model3b_step4_ct_volume_audit.csv"
)

audit_df.to_csv(
    audit_csv,
    index=False
)

# =============================================================================
# SUMMARY
# =============================================================================

summary = {

    "total_volumes":
        len(volume_paths),

    "successfully_read":
        len(rows),

    "invalid":
        len(invalid),

    "failed":
        len(failed),

    "classes":
        {
            name:
                int(
                    (
                        audit_df[
                            "class_name"
                        ] == name
                    ).sum()
                )
            for name in CLASSES.values()
        },

    "unique_patients":
        int(
            audit_df[
                "patient_id"
            ].nunique()
        ),

    "unique_shapes":
        [
            str(x)
            for x in
            sorted(
                audit_df[
                    "shape"
                ].unique()
            )
        ],

    "raw_ct_modified":
        False
}

summary_json = (
    AUDIT_DIR /
    "model3b_step4_ct_volume_audit_summary.json"
)

with open(
    summary_json,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# PRINT SUMMARY
# =============================================================================

print("\n")
print("=" * 80)
print("STEP 4 SUMMARY")
print("=" * 80)

print(
    "\nTotal CT volumes:",
    len(volume_paths)
)

print(
    "Successfully read:",
    len(rows)
)

print(
    "Invalid:",
    len(invalid)
)

print(
    "Failed:",
    len(failed)
)

print(
    "\nClass distribution:"
)

print(
    audit_df[
        "class_name"
    ].value_counts()
)

print(
    "\nUnique patients:",
    audit_df[
        "patient_id"
    ].nunique()
)

print(
    "\nUnique volume shapes:"
)

print(
    audit_df[
        "shape"
    ].value_counts()
)

print(
    "\nIntensity range:"
)

print(
    "Minimum:",
    audit_df["min"].min()
)

print(
    "Maximum:",
    audit_df["max"].max()
)

print(
    "\nAudit CSV:"
)

print(
    audit_csv
)

print(
    "\nSummary JSON:"
)

print(
    summary_json
)

# =============================================================================
# SAFETY CHECK
# =============================================================================

if len(invalid) > 0:

    print(
        "\n⚠️ INVALID VOLUMES:"
    )

    for p in invalid:
        print(p)

elif len(failed) > 0:

    print(
        "\n⚠️ FAILED VOLUMES:"
    )

    for x in failed:
        print(
            x["path"],
            "->",
            x["error"]
        )

else:

    print(
        "\n✅ ALL CT VOLUMES PASSED BASIC AUDIT."
    )

print("\n")
print("=" * 80)
print("MODEL 3B - STEP 4 COMPLETE")
print("=" * 80)

MODEL 3B - STEP 4
CT VOLUME / MODALITY AUDIT

CT volumes found: 80
[1/80] Meningioma / sub-01
[2/80] Meningioma / sub-02
[3/80] Meningioma / sub-03
[4/80] Meningioma / sub-04
[5/80] Meningioma / sub-05
[6/80] Meningioma / sub-06
[7/80] Meningioma / sub-07
[8/80] Meningioma / sub-08
[9/80] Meningioma / sub-09
[10/80] Meningioma / sub-10
[11/80] Meningioma / sub-11
[12/80] Meningioma / sub-12
[13/80] Meningioma / sub-13
[14/80] Meningioma / sub-14
[15/80] Meningioma / sub-15
[16/80] Meningioma / sub-16
[17/80] Meningioma / sub-17
[18/80] Meningioma / sub-18
[19/80] Meningioma / sub-19
[20/80] Meningioma / sub-20
[21/80] Pituitary / sub-01
[22/80] Pituitary / sub-02
[23/80] Pituitary / sub-03
[24/80] Pituitary / sub-04
[25/80] Pituitary / sub-05
[26/80] Pituitary / sub-06
[27/80] Pituitary / sub-07
[28/80] Pituitary / sub-08
[29/80] Pituitary / sub-09
[30/80] Pituitary / sub-10
[31/80] Pituitary / sub-11
[32/80] Pituitary / sub-12
[33/80] Pituitary / sub-13
[34/80] Pituitary / sub-14
[35/

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

for cls in ["Meningioma", "Pituitary", "Brain_Metastasis", "Schwannoma"]:
    p = Path("/content/model3b/raw") / cls
    files = list(p.glob("sub-*/CT-BRAIN.nii.gz")) if p.exists() else []
    print(f"{cls:20s}: {len(files)}")

Meningioma          : 20
Pituitary           : 60
Brain_Metastasis    : 0
Schwannoma          : 0


In [ ]:
# =============================================================================
# MODEL 3B - STEP 5
# BRAIN METASTASIS CT SOURCE
# =============================================================================

import requests
import json

url = "https://zenodo.org/api/records/14525197"

r = requests.get(url, timeout=60)
r.raise_for_status()

record = r.json()

print("=" * 80)
print("BRAIN METASTASIS SOURCE")
print("=" * 80)

print("Record ID:", record.get("id"))
print("Title:", record["metadata"].get("title"))
print("License:", record["metadata"].get("license"))

print("\nFiles:")

for f in record.get("files", []):
    print(
        f"{f['key']} | "
        f"{f.get('size', 0) / 1024**3:.3f} GB"
    )

print("=" * 80)

BRAIN METASTASIS SOURCE
Record ID: 14525197
Title: RFUds - A Brain Metastases Imaging Dataset of Radiotherapy Follow-Up
License: {'id': 'cc-by-4.0'}

Files:
Transformation Matrices.pdf | 0.000 GB
RFUds.zip | 7.280 GB


In [ ]:
# =============================================================================
# MODEL 3B - STEP 5B
# RFUds SELECTIVE BRAIN-METASTASIS CT ACQUISITION
# =============================================================================

import os
import re
import struct
import zlib
import hashlib
import requests
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------

ROOT = Path("/content/model3b")
RAW_DIR = ROOT / "raw"
MANIFEST_DIR = ROOT / "manifests"
AUDIT_DIR = ROOT / "audit"

OUT_DIR = RAW_DIR / "Brain_Metastasis"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# SOURCE
# -----------------------------------------------------------------------------

URL = (
    "https://zenodo.org/api/records/14525197/"
    "files/RFUds.zip/content"
)

CLASS_ID = 2
CLASS_NAME = "Brain_Metastasis"

session = requests.Session()
session.headers.update({
    "User-Agent": "Model3B-CT-Acquisition/1.0"
})

# -----------------------------------------------------------------------------
# RANGE DOWNLOAD
# -----------------------------------------------------------------------------

def get_range(start, end):

    r = session.get(
        URL,
        headers={
            "Range": f"bytes={start}-{end}"
        },
        timeout=180
    )

    r.raise_for_status()

    if r.status_code != 206:
        raise RuntimeError(
            f"Expected HTTP 206, got {r.status_code}"
        )

    return r.content


# -----------------------------------------------------------------------------
# ARCHIVE SIZE
# -----------------------------------------------------------------------------

r = session.get(
    URL,
    headers={"Range": "bytes=0-0"},
    timeout=60
)

r.raise_for_status()

content_range = r.headers.get(
    "Content-Range",
    ""
)

m = re.search(
    r"/(\d+)$",
    content_range
)

if not m:
    raise RuntimeError(
        f"Could not determine archive size: {content_range}"
    )

ARCHIVE_SIZE = int(m.group(1))

print("=" * 80)
print("MODEL 3B - STEP 5B")
print("RFUds SELECTIVE BRAIN-METASTASIS CT ACQUISITION")
print("=" * 80)

print(
    f"\nArchive size: "
    f"{ARCHIVE_SIZE / 1024**3:.3f} GB"
)

# -----------------------------------------------------------------------------
# ZIP TAIL
# -----------------------------------------------------------------------------

TAIL_SIZE = min(
    8 * 1024 * 1024,
    ARCHIVE_SIZE
)

tail_start = ARCHIVE_SIZE - TAIL_SIZE

tail = get_range(
    tail_start,
    ARCHIVE_SIZE - 1
)

print(
    f"ZIP tail downloaded: "
    f"{len(tail) / 1024**2:.2f} MB"
)

# -----------------------------------------------------------------------------
# EOCD
# -----------------------------------------------------------------------------

EOCD = b"PK\x05\x06"

eocd_pos = tail.rfind(EOCD)

if eocd_pos < 0:
    raise RuntimeError("EOCD not found.")

eocd_abs = tail_start + eocd_pos

(
    signature,
    disk,
    cd_disk,
    disk_entries,
    total_entries_32,
    cd_size_32,
    cd_offset_32,
    comment_length
) = struct.unpack(
    "<4sHHHHIIH",
    tail[eocd_pos:eocd_pos + 22]
)

print(
    "EOCD absolute offset:",
    eocd_abs
)

# -----------------------------------------------------------------------------
# ZIP64
# -----------------------------------------------------------------------------

if (
    total_entries_32 == 0xFFFF
    or
    cd_size_32 == 0xFFFFFFFF
    or
    cd_offset_32 == 0xFFFFFFFF
):

    locator_sig = b"PK\x06\x07"

    locator_pos = tail.rfind(
        locator_sig,
        0,
        eocd_pos
    )

    if locator_pos < 0:
        raise RuntimeError(
            "ZIP64 locator not found."
        )

    (
        _,
        disk_number,
        zip64_offset,
        total_disks
    ) = struct.unpack(
        "<4sIQI",
        tail[
            locator_pos:
            locator_pos + 20
        ]
    )

    zip64 = get_range(
        zip64_offset,
        zip64_offset + 55
    )

    (
        sig,
        record_size,
        version_made,
        version_needed,
        disk_number,
        cd_disk_number,
        entries_disk,
        total_entries,
        cd_size,
        cd_offset
    ) = struct.unpack(
        "<4sQHHIIQQQQ",
        zip64[:56]
    )

else:

    total_entries = total_entries_32
    cd_size = cd_size_32
    cd_offset = cd_offset_32

print(
    "Total ZIP entries:",
    total_entries
)

print(
    "Central directory:",
    f"{cd_size / 1024**2:.3f} MB"
)

print(
    "Central directory offset:",
    cd_offset
)

# -----------------------------------------------------------------------------
# DOWNLOAD CENTRAL DIRECTORY
# -----------------------------------------------------------------------------

cd = get_range(
    cd_offset,
    cd_offset + cd_size - 1
)

print(
    "Central directory downloaded:",
    f"{len(cd) / 1024**2:.3f} MB"
)

# -----------------------------------------------------------------------------
# PARSE CENTRAL DIRECTORY
# -----------------------------------------------------------------------------

entries = []

pos = 0

while pos < len(cd):

    idx = cd.find(
        b"PK\x01\x02",
        pos
    )

    if idx < 0:
        break

    fixed = cd[
        idx:
        idx + 46
    ]

    if len(fixed) < 46:
        break

    (
        sig,
        version_made,
        version_needed,
        flag,
        compression,
        mod_time,
        mod_date,
        crc32,
        compressed_size,
        uncompressed_size,
        filename_len,
        extra_len,
        comment_len,
        disk_start,
        internal_attr,
        external_attr,
        local_offset
    ) = struct.unpack(
        "<4s6H3I5H2I",
        fixed
    )

    name_start = idx + 46
    name_end = name_start + filename_len
    extra_end = name_end + extra_len
    comment_end = extra_end + comment_len

    name = cd[
        name_start:name_end
    ].decode(
        "utf-8",
        errors="replace"
    )

    extra = cd[
        name_end:extra_end
    ]

    # -------------------------------------------------------------------------
    # ZIP64 EXTRA FIELD
    # -------------------------------------------------------------------------

    if (
        compressed_size == 0xFFFFFFFF
        or
        uncompressed_size == 0xFFFFFFFF
        or
        local_offset == 0xFFFFFFFF
    ):

        ep = 0

        while ep + 4 <= len(extra):

            header_id, data_size = struct.unpack(
                "<HH",
                extra[ep:ep + 4]
            )

            data = extra[
                ep + 4:
                ep + 4 + data_size
            ]

            if header_id == 0x0001:

                dp = 0

                if uncompressed_size == 0xFFFFFFFF:
                    uncompressed_size = struct.unpack(
                        "<Q",
                        data[dp:dp + 8]
                    )[0]
                    dp += 8

                if compressed_size == 0xFFFFFFFF:
                    compressed_size = struct.unpack(
                        "<Q",
                        data[dp:dp + 8]
                    )[0]
                    dp += 8

                if local_offset == 0xFFFFFFFF:
                    local_offset = struct.unpack(
                        "<Q",
                        data[dp:dp + 8]
                    )[0]
                    dp += 8

                break

            ep += 4 + data_size

    entries.append({
        "archive_path": name,
        "compression": compression,
        "compressed_size": compressed_size,
        "uncompressed_size": uncompressed_size,
        "crc32": crc32,
        "local_offset": local_offset
    })

    pos = comment_end

print(
    "Parsed entries:",
    len(entries)
)

# -----------------------------------------------------------------------------
# SELECT CT IMAGE VOLUMES
# -----------------------------------------------------------------------------

ct_entries = []

for e in entries:

    name = e["archive_path"]
    n = name.lower()

    if not n.endswith(".nii.gz"):
        continue

    # Exclude masks / structures
    if any(
        x in n
        for x in [
            "mask",
            "segmentation",
            "structure",
            "contour",
            "label"
        ]
    ):
        continue

    # Exclude MRI
    if "mri" in n:
        continue

    # RFUds CT naming
    if (
        "_ct_" not in n
        and
        "/ct-" not in n
        and
        "ct_" not in n
    ):
        continue

    # Require windowed/denoised CT image
    if (
        "windowed" not in n
        and
        "denoised" not in n
    ):
        continue

    # Patient ID
    match = re.search(
        r"patient[_-](\d+)",
        n
    )

    if not match:
        continue

    patient_num = int(
        match.group(1)
    )

    patient_id = (
        f"Patient_{patient_num:02d}"
    )

    # CT number
    ct_match = re.search(
        r"_ct_(\d+)_",
        n
    )

    ct_number = (
        int(ct_match.group(1))
        if ct_match
        else None
    )

    ct_entries.append({
        **e,
        "patient_id": patient_id,
        "ct_number": ct_number,
        "class_id": CLASS_ID,
        "class_name": CLASS_NAME
    })

# -----------------------------------------------------------------------------
# DEDUPLICATE EXACT PATHS
# -----------------------------------------------------------------------------

unique = {}

for e in ct_entries:
    unique[e["archive_path"]] = e

ct_entries = list(
    unique.values()
)

print("\n")
print("=" * 80)
print("CT SELECTION")
print("=" * 80)

print(
    "Exact CT volumes:",
    len(ct_entries)
)

print(
    "Unique patients:",
    len({
        x["patient_id"]
        for x in ct_entries
    })
)

if not ct_entries:

    raise RuntimeError(
        "No CT volumes detected. "
        "STOP — do not download."
    )

# -----------------------------------------------------------------------------
# PATIENT SUMMARY
# -----------------------------------------------------------------------------

selection_df = pd.DataFrame(
    ct_entries
)

print("\nCT volumes per patient:")

print(
    selection_df[
        "patient_id"
    ].value_counts().sort_index()
)

compressed_gb = (
    selection_df[
        "compressed_size"
    ].sum()
    / 1024**3
)

uncompressed_gb = (
    selection_df[
        "uncompressed_size"
    ].sum()
    / 1024**3
)

print(
    f"\nCompressed CT size: "
    f"{compressed_gb:.3f} GB"
)

print(
    f"Uncompressed CT size: "
    f"{uncompressed_gb:.3f} GB"
)

# -----------------------------------------------------------------------------
# SAVE SELECTION MANIFEST
# -----------------------------------------------------------------------------

selection_manifest = (
    AUDIT_DIR /
    "model3b_step5b_rfuds_ct_selection.csv"
)

selection_df.to_csv(
    selection_manifest,
    index=False
)

print(
    "\nSelection manifest:"
)

print(
    selection_manifest
)

# -----------------------------------------------------------------------------
# ACQUISITION FUNCTION
# -----------------------------------------------------------------------------

def acquire_member(
    entry,
    output_path
):

    local_offset = entry[
        "local_offset"
    ]

    # Local file header
    header = get_range(
        local_offset,
        local_offset + 29
    )

    (
        sig,
        version,
        flag,
        compression,
        mod_time,
        mod_date,
        crc32_local,
        compressed_size_local,
        uncompressed_size_local,
        filename_len,
        extra_len
    ) = struct.unpack(
        "<4s5H3I2H",
        header
    )

    if sig != b"PK\x03\x04":
        raise RuntimeError(
            "Invalid ZIP local header."
        )

    data_start = (
        local_offset
        + 30
        + filename_len
        + extra_len
    )

    data_end = (
        data_start
        + entry["compressed_size"]
        - 1
    )

    compressed = get_range(
        data_start,
        data_end
    )

    # Deflate
    if entry["compression"] == 8:

        raw = zlib.decompress(
            compressed,
            -15
        )

    # Stored
    elif entry["compression"] == 0:

        raw = compressed

    else:

        raise RuntimeError(
            f"Unsupported compression: "
            f"{entry['compression']}"
        )

    # Size check
    if len(raw) != entry[
        "uncompressed_size"
    ]:

        raise RuntimeError(
            "Uncompressed size mismatch."
        )

    # CRC
    actual_crc = (
        zlib.crc32(raw)
        & 0xffffffff
    )

    if actual_crc != entry[
        "crc32"
    ]:

        raise RuntimeError(
            "CRC32 mismatch."
        )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_path,
        "wb"
    ) as f:

        f.write(raw)

    sha = hashlib.sha256(
        raw
    ).hexdigest()

    return sha


# -----------------------------------------------------------------------------
# ACQUIRE
# -----------------------------------------------------------------------------

print("\n")
print("=" * 80)
print("SELECTIVE CT ACQUISITION")
print("=" * 80)

acquired_rows = []

for i, entry in enumerate(
    ct_entries,
    1
):

    patient = entry[
        "patient_id"
    ]

    ct_number = entry[
        "ct_number"
    ]

    # Keep separate CT studies if patient has multiple
    if ct_number is not None:

        filename = (
            f"{patient}_CT_{ct_number}.nii.gz"
        )

    else:

        filename = (
            Path(
                entry["archive_path"]
            ).name
        )

    output_path = (
        OUT_DIR /
        patient /
        filename
    )

    expected_size = entry[
        "uncompressed_size"
    ]

    print(
        f"[{i}/{len(ct_entries)}] "
        f"{patient} "
        f"CT-{ct_number} "
        f"→ "
        f"{entry['compressed_size'] / 1024**2:.2f} MB"
    )

    # Resume
    if (
        output_path.exists()
        and
        output_path.stat().st_size
        == expected_size
    ):

        print(
            "    already complete"
        )

        with open(
            output_path,
            "rb"
        ) as f:

            sha = hashlib.sha256(
                f.read()
            ).hexdigest()

    else:

        sha = acquire_member(
            entry,
            output_path
        )

        print(
            "    ✅ verified"
        )

    acquired_rows.append({
        "class_id": CLASS_ID,
        "class_name": CLASS_NAME,
        "patient_id": patient,
        "ct_number": ct_number,
        "archive_path":
            entry["archive_path"],
        "compressed_size":
            entry["compressed_size"],
        "uncompressed_size":
            entry["uncompressed_size"],
        "compression_method":
            entry["compression"],
        "local_header_offset":
            entry["local_offset"],
        "sha256":
            sha,
        "local_path":
            str(output_path)
    })

# -----------------------------------------------------------------------------
# SAVE ACQUISITION MANIFEST
# -----------------------------------------------------------------------------

acquisition_df = pd.DataFrame(
    acquired_rows
)

acquisition_manifest = (
    MANIFEST_DIR /
    "brain_metastasis_acquisition_manifest.csv"
)

acquisition_df.to_csv(
    acquisition_manifest,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL CHECK
# -----------------------------------------------------------------------------

missing = []

for row in acquired_rows:

    p = Path(
        row["local_path"]
    )

    if not p.exists():
        missing.append(
            str(p)
        )

print("\n")
print("=" * 80)
print("MODEL 3B - STEP 5B COMPLETE")
print("=" * 80)

print(
    "\nCT volumes selected:",
    len(ct_entries)
)

print(
    "CT volumes acquired:",
    len(acquired_rows)
)

print(
    "Missing files:",
    len(missing)
)

print(
    "Unique patients:",
    acquisition_df[
        "patient_id"
    ].nunique()
)

print(
    "\nManifest:"
)

print(
    acquisition_manifest
)

if missing:

    print(
        "\n⚠️ Missing:"
    )

    for x in missing:
        print(x)

else:

    print(
        "\n✅ ALL SELECTED BRAIN-METASTASIS CT FILES ACQUIRED."
    )

print("=" * 80)

MODEL 3B - STEP 5B
RFUds SELECTIVE BRAIN-METASTASIS CT ACQUISITION

Archive size: 7.280 GB
ZIP tail downloaded: 8.00 MB
EOCD absolute offset: 7817181141
Total ZIP entries: 2243
Central directory: 0.221 MB
Central directory offset: 7816949032
Central directory downloaded: 0.221 MB
Parsed entries: 2243


CT SELECTION
Exact CT volumes: 91
Unique patients: 44

CT volumes per patient:
patient_id
Patient_01    2
Patient_02    2
Patient_03    2
Patient_04    1
Patient_05    3
Patient_06    2
Patient_07    1
Patient_08    2
Patient_09    2
Patient_10    2
Patient_11    2
Patient_12    3
Patient_14    2
Patient_15    2
Patient_16    2
Patient_17    2
Patient_18    2
Patient_19    3
Patient_20    2
Patient_21    2
Patient_22    2
Patient_25    1
Patient_26    2
Patient_27    2
Patient_28    2
Patient_31    2
Patient_32    2
Patient_33    2
Patient_34    4
Patient_35    2
Patient_37    2
Patient_38    2
Patient_39    2
Patient_42    2
Patient_43    2
Patient_44    2
Patient_45    2
Patient_46    

In [ ]:
# =============================================================================
# MODEL 3B - STEP 5C
# SCHWANNOMA CT SOURCE CHECK
# =============================================================================

from pathlib import Path
import os

ROOT = Path("/content/model3b")
RAW_DIR = ROOT / "raw"

print("=" * 80)
print("SCHWANNOMA CT SOURCE CHECK")
print("=" * 80)

# Search current filesystem
patterns = [
    "**/*schwannoma*",
    "**/*Schwannoma*",
    "**/*SCHWANNOMA*"
]

matches = []

for pattern in patterns:
    matches.extend(
        RAW_DIR.glob(pattern)
    )

matches = sorted(
    set(matches)
)

print(
    f"\nSchwannoma-related filesystem entries: {len(matches)}"
)

for x in matches[:100]:
    print(x)

# Search for CT NIfTI files
ct_files = []

for p in RAW_DIR.rglob("*.nii.gz"):

    name = str(p).lower()

    if "schwannoma" in name:

        if (
            "ct" in name
            and "mri" not in name
        ):
            ct_files.append(p)

print(
    f"\nPossible Schwannoma CT NIfTI files: {len(ct_files)}"
)

for p in ct_files[:50]:
    print(p)

print("\n" + "=" * 80)

if ct_files:
    print("✅ Schwannoma CT data already exists.")
else:
    print("⚠️ No Schwannoma CT data found locally.")
    print("We need to locate an actual CT source before proceeding.")

print("=" * 80)

SCHWANNOMA CT SOURCE CHECK

Schwannoma-related filesystem entries: 0

Possible Schwannoma CT NIfTI files: 0

⚠️ No Schwannoma CT data found locally.
We need to locate an actual CT source before proceeding.


In [ ]:
# =============================================================================
# MODEL 3B - STEP 6
# FINAL 3-CLASS CT AUDIT + IMPORTANT DATA BACKUP
# =============================================================================

import os
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import nibabel as nib

from pathlib import Path
from datetime import datetime

# =============================================================================
# PATHS
# =============================================================================

ROOT = Path("/content/model3b")

RAW_DIR = ROOT / "raw"
MANIFEST_DIR = ROOT / "manifests"
AUDIT_DIR = ROOT / "audit"

DRIVE_ROOT = Path("/content/drive/MyDrive/Model3B")

DRIVE_MANIFEST = DRIVE_ROOT / "manifests"
DRIVE_AUDIT = DRIVE_ROOT / "audits"
DRIVE_CONFIG = DRIVE_ROOT / "configs"

for p in [
    DRIVE_ROOT,
    DRIVE_MANIFEST,
    DRIVE_AUDIT,
    DRIVE_CONFIG
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

# =============================================================================
# FINAL CLASSES
# =============================================================================

CLASSES = {
    0: "Meningioma",
    1: "Pituitary",
    2: "Brain_Metastasis"
}

print("=" * 80)
print("MODEL 3B - STEP 6")
print("FINAL 3-CLASS CT AUDIT")
print("=" * 80)

# =============================================================================
# DISCOVER VOLUMES
# =============================================================================

volume_paths = []

for class_id, class_name in CLASSES.items():

    class_dir = RAW_DIR / class_name

    if not class_dir.exists():
        print(
            f"⚠️ Missing directory: {class_dir}"
        )
        continue

    for path in class_dir.rglob("*.nii.gz"):

        volume_paths.append({
            "class_id": class_id,
            "class_name": class_name,
            "patient_id": path.parent.name,
            "path": path
        })

print(
    f"\nCT volumes discovered: {len(volume_paths)}"
)

# =============================================================================
# EXPECTED COUNTS
# =============================================================================

expected_patients = {
    "Meningioma": 20,
    "Pituitary": 60,
    "Brain_Metastasis": 44
}

# =============================================================================
# AUDIT
# =============================================================================

rows = []
failed = []

for i, item in enumerate(
    volume_paths,
    1
):

    path = item["path"]

    print(
        f"[{i}/{len(volume_paths)}] "
        f"{item['class_name']} / "
        f"{item['patient_id']}"
    )

    try:

        img = nib.load(
            str(path)
        )

        shape = tuple(
            int(x)
            for x in img.shape
        )

        zooms = tuple(
            float(x)
            for x in img.header.get_zooms()
        )

        dtype = str(
            img.get_data_dtype()
        )

        # -------------------------------------------------------------
        # LOAD DATA
        # -------------------------------------------------------------

        data = img.get_fdata(
            dtype=np.float32
        )

        finite = bool(
            np.isfinite(data).all()
        )

        is_3d = (
            len(shape) == 3
        )

        # -------------------------------------------------------------
        # STATISTICS
        # -------------------------------------------------------------

        if finite and data.size > 0:

            vmin = float(
                np.min(data)
            )

            vmax = float(
                np.max(data)
            )

            mean = float(
                np.mean(data)
            )

            std = float(
                np.std(data)
            )

            p01 = float(
                np.percentile(
                    data,
                    1
                )
            )

            p99 = float(
                np.percentile(
                    data,
                    99
                )
            )

        else:

            vmin = np.nan
            vmax = np.nan
            mean = np.nan
            std = np.nan
            p01 = np.nan
            p99 = np.nan

        # -------------------------------------------------------------
        # SHA256
        # -------------------------------------------------------------

        sha = hashlib.sha256()

        with open(
            path,
            "rb"
        ) as f:

            for chunk in iter(
                lambda: f.read(1024 * 1024),
                b""
            ):

                sha.update(chunk)

        file_sha256 = sha.hexdigest()

        # -------------------------------------------------------------
        # VALIDATION
        # -------------------------------------------------------------

        valid_spacing = (
            len(zooms) >= 3
            and
            all(
                np.isfinite(zooms[:3])
            )
            and
            all(
                x > 0
                for x in zooms[:3]
            )
        )

        valid = (
            is_3d
            and
            finite
            and
            valid_spacing
        )

        rows.append({

            "class_id":
                item["class_id"],

            "class_name":
                item["class_name"],

            "patient_id":
                item["patient_id"],

            "path":
                str(path),

            "shape":
                str(shape),

            "x":
                shape[0] if len(shape) > 0 else np.nan,

            "y":
                shape[1] if len(shape) > 1 else np.nan,

            "z":
                shape[2] if len(shape) > 2 else np.nan,

            "spacing_x":
                zooms[0] if len(zooms) > 0 else np.nan,

            "spacing_y":
                zooms[1] if len(zooms) > 1 else np.nan,

            "spacing_z":
                zooms[2] if len(zooms) > 2 else np.nan,

            "dtype":
                dtype,

            "min":
                vmin,

            "max":
                vmax,

            "mean":
                mean,

            "std":
                std,

            "p01":
                p01,

            "p99":
                p99,

            "is_3d":
                is_3d,

            "finite":
                finite,

            "valid":
                valid,

            "file_size_mb":
                path.stat().st_size / 1024**2,

            "sha256":
                file_sha256
        })

        del data

    except Exception as e:

        failed.append({
            "path": str(path),
            "error": str(e)
        })

# =============================================================================
# DATAFRAME
# =============================================================================

audit_df = pd.DataFrame(
    rows
)

# =============================================================================
# PATIENT COUNTS
# =============================================================================

patient_counts = (
    audit_df
    .groupby("class_name")["patient_id"]
    .nunique()
    .to_dict()
)

volume_counts = (
    audit_df
    .groupby("class_name")
    .size()
    .to_dict()
)

# =============================================================================
# DUPLICATE HASH CHECK
# =============================================================================

duplicate_hashes = (
    audit_df[
        audit_df["sha256"].duplicated(
            keep=False
        )
    ]
)

# =============================================================================
# INVALID FILES
# =============================================================================

invalid_df = audit_df[
    ~audit_df["valid"]
].copy()

# =============================================================================
# SAVE LOCAL AUDIT
# =============================================================================

local_audit_csv = (
    AUDIT_DIR /
    "model3b_final_3class_ct_audit.csv"
)

audit_df.to_csv(
    local_audit_csv,
    index=False
)

# =============================================================================
# SUMMARY
# =============================================================================

summary = {

    "model":
        "Model 3B",

    "dataset":
        "3-class CT brain tumor dataset",

    "classes":
        CLASSES,

    "expected_patients":
        expected_patients,

    "actual_patients":
        {
            k: int(v)
            for k, v in patient_counts.items()
        },

    "volume_counts":
        {
            k: int(v)
            for k, v in volume_counts.items()
        },

    "total_volumes":
        int(len(audit_df)),

    "total_patients":
        int(
            audit_df[
                "patient_id"
            ].nunique()
        ),

    "invalid_volumes":
        int(len(invalid_df)),

    "failed_volumes":
        int(len(failed)),

    "duplicate_hash_groups":
        int(
            duplicate_hashes[
                "sha256"
            ].nunique()
        ),

    "raw_ct_modified":
        False,

    "audit_timestamp":
        datetime.now().isoformat()
}

local_summary = (
    AUDIT_DIR /
    "model3b_final_3class_ct_audit_summary.json"
)

with open(
    local_summary,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# SAVE FAILED LIST
# =============================================================================

if failed:

    with open(
        AUDIT_DIR /
        "model3b_final_3class_failed.json",
        "w"
    ) as f:

        json.dump(
            failed,
            f,
            indent=2
        )

# =============================================================================
# COPY IMPORTANT INFORMATION TO DRIVE
# =============================================================================

print("\n")
print("=" * 80)
print("SAVING IMPORTANT DATA TO DRIVE")
print("=" * 80)

# Audit
shutil.copy2(
    local_audit_csv,
    DRIVE_AUDIT /
    local_audit_csv.name
)

shutil.copy2(
    local_summary,
    DRIVE_AUDIT /
    local_summary.name
)

# Existing manifests
if MANIFEST_DIR.exists():

    for file in MANIFEST_DIR.glob(
        "*.csv"
    ):

        shutil.copy2(
            file,
            DRIVE_MANIFEST /
            file.name
        )

    for file in MANIFEST_DIR.glob(
        "*.json"
    ):

        shutil.copy2(
            file,
            DRIVE_MANIFEST /
            file.name
        )

# =============================================================================
# SAVE DATASET CONFIG
# =============================================================================

config = {

    "model":
        "Model 3B",

    "classes": {
        "0": "Meningioma",
        "1": "Pituitary",
        "2": "Brain_Metastasis"
    },

    "ct_only":
        True,

    "mri_used":
        False,

    "segmentation_used":
        False,

    "augmentation_used":
        False,

    "raw_ct_modified":
        False,

    "expected_patients":
        expected_patients,

    "expected_total_patients":
        124,

    "drive_policy":
        "Save manifests, audits, processed data and models only. "
        "Do not copy raw CT volumes to Drive.",

    "saved_at":
        datetime.now().isoformat()
}

config_path = (
    DRIVE_CONFIG /
    "model3b_dataset_config.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

# =============================================================================
# FINAL OUTPUT
# =============================================================================

print("\n")
print("=" * 80)
print("FINAL 3-CLASS AUDIT")
print("=" * 80)

print(
    "\nPatients:"
)

for cls in CLASSES.values():

    print(
        f"{cls:20s}: "
        f"{patient_counts.get(cls, 0)}"
    )

print(
    "\nTotal patients:",
    audit_df[
        "patient_id"
    ].nunique()
)

print(
    "Total CT volumes:",
    len(audit_df)
)

print(
    "Invalid volumes:",
    len(invalid_df)
)

print(
    "Failed volumes:",
    len(failed)
)

print(
    "Duplicate hash groups:",
    duplicate_hashes[
        "sha256"
    ].nunique()
)

print(
    "\nDrive backup:"
)

print(
    DRIVE_ROOT
)

print(
    "\n✅ Manifests + audit + configuration saved to Drive."
)

print("=" * 80)

# =============================================================================
# SAFETY STOP
# =============================================================================

if (
    len(invalid_df) > 0
    or
    len(failed) > 0
):

    print(
        "\n⚠️ AUDIT HAS PROBLEMS."
    )

    print(
        "DO NOT START PREPROCESSING YET."
    )

else:

    print(
        "\n✅ ALL CT VOLUMES PASSED BASIC AUDIT."
    )

    print(
        "NEXT: FAST 3D → 2D PREPROCESSING."
    )

print("=" * 80)

MODEL 3B - STEP 6
FINAL 3-CLASS CT AUDIT

CT volumes discovered: 170
[1/170] Meningioma / sub-13
[2/170] Meningioma / sub-17
[3/170] Meningioma / sub-10
[4/170] Meningioma / sub-08
[5/170] Meningioma / sub-15
[6/170] Meningioma / sub-09
[7/170] Meningioma / sub-12
[8/170] Meningioma / sub-04
[9/170] Meningioma / sub-06
[10/170] Meningioma / sub-16
[11/170] Meningioma / sub-20
[12/170] Meningioma / sub-03
[13/170] Meningioma / sub-19
[14/170] Meningioma / sub-14
[15/170] Meningioma / sub-05
[16/170] Meningioma / sub-18
[17/170] Meningioma / sub-11
[18/170] Meningioma / sub-07
[19/170] Meningioma / sub-02
[20/170] Meningioma / sub-01
[21/170] Pituitary / sub-13
[22/170] Pituitary / sub-56
[23/170] Pituitary / sub-17
[24/170] Pituitary / sub-10
[25/170] Pituitary / sub-08
[26/170] Pituitary / sub-38
[27/170] Pituitary / sub-44
[28/170] Pituitary / sub-42
[29/170] Pituitary / sub-46
[30/170] Pituitary / sub-15
[31/170] Pituitary / sub-09
[32/170] Pituitary / sub-27
[33/170] Pituitary / sub

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# MODEL 3B - STEP 6-FAST
# FAST 3-CLASS CT AUDIT
# HEADER/GEOMETRY AUDIT - NO FULL VOLUME LOADING
# =============================================================================

import json
import hashlib
import pandas as pd
import nibabel as nib

from pathlib import Path
from datetime import datetime

# =============================================================================
# PATHS
# =============================================================================

ROOT = Path("/content/model3b")
RAW_DIR = ROOT / "raw"
MANIFEST_DIR = ROOT / "manifests"
AUDIT_DIR = ROOT / "audit"

DRIVE_ROOT = Path("/content/drive/MyDrive/Model3B")
DRIVE_AUDIT = DRIVE_ROOT / "audits"
DRIVE_MANIFEST = DRIVE_ROOT / "manifests"
DRIVE_CONFIG = DRIVE_ROOT / "configs"

for p in [
    AUDIT_DIR,
    DRIVE_AUDIT,
    DRIVE_MANIFEST,
    DRIVE_CONFIG
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

# =============================================================================
# FINAL 3 CLASSES
# =============================================================================

CLASSES = {
    0: "Meningioma",
    1: "Pituitary",
    2: "Brain_Metastasis"
}

EXPECTED_PATIENTS = {
    "Meningioma": 20,
    "Pituitary": 60,
    "Brain_Metastasis": 44
}

EXPECTED_VOLUMES = {
    "Meningioma": 20,
    "Pituitary": 60,
    "Brain_Metastasis": 91
}

print("=" * 80)
print("MODEL 3B - STEP 6-FAST")
print("3-CLASS CT HEADER / GEOMETRY AUDIT")
print("=" * 80)

# =============================================================================
# DISCOVER FILES
# =============================================================================

files = []

for class_id, class_name in CLASSES.items():

    class_dir = RAW_DIR / class_name

    if not class_dir.exists():
        print(
            f"⚠️ Missing directory: {class_dir}"
        )
        continue

    for path in sorted(
        class_dir.rglob("*.nii.gz")
    ):

        files.append({
            "class_id": class_id,
            "class_name": class_name,
            "path": path
        })

print(
    f"\nCT volumes discovered: {len(files)}"
)

print(
    "Expected CT volumes: 171"
)

# =============================================================================
# AUDIT
# =============================================================================

rows = []
failed = []

for i, item in enumerate(
    files,
    1
):

    path = item["path"]

    print(
        f"[{i}/{len(files)}] "
        f"{item['class_name']} / "
        f"{path.parent.name} / "
        f"{path.name}"
    )

    try:

        # ---------------------------------------------------------------------
        # HEADER ONLY
        # ---------------------------------------------------------------------

        img = nib.load(
            str(path)
        )

        shape = tuple(
            int(x)
            for x in img.shape
        )

        zooms = tuple(
            float(x)
            for x in img.header.get_zooms()
        )

        dtype = str(
            img.header.get_data_dtype()
        )

        ndim = len(shape)

        is_3d = (
            ndim == 3
        )

        valid_shape = (
            is_3d
            and
            all(
                x > 0
                for x in shape
            )
        )

        valid_spacing = (
            len(zooms) >= 3
            and
            all(
                x > 0
                for x in zooms[:3]
            )
        )

        # ---------------------------------------------------------------------
        # PATIENT ID
        # ---------------------------------------------------------------------

        patient_id = path.parent.name

        # ---------------------------------------------------------------------
        # LIGHTWEIGHT FILE HASH
        # ---------------------------------------------------------------------
        # Hash compressed file in chunks.
        # This does NOT decompress the NIfTI volume.

        sha = hashlib.sha256()

        with open(
            path,
            "rb"
        ) as f:

            while True:

                chunk = f.read(
                    4 * 1024 * 1024
                )

                if not chunk:
                    break

                sha.update(chunk)

        file_sha256 = sha.hexdigest()

        # ---------------------------------------------------------------------
        # VALID
        # ---------------------------------------------------------------------

        valid = (
            valid_shape
            and
            valid_spacing
        )

        rows.append({

            "class_id":
                item["class_id"],

            "class_name":
                item["class_name"],

            "patient_id":
                patient_id,

            "path":
                str(path),

            "shape":
                str(shape),

            "x":
                shape[0] if len(shape) > 0 else None,

            "y":
                shape[1] if len(shape) > 1 else None,

            "z":
                shape[2] if len(shape) > 2 else None,

            "spacing_x":
                zooms[0] if len(zooms) > 0 else None,

            "spacing_y":
                zooms[1] if len(zooms) > 1 else None,

            "spacing_z":
                zooms[2] if len(zooms) > 2 else None,

            "dtype":
                dtype,

            "ndim":
                ndim,

            "is_3d":
                is_3d,

            "valid_shape":
                valid_shape,

            "valid_spacing":
                valid_spacing,

            "valid":
                valid,

            "file_size_mb":
                path.stat().st_size / 1024**2,

            "sha256":
                file_sha256
        })

    except Exception as e:

        failed.append({
            "path": str(path),
            "error": str(e)
        })

# =============================================================================
# DATAFRAME
# =============================================================================

audit_df = pd.DataFrame(
    rows
)

# =============================================================================
# CLASS COUNTS
# =============================================================================

print("\n")
print("=" * 80)
print("CLASS COUNTS")
print("=" * 80)

class_counts = (
    audit_df
    .groupby("class_name")
    .agg(
        volumes=("path", "count"),
        patients=("patient_id", "nunique")
    )
)

print(
    class_counts
)

# =============================================================================
# FIND MISSING VOLUMES
# =============================================================================

print("\n")
print("=" * 80)
print("EXPECTED VS ACTUAL")
print("=" * 80)

for class_name in CLASSES.values():

    actual_v = int(
        (
            audit_df["class_name"]
            == class_name
        ).sum()
    )

    actual_p = int(
        audit_df[
            audit_df["class_name"]
            == class_name
        ]["patient_id"].nunique()
    )

    expected_v = EXPECTED_VOLUMES[
        class_name
    ]

    expected_p = EXPECTED_PATIENTS[
        class_name
    ]

    print(
        f"\n{class_name}"
    )

    print(
        f"  Patients: "
        f"{actual_p}/{expected_p}"
    )

    print(
        f"  Volumes : "
        f"{actual_v}/{expected_v}"
    )

# =============================================================================
# BRAIN METASTASIS - EXPECTED PATIENTS
# =============================================================================

brain_df = audit_df[
    audit_df["class_name"]
    == "Brain_Metastasis"
]

print("\n")
print("=" * 80)
print("BRAIN METASTASIS PATIENT DISTRIBUTION")
print("=" * 80)

print(
    brain_df[
        "patient_id"
    ].value_counts().sort_index()
)

# =============================================================================
# DUPLICATE HASHES
# =============================================================================

duplicate_hash_df = audit_df[
    audit_df["sha256"].duplicated(
        keep=False
    )
].sort_values(
    "sha256"
)

print("\n")
print("=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)

print(
    "Duplicate hash groups:",
    duplicate_hash_df[
        "sha256"
    ].nunique()
)

# =============================================================================
# INVALID
# =============================================================================

invalid_df = audit_df[
    ~audit_df["valid"]
]

print(
    "Invalid volumes:",
    len(invalid_df)
)

# =============================================================================
# SHAPE DISTRIBUTION
# =============================================================================

print("\n")
print("=" * 80)
print("SHAPE DISTRIBUTION")
print("=" * 80)

print(
    audit_df[
        "shape"
    ].value_counts().head(20)
)

# =============================================================================
# SPACING DISTRIBUTION
# =============================================================================

print("\n")
print("=" * 80)
print("VOXEL SPACING")
print("=" * 80)

print(
    audit_df[
        [
            "spacing_x",
            "spacing_y",
            "spacing_z"
        ]
    ].describe()
)

# =============================================================================
# SAVE AUDIT CSV
# =============================================================================

audit_csv = (
    AUDIT_DIR /
    "model3b_step6_fast_3class_ct_audit.csv"
)

audit_df.to_csv(
    audit_csv,
    index=False
)

# =============================================================================
# SAVE FAILED
# =============================================================================

if failed:

    failed_path = (
        AUDIT_DIR /
        "model3b_step6_fast_failed.json"
    )

    with open(
        failed_path,
        "w"
    ) as f:

        json.dump(
            failed,
            f,
            indent=2
        )

# =============================================================================
# SUMMARY
# =============================================================================

summary = {

    "model":
        "Model 3B",

    "classes":
        CLASSES,

    "expected_total_volumes":
        171,

    "actual_total_volumes":
        int(len(audit_df)),

    "expected_patients":
        EXPECTED_PATIENTS,

    "expected_volumes":
        EXPECTED_VOLUMES,

    "actual_class_counts":
        {
            cls: {
                "volumes": int(
                    (
                        audit_df[
                            "class_name"
                        ] == cls
                    ).sum()
                ),
                "patients": int(
                    audit_df[
                        audit_df[
                            "class_name"
                        ] == cls
                    ]["patient_id"].nunique()
                )
            }
            for cls in CLASSES.values()
        },

    "invalid_volumes":
        int(len(invalid_df)),

    "failed_volumes":
        int(len(failed)),

    "duplicate_hash_groups":
        int(
            duplicate_hash_df[
                "sha256"
            ].nunique()
        ),

    "raw_ct_modified":
        False,

    "audit_timestamp":
        datetime.now().isoformat()
}

summary_path = (
    AUDIT_DIR /
    "model3b_step6_fast_3class_summary.json"
)

with open(
    summary_path,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# BACKUP TO DRIVE
# =============================================================================

print("\n")
print("=" * 80)
print("BACKING UP AUDIT TO DRIVE")
print("=" * 80)

import shutil

shutil.copy2(
    audit_csv,
    DRIVE_AUDIT /
    audit_csv.name
)

shutil.copy2(
    summary_path,
    DRIVE_AUDIT /
    summary_path.name
)

# Existing manifests
if MANIFEST_DIR.exists():

    for p in MANIFEST_DIR.glob("*.csv"):

        shutil.copy2(
            p,
            DRIVE_MANIFEST /
            p.name
        )

    for p in MANIFEST_DIR.glob("*.json"):

        shutil.copy2(
            p,
            DRIVE_MANIFEST /
            p.name
        )

# Dataset configuration
config = {

    "model":
        "Model 3B",

    "dataset":
        "3-class brain tumor CT",

    "classes":
        {
            "0": "Meningioma",
            "1": "Pituitary",
            "2": "Brain_Metastasis"
        },

    "ct_only":
        True,

    "mri":
        False,

    "segmentation":
        False,

    "augmentation":
        False,

    "expected_patients":
        124,

    "expected_volumes":
        171,

    "raw_ct_policy":
        "Raw CT remains in runtime; "
        "only processed data and audit artifacts "
        "will be saved to Drive.",

    "created":
        datetime.now().isoformat()
}

config_path = (
    DRIVE_CONFIG /
    "model3b_3class_dataset_config.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

# =============================================================================
# FINAL
# =============================================================================

print("\n")
print("=" * 80)
print("MODEL 3B - STEP 6-FAST COMPLETE")
print("=" * 80)

print(
    "\nTotal discovered:",
    len(audit_df)
)

print(
    "Expected:",
    171
)

print(
    "Invalid:",
    len(invalid_df)
)

print(
    "Failed:",
    len(failed)
)

print(
    "Duplicate groups:",
    duplicate_hash_df[
        "sha256"
    ].nunique()
)

print(
    "\nAudit saved:"
)

print(
    audit_csv
)

print(
    "\nDrive backup:"
)

print(
    DRIVE_AUDIT
)

print("=" * 80)

if (
    len(audit_df) == 171
    and
    len(invalid_df) == 0
    and
    len(failed) == 0
    and
    duplicate_hash_df[
        "sha256"
    ].nunique() == 0
):

    print(
        "\n✅ ALL 171 CT VOLUMES PASSED."
    )

    print(
        "NEXT → FAST PREPROCESSING"
    )

else:

    print(
        "\n⚠️ AUDIT DOES NOT MATCH EXPECTED DATA."
    )

    print(
        "DO NOT PREPROCESS YET."
    )

print("=" * 80)

MODEL 3B - STEP 6-FAST
3-CLASS CT HEADER / GEOMETRY AUDIT

CT volumes discovered: 170
Expected CT volumes: 171
[1/170] Meningioma / sub-01 / CT-BRAIN.nii.gz
[2/170] Meningioma / sub-02 / CT-BRAIN.nii.gz
[3/170] Meningioma / sub-03 / CT-BRAIN.nii.gz
[4/170] Meningioma / sub-04 / CT-BRAIN.nii.gz
[5/170] Meningioma / sub-05 / CT-BRAIN.nii.gz
[6/170] Meningioma / sub-06 / CT-BRAIN.nii.gz
[7/170] Meningioma / sub-07 / CT-BRAIN.nii.gz
[8/170] Meningioma / sub-08 / CT-BRAIN.nii.gz
[9/170] Meningioma / sub-09 / CT-BRAIN.nii.gz
[10/170] Meningioma / sub-10 / CT-BRAIN.nii.gz
[11/170] Meningioma / sub-11 / CT-BRAIN.nii.gz
[12/170] Meningioma / sub-12 / CT-BRAIN.nii.gz
[13/170] Meningioma / sub-13 / CT-BRAIN.nii.gz
[14/170] Meningioma / sub-14 / CT-BRAIN.nii.gz
[15/170] Meningioma / sub-15 / CT-BRAIN.nii.gz
[16/170] Meningioma / sub-16 / CT-BRAIN.nii.gz
[17/170] Meningioma / sub-17 / CT-BRAIN.nii.gz
[18/170] Meningioma / sub-18 / CT-BRAIN.nii.gz
[19/170] Meningioma / sub-19 / CT-BRAIN.nii.gz
[20/1

In [ ]:
# =============================================================================
# MODEL 3B - STEP 6B
# FIND MISSING BRAIN-METASTASIS CT VOLUME
# =============================================================================

import pandas as pd
from pathlib import Path

ROOT = Path("/content/model3b")
RAW = ROOT / "raw" / "Brain_Metastasis"

manifest = (
    ROOT /
    "manifests" /
    "brain_metastasis_acquisition_manifest.csv"
)

df = pd.read_csv(manifest)

print("=" * 80)
print("MODEL 3B - STEP 6B")
print("BRAIN METASTASIS MISSING CT CHECK")
print("=" * 80)

print("\nManifest rows:", len(df))

# Expected files from manifest
expected = set()

for _, row in df.iterrows():

    patient = str(
        row["patient_id"]
    )

    ct_number = row["ct_number"]

    if pd.notna(ct_number):
        ct_number = int(ct_number)

        filename = (
            f"{patient}_CT_{ct_number}.nii.gz"
        )
    else:
        filename = Path(
            row["archive_path"]
        ).name

    expected.add(
        f"{patient}/{filename}"
    )

# Actual files
actual = set()

for p in RAW.rglob("*.nii.gz"):

    actual.add(
        f"{p.parent.name}/{p.name}"
    )

missing = sorted(
    expected - actual
)

extra = sorted(
    actual - expected
)

print("\nExpected from manifest:", len(expected))
print("Actually present:", len(actual))

print("\nMissing files:", len(missing))

for x in missing:
    print("  MISSING:", x)

print("\nExtra files:", len(extra))

for x in extra:
    print("  EXTRA:", x)

print("\n" + "=" * 80)

if len(missing) == 1:

    print("⚠️ Exactly ONE CT volume is missing.")

    print(
        "\nRun the acquisition-resume code next "
        "to download ONLY this missing volume."
    )

elif len(missing) == 0:

    print(
        "✅ No missing files according to the manifest."
    )

else:

    print(
        "⚠️ Multiple files are missing."
    )

print("=" * 80)

MODEL 3B - STEP 6B
BRAIN METASTASIS MISSING CT CHECK

Manifest rows: 91

Expected from manifest: 90
Actually present: 90

Missing files: 0

Extra files: 0

✅ No missing files according to the manifest.


In [ ]:
# =============================================================================
# MODEL 3B - STEP 7
# FAST CT PREPROCESSING + CONTROLLED 2D SLICE GENERATION
# =============================================================================

import os
import json
import random
import hashlib
import numpy as np
import pandas as pd
import nibabel as nib

from pathlib import Path
from datetime import datetime
from scipy.ndimage import zoom

# =============================================================================
# CONFIG
# =============================================================================

ROOT = Path("/content/model3b")
RAW_DIR = ROOT / "raw"
MANIFEST_DIR = ROOT / "manifests"
PROCESSED_DIR = ROOT / "processed" / "3class"
AUDIT_DIR = ROOT / "audit"

DRIVE_ROOT = Path("/content/drive/MyDrive/Model3B")
DRIVE_PROCESSED = DRIVE_ROOT / "processed"
DRIVE_MANIFEST = DRIVE_ROOT / "manifests"
DRIVE_AUDIT = DRIVE_ROOT / "audits"
DRIVE_CONFIG = DRIVE_ROOT / "configs"

for p in [
    PROCESSED_DIR,
    MANIFEST_DIR,
    AUDIT_DIR,
    DRIVE_PROCESSED,
    DRIVE_MANIFEST,
    DRIVE_AUDIT,
    DRIVE_CONFIG
]:
    p.mkdir(parents=True, exist_ok=True)

# =============================================================================
# FINAL CLASSES
# =============================================================================

CLASSES = {
    0: "Meningioma",
    1: "Pituitary",
    2: "Brain_Metastasis"
}

# =============================================================================
# FAST SETTINGS
# =============================================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 15 slices per CT volume
SLICES_PER_VOLUME = 15

# Resize final image
IMAGE_SIZE = 224

# CT window
HU_MIN = -1000.0
HU_MAX = 2000.0

# Skip outer slices
EDGE_EXCLUSION = 5

# Minimum fraction of non-background pixels
CONTENT_THRESHOLD = 0.05

print("=" * 80)
print("MODEL 3B - STEP 7")
print("FAST CT PREPROCESSING + 2D SLICE GENERATION")
print("=" * 80)

# =============================================================================
# DISCOVER UNIQUE CT VOLUMES
# =============================================================================

volume_records = []

for class_id, class_name in CLASSES.items():

    class_dir = RAW_DIR / class_name

    if not class_dir.exists():
        continue

    for path in sorted(
        class_dir.rglob("*.nii.gz")
    ):

        # ---------------------------------------------------------------------
        # Patient
        # ---------------------------------------------------------------------

        patient_id = path.parent.name

        volume_records.append({

            "class_id": class_id,
            "class_name": class_name,
            "patient_id": patient_id,
            "path": path
        })

# -------------------------------------------------------------------------
# REMOVE DUPLICATE PATHS
# -------------------------------------------------------------------------

unique = {}

for row in volume_records:
    unique[str(row["path"])] = row

volume_records = list(
    unique.values()
)

print(
    "\nUnique CT volumes:",
    len(volume_records)
)

print(
    "Patients:",
    len(
        set(
            x["patient_id"]
            for x in volume_records
        )
    )
)

# =============================================================================
# PATIENT-LEVEL SPLIT
# =============================================================================

patient_class = {}

for row in volume_records:

    patient_class[
        row["patient_id"]
    ] = row["class_id"]

patients = sorted(
    patient_class.keys()
)

# Stratified patient split
train_patients = []
val_patients = []
test_patients = []

for class_id in sorted(
    set(patient_class.values())
):

    class_patients = [
        p for p in patients
        if patient_class[p] == class_id
    ]

    rng = random.Random(
        SEED + class_id
    )

    rng.shuffle(
        class_patients
    )

    n = len(class_patients)

    n_test = max(
        1,
        round(n * 0.15)
    )

    n_val = max(
        1,
        round(n * 0.15)
    )

    test = class_patients[
        :n_test
    ]

    val = class_patients[
        n_test:n_test+n_val
    ]

    train = class_patients[
        n_test+n_val:
    ]

    train_patients.extend(train)
    val_patients.extend(val)
    test_patients.extend(test)

print("\n")
print("=" * 80)
print("PATIENT SPLIT")
print("=" * 80)

print(
    "Train patients:",
    len(train_patients)
)

print(
    "Validation patients:",
    len(val_patients)
)

print(
    "Test patients:",
    len(test_patients)
)

# =============================================================================
# SPLIT LOOKUP
# =============================================================================

split_lookup = {}

for p in train_patients:
    split_lookup[p] = "train"

for p in val_patients:
    split_lookup[p] = "validation"

for p in test_patients:
    split_lookup[p] = "test"

assert (
    len(set(train_patients) &
        set(val_patients)) == 0
)

assert (
    len(set(train_patients) &
        set(test_patients)) == 0
)

assert (
    len(set(val_patients) &
        set(test_patients)) == 0
)

print(
    "✅ No patient leakage."
)

# =============================================================================
# SAVE PATIENT SPLIT
# =============================================================================

split_rows = []

for p in patients:

    split_rows.append({

        "patient_id": p,

        "class_id":
            patient_class[p],

        "class_name":
            CLASSES[
                patient_class[p]
            ],

        "split":
            split_lookup[p]
    })

split_df = pd.DataFrame(
    split_rows
)

split_path = (
    MANIFEST_DIR /
    "model3b_step7_patient_split.csv"
)

split_df.to_csv(
    split_path,
    index=False
)

# =============================================================================
# PREPROCESS FUNCTION
# =============================================================================

def preprocess_volume(
    path
):

    img = nib.load(
        str(path)
    )

    data = img.get_fdata(
        dtype=np.float32
    )

    # -------------------------------------------------------------
    # Remove NaN / Inf
    # -------------------------------------------------------------

    data = np.nan_to_num(
        data,
        nan=0.0,
        posinf=HU_MAX,
        neginf=HU_MIN
    )

    # -------------------------------------------------------------
    # CT WINDOW
    # -------------------------------------------------------------

    data = np.clip(
        data,
        HU_MIN,
        HU_MAX
    )

    # -------------------------------------------------------------
    # NORMALIZE [0,1]
    # -------------------------------------------------------------

    data = (
        data - HU_MIN
    ) / (
        HU_MAX - HU_MIN
    )

    return data


# =============================================================================
# RESIZE SLICE
# =============================================================================

def resize_slice(
    image
):

    h, w = image.shape

    scale_y = IMAGE_SIZE / h
    scale_x = IMAGE_SIZE / w

    out = zoom(
        image,
        (
            scale_y,
            scale_x
        ),
        order=1
    )

    # Safety crop/pad
    result = np.zeros(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        ),
        dtype=np.float32
    )

    hh = min(
        IMAGE_SIZE,
        out.shape[0]
    )

    ww = min(
        IMAGE_SIZE,
        out.shape[1]
    )

    result[
        :hh,
        :ww
    ] = out[
        :hh,
        :ww
    ]

    return result


# =============================================================================
# SLICE SELECTION
# =============================================================================

def select_slices(
    volume
):

    z = volume.shape[2]

    start = EDGE_EXCLUSION
    end = z - EDGE_EXCLUSION

    if end <= start:
        start = 0
        end = z

    indices = np.arange(
        start,
        end
    )

    candidates = []

    for idx in indices:

        sl = volume[
            :,
            :,
            idx
        ]

        # Content fraction
        content = np.mean(
            sl > 0.05
        )

        if content >= CONTENT_THRESHOLD:

            candidates.append(
                (
                    idx,
                    float(content)
                )
            )

    # -------------------------------------------------------------------------
    # If too few candidates, use available central slices
    # -------------------------------------------------------------------------

    if len(candidates) < SLICES_PER_VOLUME:

        center = (
            start + end
        ) // 2

        fallback = np.linspace(
            start,
            end - 1,
            min(
                SLICES_PER_VOLUME,
                max(
                    1,
                    end - start
                )
            ),
            dtype=int
        )

        selected = list(
            dict.fromkeys(
                fallback.tolist()
            )
        )

    else:

        # Sort by anatomical content
        candidates.sort(
            key=lambda x: x[1],
            reverse=True
        )

        # Select evenly among high-content candidates
        top = candidates[
            :min(
                len(candidates),
                SLICES_PER_VOLUME * 4
            )
        ]

        top_indices = [
            x[0]
            for x in top
        ]

        positions = np.linspace(
            0,
            len(top_indices) - 1,
            SLICES_PER_VOLUME,
            dtype=int
        )

        selected = [
            top_indices[i]
            for i in positions
        ]

    return sorted(
        set(selected)
    )


# =============================================================================
# PROCESS
# =============================================================================

slice_rows = []

total_volumes = len(
    volume_records
)

print("\n")
print("=" * 80)
print("PROCESSING CT VOLUMES")
print("=" * 80)

for vnum, row in enumerate(
    volume_records,
    1
):

    path = row["path"]

    patient_id = row[
        "patient_id"
    ]

    class_id = row[
        "class_id"
    ]

    class_name = row[
        "class_name"
    ]

    split = split_lookup[
        patient_id
    ]

    print(
        f"[{vnum}/{total_volumes}] "
        f"{class_name} / "
        f"{patient_id}"
    )

    try:

        volume = preprocess_volume(
            path
        )

        selected = select_slices(
            volume
        )

        # If fewer than 15, evenly sample with replacement
        if len(selected) < SLICES_PER_VOLUME:

            selected = np.linspace(
                0,
                volume.shape[2] - 1,
                SLICES_PER_VOLUME,
                dtype=int
            ).tolist()

        # Exactly 15
        selected = selected[
            :SLICES_PER_VOLUME
        ]

        # -------------------------------------------------------------
        # Create patient output directory
        # -------------------------------------------------------------

        out_dir = (
            PROCESSED_DIR /
            split /
            class_name /
            patient_id
        )

        out_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        # -------------------------------------------------------------
        # Generate slices
        # -------------------------------------------------------------

        for slice_no, z in enumerate(
            selected,
            1
        ):

            image = volume[
                :,
                :,
                z
            ]

            image = resize_slice(
                image
            )

            image = np.clip(
                image,
                0.0,
                1.0
            ).astype(
                np.float32
            )

            filename = (
                f"{patient_id}_"
                f"slice_{slice_no:02d}.npy"
            )

            output = (
                out_dir /
                filename
            )

            np.save(
                output,
                image
            )

            # -------------------------------------------------------------
            # Hash processed slice
            # -------------------------------------------------------------

            sha = hashlib.sha256(
                image.tobytes()
            ).hexdigest()

            slice_rows.append({

                "class_id":
                    class_id,

                "class_name":
                    class_name,

                "patient_id":
                    patient_id,

                "split":
                    split,

                "slice_number":
                    slice_no,

                "source_z":
                    int(z),

                "height":
                    IMAGE_SIZE,

                "width":
                    IMAGE_SIZE,

                "min":
                    float(image.min()),

                "max":
                    float(image.max()),

                "mean":
                    float(image.mean()),

                "sha256":
                    sha,

                "path":
                    str(output)
            })

        del volume

    except Exception as e:

        print(
            f"    ❌ FAILED: {e}"
        )

# =============================================================================
# MANIFEST
# =============================================================================

slice_df = pd.DataFrame(
    slice_rows
)

slice_manifest = (
    MANIFEST_DIR /
    "model3b_step7_3class_slice_manifest.csv"
)

slice_df.to_csv(
    slice_manifest,
    index=False
)

# =============================================================================
# VALIDATION
# =============================================================================

print("\n")
print("=" * 80)
print("STEP 7 VALIDATION")
print("=" * 80)

print(
    "\nTotal slices:",
    len(slice_df)
)

print(
    "\nSplit distribution:"
)

print(
    slice_df[
        "split"
    ].value_counts()
)

print(
    "\nClass distribution:"
)

print(
    slice_df[
        "class_name"
    ].value_counts()
)

print(
    "\nPatient distribution:"
)

print(
    slice_df
    .groupby(
        [
            "split",
            "class_name"
        ]
    )[
        "patient_id"
    ]
    .nunique()
)

# =============================================================================
# PATIENT LEAKAGE
# =============================================================================

train_p = set(
    slice_df[
        slice_df["split"] == "train"
    ]["patient_id"]
)

val_p = set(
    slice_df[
        slice_df["split"] == "validation"
    ]["patient_id"]
)

test_p = set(
    slice_df[
        slice_df["split"] == "test"
    ]["patient_id"]
)

print(
    "\nTrain ∩ Validation:",
    len(train_p & val_p)
)

print(
    "Train ∩ Test:",
    len(train_p & test_p)
)

print(
    "Validation ∩ Test:",
    len(val_p & test_p)
)

# =============================================================================
# DUPLICATES
# =============================================================================

duplicates = (
    slice_df[
        slice_df["sha256"].duplicated(
            keep=False
        )
    ]
)

print(
    "\nDuplicate processed hashes:",
    duplicates[
        "sha256"
    ].nunique()
)

# =============================================================================
# NORMALIZATION
# =============================================================================

print(
    "\nGlobal minimum:",
    slice_df["min"].min()
)

print(
    "Global maximum:",
    slice_df["max"].max()
)

# =============================================================================
# SAVE AUDIT
# =============================================================================

audit = {

    "step":
        "7",

    "classes":
        CLASSES,

    "total_source_volumes":
        len(volume_records),

    "total_processed_slices":
        len(slice_df),

    "slices_per_volume":
        SLICES_PER_VOLUME,

    "image_size":
        IMAGE_SIZE,

    "hu_min":
        HU_MIN,

    "hu_max":
        HU_MAX,

    "edge_exclusion":
        EDGE_EXCLUSION,

    "content_threshold":
        CONTENT_THRESHOLD,

    "seed":
        SEED,

    "patient_leakage":
        (
            len(train_p & val_p)
            +
            len(train_p & test_p)
            +
            len(val_p & test_p)
        ),

    "duplicate_hash_groups":
        int(
            duplicates[
                "sha256"
            ].nunique()
        ),

    "global_min":
        float(
            slice_df["min"].min()
        ),

    "global_max":
        float(
            slice_df["max"].max()
        ),

    "created":
        datetime.now().isoformat()
}

audit_path = (
    AUDIT_DIR /
    "model3b_step7_preprocessing_audit.json"
)

with open(
    audit_path,
    "w"
) as f:

    json.dump(
        audit,
        f,
        indent=2
    )

# =============================================================================
# BACKUP MANIFEST + AUDIT TO DRIVE
# =============================================================================

import shutil

shutil.copy2(
    slice_manifest,
    DRIVE_MANIFEST /
    slice_manifest.name
)

shutil.copy2(
    split_path,
    DRIVE_MANIFEST /
    split_path.name
)

shutil.copy2(
    audit_path,
    DRIVE_AUDIT /
    audit_path.name
)

# =============================================================================
# CONFIG
# =============================================================================

config = {

    "classes":
        CLASSES,

    "image_size":
        IMAGE_SIZE,

    "hu_window":
        [
            HU_MIN,
            HU_MAX
        ],

    "slices_per_volume":
        SLICES_PER_VOLUME,

    "edge_exclusion":
        EDGE_EXCLUSION,

    "content_threshold":
        CONTENT_THRESHOLD,

    "augmentation":
        False,

    "segmentation":
        False,

    "normalization":
        "[0,1]",

    "raw_ct_modified":
        False
}

config_path = (
    DRIVE_CONFIG /
    "model3b_step7_preprocessing_config.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

# =============================================================================
# FINAL
# =============================================================================

print("\n")
print("=" * 80)
print("MODEL 3B - STEP 7 COMPLETE")
print("=" * 80)

print(
    "\nProcessed slices:",
    len(slice_df)
)

print(
    "Processed data:",
    PROCESSED_DIR
)

print(
    "\nManifest:",
    slice_manifest
)

print(
    "\nDrive:"
)

print(
    DRIVE_ROOT
)

print("\n✅ PREPROCESSING COMPLETE")

print(
    "\nNEXT → STEP 8 FINAL PROCESSED-DATA VERIFICATION"
)

print("=" * 80)

MODEL 3B - STEP 7
FAST CT PREPROCESSING + 2D SLICE GENERATION

Unique CT volumes: 170
Patients: 104


PATIENT SPLIT
Train patients: 72
Validation patients: 16
Test patients: 16
✅ No patient leakage.


PROCESSING CT VOLUMES
[1/170] Meningioma / sub-01
[2/170] Meningioma / sub-02
[3/170] Meningioma / sub-03
[4/170] Meningioma / sub-04
[5/170] Meningioma / sub-05
[6/170] Meningioma / sub-06
[7/170] Meningioma / sub-07
[8/170] Meningioma / sub-08
[9/170] Meningioma / sub-09
[10/170] Meningioma / sub-10
[11/170] Meningioma / sub-11
[12/170] Meningioma / sub-12
[13/170] Meningioma / sub-13
[14/170] Meningioma / sub-14
[15/170] Meningioma / sub-15
[16/170] Meningioma / sub-16
[17/170] Meningioma / sub-17
[18/170] Meningioma / sub-18
[19/170] Meningioma / sub-19
[20/170] Meningioma / sub-20
[21/170] Pituitary / sub-01
[22/170] Pituitary / sub-02
[23/170] Pituitary / sub-03
[24/170] Pituitary / sub-04
[25/170] Pituitary / sub-05
[26/170] Pituitary / sub-06
[27/170] Pituitary / sub-07
[28/170] P

In [ ]:
# =============================================================================
# MODEL 3B - STEP 8
# FINAL PROCESSED-DATA VERIFICATION
# =============================================================================

import os
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path

ROOT = Path("/content/model3b")
PROCESSED_DIR = ROOT / "processed" / "3class"
MANIFEST_PATH = (
    ROOT /
    "manifests" /
    "model3b_step7_3class_slice_manifest.csv"
)

print("=" * 80)
print("MODEL 3B - STEP 8")
print("FINAL PROCESSED-DATA VERIFICATION")
print("=" * 80)

# =============================================================================
# LOAD MANIFEST
# =============================================================================

df = pd.read_csv(
    MANIFEST_PATH
)

print(
    "\nManifest rows:",
    len(df)
)

print(
    "Unique patients:",
    df["patient_id"].nunique()
)

# =============================================================================
# FILE EXISTENCE
# =============================================================================

print("\n" + "=" * 80)
print("1. FILE EXISTENCE")
print("=" * 80)

missing = []

for path in df["path"]:

    if not Path(path).exists():
        missing.append(path)

print(
    "Expected files:",
    len(df)
)

print(
    "Missing files:",
    len(missing)
)

if missing:

    for x in missing[:20]:
        print("MISSING:", x)

else:

    print(
        "✅ All processed files exist."
    )

# =============================================================================
# LOAD AND VALIDATE ARRAYS
# =============================================================================

print("\n" + "=" * 80)
print("2. NUMERICAL VALIDATION")
print("=" * 80)

invalid = []
mins = []
maxs = []
shapes = []

for i, path in enumerate(
    df["path"],
    1
):

    arr = np.load(
        path
    )

    shapes.append(
        arr.shape
    )

    if not np.isfinite(arr).all():

        invalid.append({
            "path": path,
            "reason": "non-finite"
        })

    if arr.min() < 0:

        invalid.append({
            "path": path,
            "reason": "below_0"
        })

    if arr.max() > 1:

        invalid.append({
            "path": path,
            "reason": "above_1"
        })

    mins.append(
        float(arr.min())
    )

    maxs.append(
        float(arr.max())
    )

print(
    "Invalid arrays:",
    len(invalid)
)

print(
    "Global minimum:",
    min(mins)
)

print(
    "Global maximum:",
    max(maxs)
)

print(
    "Unique shapes:",
    sorted(
        set(
            shapes
        )
    )
)

# =============================================================================
# PATIENT SLICE COUNTS
# =============================================================================

print("\n" + "=" * 80)
print("3. PATIENT SLICE COUNTS")
print("=" * 80)

patient_counts = (
    df.groupby(
        [
            "split",
            "class_name",
            "patient_id"
        ]
    )
    .size()
)

print(
    patient_counts
)

bad_patients = (
    patient_counts[
        patient_counts != 15
    ]
)

print(
    "\nPatients not having exactly 15 slices:",
    len(bad_patients)
)

if len(bad_patients) == 0:

    print(
        "✅ Every patient has exactly 15 slices."
    )

# =============================================================================
# CLASS DISTRIBUTION
# =============================================================================

print("\n" + "=" * 80)
print("4. CLASS DISTRIBUTION")
print("=" * 80)

print(
    df.groupby(
        [
            "split",
            "class_name"
        ]
    ).size()
)

# =============================================================================
# PATIENT LEAKAGE
# =============================================================================

print("\n" + "=" * 80)
print("5. PATIENT LEAKAGE")
print("=" * 80)

train = set(
    df[
        df["split"] == "train"
    ]["patient_id"]
)

val = set(
    df[
        df["split"] == "validation"
    ]["patient_id"]
)

test = set(
    df[
        df["split"] == "test"
    ]["patient_id"]
)

print(
    "Train ∩ Validation:",
    len(train & val)
)

print(
    "Train ∩ Test:",
    len(train & test)
)

print(
    "Validation ∩ Test:",
    len(val & test)
)

# =============================================================================
# DUPLICATE HASH ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("6. DUPLICATE PROCESSED-SLICE ANALYSIS")
print("=" * 80)

duplicate_groups = (
    df[
        df["sha256"].duplicated(
            keep=False
        )
    ]
    .sort_values("sha256")
)

print(
    "Duplicate hash groups:",
    duplicate_groups[
        "sha256"
    ].nunique()
)

if len(duplicate_groups):

    print(
        "\nDuplicate entries:"
    )

    print(
        duplicate_groups[
            [
                "class_name",
                "patient_id",
                "split",
                "slice_number",
                "source_z",
                "path"
            ]
        ].to_string(
            index=False
        )
    )

# =============================================================================
# CROSS-PATIENT DUPLICATE CHECK
# =============================================================================

print("\n" + "=" * 80)
print("7. CROSS-PATIENT DUPLICATE CHECK")
print("=" * 80)

cross_patient = (
    duplicate_groups
    .groupby("sha256")["patient_id"]
    .nunique()
)

cross_patient_duplicates = (
    cross_patient[
        cross_patient > 1
    ]
)

print(
    "Duplicate groups across different patients:",
    len(cross_patient_duplicates)
)

if len(cross_patient_duplicates) == 0:

    print(
        "✅ No cross-patient duplicate slices."
    )

# =============================================================================
# CROSS-SPLIT DUPLICATE CHECK
# =============================================================================

print("\n" + "=" * 80)
print("8. CROSS-SPLIT DUPLICATE CHECK")
print("=" * 80)

cross_split = (
    duplicate_groups
    .groupby("sha256")["split"]
    .nunique()
)

cross_split_duplicates = (
    cross_split[
        cross_split > 1
    ]
)

print(
    "Duplicate groups across splits:",
    len(cross_split_duplicates)
)

if len(cross_split_duplicates) == 0:

    print(
        "✅ No duplicate slices across train/validation/test."
    )

# =============================================================================
# DIRECTORY COUNT
# =============================================================================

print("\n" + "=" * 80)
print("9. ACTUAL FILE COUNT")
print("=" * 80)

actual_files = list(
    PROCESSED_DIR.rglob(
        "*.npy"
    )
)

print(
    "Actual .npy files:",
    len(actual_files)
)

print(
    "Manifest rows:",
    len(df)
)

# =============================================================================
# FINAL DECISION
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 3B - STEP 8 FINAL RESULT")
print("=" * 80)

checks = {

    "missing_files":
        len(missing) == 0,

    "invalid_arrays":
        len(invalid) == 0,

    "all_224x224":
        all(
            s == (224, 224)
            for s in shapes
        ),

    "15_slices_per_patient":
        len(bad_patients) == 0,

    "patient_leakage":
        (
            len(train & val)
            +
            len(train & test)
            +
            len(val & test)
        ) == 0,

    "cross_patient_duplicates":
        len(cross_patient_duplicates) == 0,

    "cross_split_duplicates":
        len(cross_split_duplicates) == 0,

    "file_count_matches":
        len(actual_files) == len(df)
}

for name, result in checks.items():

    print(
        f"{name:30s}: "
        f"{'✅ PASS' if result else '❌ FAIL'}"
    )

print("\n" + "=" * 80)

if all(checks.values()):

    print(
        "✅ STEP 8 PASSED"
    )

    print(
        "\nDataset is ready for GPU training."
    )

    print(
        "NEXT → STEP 9: CNN TRAINING"
    )

else:

    print(
        "⚠️ STEP 8 HAS ISSUES."
    )

    print(
        "DO NOT TRAIN YET."
    )

print("=" * 80)

MODEL 3B - STEP 8
FINAL PROCESSED-DATA VERIFICATION

Manifest rows: 2550
Unique patients: 104

1. FILE EXISTENCE
Expected files: 2550
Missing files: 0
✅ All processed files exist.

2. NUMERICAL VALIDATION
Invalid arrays: 0
Global minimum: 0.0
Global maximum: 1.0
Unique shapes: [(224, 224)]

3. PATIENT SLICE COUNTS
split       class_name        patient_id
test        Brain_Metastasis  Patient_04    15
                              Patient_15    30
                              Patient_21    30
                              Patient_31    30
                              Patient_38    30
                                            ..
validation  Pituitary         sub-41        15
                              sub-47        15
                              sub-52        15
                              sub-56        15
                              sub-60        15
Length: 124, dtype: int64

Patients not having exactly 15 slices: 41

4. CLASS DISTRIBUTION
split       class_name      
test 

In [ ]:
# =============================================================================
# MODEL 3B - STEP 7-FIX
# CORRECT 3-CLASS CT PREPROCESSING + 15 SLICES/PATIENT
# =============================================================================

import shutil
import json
import random
import hashlib
import numpy as np
import pandas as pd
import nibabel as nib

from pathlib import Path
from scipy.ndimage import zoom
from datetime import datetime

# =============================================================================
# PATHS
# =============================================================================

ROOT = Path("/content/model3b")
RAW_DIR = ROOT / "raw"

PROCESSED_DIR = ROOT / "processed" / "3class"
MANIFEST_DIR = ROOT / "manifests"
AUDIT_DIR = ROOT / "audit"

DRIVE_ROOT = Path("/content/drive/MyDrive/Model3B")
DRIVE_MANIFEST = DRIVE_ROOT / "manifests"
DRIVE_AUDIT = DRIVE_ROOT / "audits"
DRIVE_CONFIG = DRIVE_ROOT / "configs"

# Remove ONLY bad processed dataset
if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for p in [
    MANIFEST_DIR,
    AUDIT_DIR,
    DRIVE_MANIFEST,
    DRIVE_AUDIT,
    DRIVE_CONFIG
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

# =============================================================================
# CONFIG
# =============================================================================

CLASSES = {
    0: "Meningioma",
    1: "Pituitary",
    2: "Brain_Metastasis"
}

SEED = 42
SLICES_PER_PATIENT = 15
IMAGE_SIZE = 224

HU_MIN = -1000.0
HU_MAX = 2000.0

EDGE_EXCLUSION = 5
CONTENT_THRESHOLD = 0.05

random.seed(SEED)
np.random.seed(SEED)

print("=" * 80)
print("MODEL 3B - STEP 7-FIX")
print("CORRECT PATIENT-LEVEL CT PREPROCESSING")
print("=" * 80)

# =============================================================================
# DISCOVER VOLUMES
# =============================================================================

records = []

for class_id, class_name in CLASSES.items():

    class_dir = RAW_DIR / class_name

    for path in sorted(
        class_dir.rglob("*.nii.gz")
    ):

        patient_id = path.parent.name

        # IMPORTANT:
        # Class + patient prevents sub-01 collisions between classes.
        patient_key = (
            f"{class_name}__{patient_id}"
        )

        # Unique volume ID prevents multiple CT volumes
        # from overwriting each other.
        volume_id = hashlib.md5(
            str(path).encode()
        ).hexdigest()[:12]

        records.append({

            "class_id": class_id,
            "class_name": class_name,
            "patient_id": patient_id,
            "patient_key": patient_key,
            "volume_id": volume_id,
            "path": path
        })

df_volumes = pd.DataFrame(records)

print(
    "\nCT volumes:",
    len(df_volumes)
)

print(
    "Unique patients:",
    df_volumes[
        "patient_key"
    ].nunique()
)

print(
    "\nPatients by class:"
)

print(
    df_volumes
    .groupby("class_name")[
        "patient_key"
    ]
    .nunique()
)

# =============================================================================
# PATIENT SPLIT
# =============================================================================

patient_table = (
    df_volumes[
        [
            "patient_key",
            "patient_id",
            "class_id",
            "class_name"
        ]
    ]
    .drop_duplicates(
        "patient_key"
    )
)

train_patients = []
val_patients = []
test_patients = []

for class_id, class_name in CLASSES.items():

    cls = patient_table[
        patient_table["class_id"] == class_id
    ]

    patients = sorted(
        cls["patient_key"].tolist()
    )

    rng = random.Random(
        SEED + class_id
    )

    rng.shuffle(patients)

    n = len(patients)

    n_test = max(
        1,
        round(n * 0.15)
    )

    n_val = max(
        1,
        round(n * 0.15)
    )

    test_patients.extend(
        patients[:n_test]
    )

    val_patients.extend(
        patients[
            n_test:n_test+n_val
        ]
    )

    train_patients.extend(
        patients[
            n_test+n_val:
        ]
    )

split_lookup = {}

for p in train_patients:
    split_lookup[p] = "train"

for p in val_patients:
    split_lookup[p] = "validation"

for p in test_patients:
    split_lookup[p] = "test"

print("\n" + "=" * 80)
print("PATIENT SPLIT")
print("=" * 80)

print(
    "Train:",
    len(train_patients)
)

print(
    "Validation:",
    len(val_patients)
)

print(
    "Test:",
    len(test_patients)
)

assert (
    len(set(train_patients) &
        set(val_patients)) == 0
)

assert (
    len(set(train_patients) &
        set(test_patients)) == 0
)

assert (
    len(set(val_patients) &
        set(test_patients)) == 0
)

assert (
    len(split_lookup)
    ==
    patient_table["patient_key"].nunique()
)

print(
    "✅ Patient split valid."
)

# =============================================================================
# SAVE PATIENT SPLIT
# =============================================================================

patient_split = patient_table.copy()

patient_split["split"] = (
    patient_split[
        "patient_key"
    ].map(split_lookup)
)

split_path = (
    MANIFEST_DIR /
    "model3b_step7fix_patient_split.csv"
)

patient_split.to_csv(
    split_path,
    index=False
)

# =============================================================================
# IMAGE PROCESSING
# =============================================================================

def preprocess_volume(path):

    img = nib.load(
        str(path)
    )

    data = img.get_fdata(
        dtype=np.float32
    )

    data = np.nan_to_num(
        data,
        nan=0.0,
        posinf=HU_MAX,
        neginf=HU_MIN
    )

    data = np.clip(
        data,
        HU_MIN,
        HU_MAX
    )

    data = (
        data - HU_MIN
    ) / (
        HU_MAX - HU_MIN
    )

    return data


def resize_slice(image):

    h, w = image.shape

    out = zoom(
        image,
        (
            IMAGE_SIZE / h,
            IMAGE_SIZE / w
        ),
        order=1
    )

    result = np.zeros(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        ),
        dtype=np.float32
    )

    hh = min(
        IMAGE_SIZE,
        out.shape[0]
    )

    ww = min(
        IMAGE_SIZE,
        out.shape[1]
    )

    result[
        :hh,
        :ww
    ] = out[
        :hh,
        :ww
    ]

    return result


# =============================================================================
# PROCESS PATIENT BY PATIENT
# =============================================================================

slice_rows = []

patient_groups = (
    df_volumes
    .groupby(
        "patient_key"
    )
)

total_patients = len(
    patient_groups
)

for pnum, (
    patient_key,
    patient_volumes
) in enumerate(
    patient_groups,
    1
):

    class_name = (
        patient_volumes.iloc[0]
        ["class_name"]
    )

    class_id = int(
        patient_volumes.iloc[0]
        ["class_id"]
    )

    patient_id = (
        patient_volumes.iloc[0]
        ["patient_id"]
    )

    split = split_lookup[
        patient_key
    ]

    print(
        f"[{pnum}/{total_patients}] "
        f"{class_name} / "
        f"{patient_id} "
        f"({len(patient_volumes)} CT volumes)"
    )

    candidates = []

    # -------------------------------------------------------------------------
    # LOAD EACH VOLUME
    # -------------------------------------------------------------------------

    for _, volume_row in (
        patient_volumes.iterrows()
    ):

        volume = preprocess_volume(
            volume_row["path"]
        )

        z_count = volume.shape[2]

        start = EDGE_EXCLUSION
        end = z_count - EDGE_EXCLUSION

        if end <= start:
            start = 0
            end = z_count

        for z in range(
            start,
            end
        ):

            image = volume[
                :,
                :,
                z
            ]

            content = float(
                np.mean(
                    image > 0.05
                )
            )

            if content < CONTENT_THRESHOLD:
                continue

            candidates.append({

                "content": content,

                "z": z,

                "volume_id":
                    volume_row[
                        "volume_id"
                    ],

                "source_path":
                    str(
                        volume_row[
                            "path"
                        ]
                    ),

                "image":
                    image.copy()
            })

        del volume

    # -------------------------------------------------------------------------
    # FALLBACK IF TOO FEW CANDIDATES
    # -------------------------------------------------------------------------

    if len(candidates) < SLICES_PER_PATIENT:

        print(
            "    ⚠️ Few candidates; using fallback."
        )

        candidates = []

        for _, volume_row in (
            patient_volumes.iterrows()
        ):

            volume = preprocess_volume(
                volume_row["path"]
            )

            z_count = volume.shape[2]

            indices = np.linspace(
                0,
                z_count - 1,
                min(
                    20,
                    z_count
                ),
                dtype=int
            )

            for z in indices:

                image = volume[
                    :,
                    :,
                    z
                ]

                candidates.append({

                    "content":
                        float(
                            np.mean(
                                image > 0.05
                            )
                        ),

                    "z":
                        int(z),

                    "volume_id":
                        volume_row[
                            "volume_id"
                        ],

                    "source_path":
                        str(
                            volume_row[
                                "path"
                            ]
                        ),

                    "image":
                        image.copy()
                })

            del volume

    # -------------------------------------------------------------------------
    # RANK CANDIDATES
    # -------------------------------------------------------------------------

    candidates.sort(
        key=lambda x: x["content"],
        reverse=True
    )

    # Keep a wider pool
    pool = candidates[
        :min(
            len(candidates),
            SLICES_PER_PATIENT * 5
        )
    ]

    # Evenly sample from high-content pool
    positions = np.linspace(
        0,
        len(pool) - 1,
        SLICES_PER_PATIENT,
        dtype=int
    )

    selected = [
        pool[int(i)]
        for i in positions
    ]

    # -------------------------------------------------------------------------
    # SAVE
    # -------------------------------------------------------------------------

    out_dir = (
        PROCESSED_DIR /
        split /
        class_name /
        patient_id
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for slice_no, candidate in enumerate(
        selected,
        1
    ):

        image = resize_slice(
            candidate["image"]
        )

        image = np.clip(
            image,
            0.0,
            1.0
        ).astype(
            np.float32
        )

        # UNIQUE patient slice filename
        filename = (
            f"{class_name}_"
            f"{patient_id}_"
            f"slice_{slice_no:02d}.npy"
        )

        output = (
            out_dir /
            filename
        )

        np.save(
            output,
            image
        )

        sha = hashlib.sha256(
            image.tobytes()
        ).hexdigest()

        slice_rows.append({

            "class_id":
                class_id,

            "class_name":
                class_name,

            "patient_id":
                patient_id,

            "patient_key":
                patient_key,

            "split":
                split,

            "slice_number":
                slice_no,

            "source_z":
                candidate["z"],

            "source_volume_id":
                candidate["volume_id"],

            "source_path":
                candidate["source_path"],

            "height":
                IMAGE_SIZE,

            "width":
                IMAGE_SIZE,

            "min":
                float(image.min()),

            "max":
                float(image.max()),

            "mean":
                float(image.mean()),

            "content":
                candidate["content"],

            "sha256":
                sha,

            "path":
                str(output)
        })

# =============================================================================
# MANIFEST
# =============================================================================

slice_df = pd.DataFrame(
    slice_rows
)

manifest_path = (
    MANIFEST_DIR /
    "model3b_step7fix_3class_slice_manifest.csv"
)

slice_df.to_csv(
    manifest_path,
    index=False
)

# =============================================================================
# VALIDATION
# =============================================================================

print("\n")
print("=" * 80)
print("STEP 7-FIX VALIDATION")
print("=" * 80)

print(
    "\nTotal slices:",
    len(slice_df)
)

print(
    "Unique patients:",
    slice_df[
        "patient_key"
    ].nunique()
)

print(
    "\nSlices per patient:"
)

patient_counts = (
    slice_df
    .groupby(
        "patient_key"
    )
    .size()
)

print(
    patient_counts.value_counts()
)

bad_patients = patient_counts[
    patient_counts !=
    SLICES_PER_PATIENT
]

print(
    "\nPatients not having 15 slices:",
    len(bad_patients)
)

# =============================================================================
# SPLIT DISTRIBUTION
# =============================================================================

print(
    "\nSplit distribution:"
)

print(
    slice_df[
        "split"
    ].value_counts()
)

print(
    "\nClass distribution:"
)

print(
    slice_df[
        "class_name"
    ].value_counts()
)

print(
    "\nClass × split:"
)

print(
    slice_df
    .groupby(
        [
            "split",
            "class_name"
        ]
    )
    .size()
)

# =============================================================================
# LEAKAGE
# =============================================================================

train = set(
    slice_df[
        slice_df["split"] == "train"
    ]["patient_key"]
)

val = set(
    slice_df[
        slice_df["split"] == "validation"
    ]["patient_key"]
)

test = set(
    slice_df[
        slice_df["split"] == "test"
    ]["patient_key"]
)

print(
    "\nTrain ∩ Validation:",
    len(train & val)
)

print(
    "Train ∩ Test:",
    len(train & test)
)

print(
    "Validation ∩ Test:",
    len(val & test)
)

# =============================================================================
# DUPLICATES
# =============================================================================

duplicate_df = slice_df[
    slice_df["sha256"].duplicated(
        keep=False
    )
]

duplicate_groups = (
    duplicate_df[
        "sha256"
    ].nunique()
)

print(
    "\nDuplicate hash groups:",
    duplicate_groups
)

cross_patient = (
    duplicate_df
    .groupby(
        "sha256"
    )[
        "patient_key"
    ]
    .nunique()
)

cross_patient_duplicates = (
    cross_patient[
        cross_patient > 1
    ]
)

print(
    "Cross-patient duplicate groups:",
    len(cross_patient_duplicates)
)

# =============================================================================
# FILE COUNT
# =============================================================================

actual_files = list(
    PROCESSED_DIR.rglob(
        "*.npy"
    )
)

print(
    "\nActual .npy files:",
    len(actual_files)
)

# =============================================================================
# NUMERICAL CHECK
# =============================================================================

print(
    "\nGlobal minimum:",
    slice_df["min"].min()
)

print(
    "Global maximum:",
    slice_df["max"].max()
)

# =============================================================================
# SAVE AUDIT
# =============================================================================

audit = {

    "step":
        "7-FIX",

    "classes":
        CLASSES,

    "source_ct_volumes":
        len(df_volumes),

    "unique_patients":
        int(
            slice_df[
                "patient_key"
            ].nunique()
        ),

    "processed_slices":
        len(slice_df),

    "slices_per_patient":
        SLICES_PER_PATIENT,

    "image_size":
        IMAGE_SIZE,

    "hu_window":
        [
            HU_MIN,
            HU_MAX
        ],

    "patient_leakage":
        (
            len(train & val)
            +
            len(train & test)
            +
            len(val & test)
        ),

    "duplicate_hash_groups":
        int(
            duplicate_groups
        ),

    "cross_patient_duplicates":
        int(
            len(
                cross_patient_duplicates
            )
        ),

    "actual_files":
        len(actual_files),

    "created":
        datetime.now().isoformat()
}

audit_path = (
    AUDIT_DIR /
    "model3b_step7fix_audit.json"
)

with open(
    audit_path,
    "w"
) as f:

    json.dump(
        audit,
        f,
        indent=2
    )

# =============================================================================
# BACKUP MANIFEST + AUDIT
# =============================================================================

shutil.copy2(
    manifest_path,
    DRIVE_MANIFEST /
    manifest_path.name
)

shutil.copy2(
    split_path,
    DRIVE_MANIFEST /
    split_path.name
)

shutil.copy2(
    audit_path,
    DRIVE_AUDIT /
    audit_path.name
)

# =============================================================================
# CONFIG
# =============================================================================

config = {

    "classes":
        CLASSES,

    "ct_only":
        True,

    "image_size":
        IMAGE_SIZE,

    "hu_window":
        [
            HU_MIN,
            HU_MAX
        ],

    "slices_per_patient":
        SLICES_PER_PATIENT,

    "patient_key":
        "class_name + patient_id",

    "multiple_ct_volumes":
        "handled independently",

    "augmentation":
        False,

    "segmentation":
        False,

    "normalization":
        "[0,1]",

    "raw_ct_modified":
        False
}

config_path = (
    DRIVE_CONFIG /
    "model3b_step7fix_config.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

# =============================================================================
# FINAL
# =============================================================================

print("\n")
print("=" * 80)
print("MODEL 3B - STEP 7-FIX COMPLETE")
print("=" * 80)

if (
    slice_df[
        "patient_key"
    ].nunique() == 124
    and
    len(slice_df) == 124 * 15
    and
    len(bad_patients) == 0
    and
    len(train & val) == 0
    and
    len(train & test) == 0
    and
    len(val & test) == 0
    and
    len(cross_patient_duplicates) == 0
    and
    len(actual_files) == len(slice_df)
):

    print(
        "\n✅ ALL CHECKS PASSED"
    )

    print(
        "Patients: 124"
    )

    print(
        "Slices: 1860"
    )

    print(
        "15 slices/patient"
    )

    print(
        "No patient leakage"
    )

    print(
        "No cross-patient duplicates"
    )

    print(
        "No overwritten slices"
    )

    print(
        "\nNEXT → STEP 8-FINAL"
    )

else:

    print(
        "\n⚠️ CHECKS FAILED."
    )

    print(
        "DO NOT TRAIN."
    )

print("=" * 80)

MODEL 3B - STEP 7-FIX
CORRECT PATIENT-LEVEL CT PREPROCESSING

CT volumes: 170
Unique patients: 124

Patients by class:
class_name
Brain_Metastasis    44
Meningioma          20
Pituitary           60
Name: patient_key, dtype: int64

PATIENT SPLIT
Train: 86
Validation: 19
Test: 19
✅ Patient split valid.
[1/124] Brain_Metastasis / Patient_01 (2 CT volumes)
[2/124] Brain_Metastasis / Patient_02 (2 CT volumes)
[3/124] Brain_Metastasis / Patient_03 (2 CT volumes)
[4/124] Brain_Metastasis / Patient_04 (1 CT volumes)
[5/124] Brain_Metastasis / Patient_05 (3 CT volumes)
[6/124] Brain_Metastasis / Patient_06 (2 CT volumes)
[7/124] Brain_Metastasis / Patient_07 (1 CT volumes)
[8/124] Brain_Metastasis / Patient_08 (2 CT volumes)
[9/124] Brain_Metastasis / Patient_09 (2 CT volumes)
[10/124] Brain_Metastasis / Patient_10 (2 CT volumes)
[11/124] Brain_Metastasis / Patient_11 (2 CT volumes)
[12/124] Brain_Metastasis / Patient_12 (3 CT volumes)
[13/124] Brain_Metastasis / Patient_14 (2 CT volumes)
[14/

In [ ]:
# =============================================================================
# MODEL 3B - STEP 7-FIX2
# REMOVE SINGLE CROSS-PATIENT DUPLICATE
# =============================================================================

import numpy as np
import pandas as pd
import nibabel as nib
import hashlib

from pathlib import Path
from scipy.ndimage import zoom

ROOT = Path("/content/model3b")

MANIFEST = (
    ROOT /
    "manifests" /
    "model3b_step7fix_3class_slice_manifest.csv"
)

PROCESSED = (
    ROOT /
    "processed" /
    "3class"
)

df = pd.read_csv(MANIFEST)

print("=" * 80)
print("MODEL 3B - STEP 7-FIX2")
print("CROSS-PATIENT DUPLICATE REPAIR")
print("=" * 80)

# =============================================================================
# FIND DUPLICATE HASH GROUPS
# =============================================================================

dups = df[
    df["sha256"].duplicated(
        keep=False
    )
].copy()

print(
    "\nDuplicate rows:",
    len(dups)
)

print(
    "Duplicate hash groups:",
    dups["sha256"].nunique()
)

print(
    "\nDuplicate records:"
)

print(
    dups[
        [
            "class_name",
            "patient_id",
            "patient_key",
            "split",
            "slice_number",
            "source_z",
            "source_volume_id",
            "source_path",
            "path",
            "sha256"
        ]
    ].to_string(
        index=False
    )
)

# =============================================================================
# CROSS-PATIENT DUPLICATES
# =============================================================================

cross = (
    dups
    .groupby("sha256")
    ["patient_key"]
    .nunique()
)

cross_hashes = cross[
    cross > 1
].index.tolist()

if not cross_hashes:

    print(
        "\n✅ No cross-patient duplicates."
    )

else:

    print(
        "\nCross-patient duplicate groups:",
        len(cross_hashes)
    )

# =============================================================================
# REPAIR
# =============================================================================

for duplicate_hash in cross_hashes:

    group = dups[
        dups["sha256"] ==
        duplicate_hash
    ].copy()

    # Keep first patient's slice.
    # Replace the later one.
    target = group.iloc[-1]

    print("\n" + "-" * 80)

    print(
        "Repairing:",
        target["class_name"],
        target["patient_id"],
        "slice",
        target["slice_number"]
    )

    source_path = Path(
        target["source_path"]
    )

    volume_id = target[
        "source_volume_id"
    ]

    used_z = set(
        df[
            (
                df["patient_key"]
                ==
                target["patient_key"]
            )
            &
            (
                df["source_volume_id"]
                ==
                volume_id
            )
        ]["source_z"]
        .astype(int)
        .tolist()
    )

    print(
        "Already-used z positions:",
        sorted(used_z)
    )

    # -------------------------------------------------------------------------
    # LOAD SOURCE CT
    # -------------------------------------------------------------------------

    img = nib.load(
        str(source_path)
    )

    volume = img.get_fdata(
        dtype=np.float32
    )

    volume = np.nan_to_num(
        volume,
        nan=0.0,
        posinf=2000.0,
        neginf=-1000.0
    )

    volume = np.clip(
        volume,
        -1000.0,
        2000.0
    )

    volume = (
        volume + 1000.0
    ) / 3000.0

    z_count = volume.shape[2]

    # -------------------------------------------------------------------------
    # SEARCH FOR NEW VALID SLICE
    # -------------------------------------------------------------------------

    replacement = None

    for z in range(
        5,
        max(
            5,
            z_count - 5
        )
    ):

        if z in used_z:
            continue

        image = volume[
            :,
            :,
            z
        ]

        content = float(
            np.mean(
                image > 0.05
            )
        )

        if content < 0.05:
            continue

        resized = zoom(
            image,
            (
                224 / image.shape[0],
                224 / image.shape[1]
            ),
            order=1
        )

        final_image = np.zeros(
            (
                224,
                224
            ),
            dtype=np.float32
        )

        h = min(
            224,
            resized.shape[0]
        )

        w = min(
            224,
            resized.shape[1]
        )

        final_image[
            :h,
            :w
        ] = resized[
            :h,
            :w
        ]

        final_image = np.clip(
            final_image,
            0,
            1
        ).astype(
            np.float32
        )

        new_hash = hashlib.sha256(
            final_image.tobytes()
        ).hexdigest()

        # Make absolutely sure it doesn't duplicate
        # any existing patient or dataset slice.
        if new_hash in set(
            df["sha256"]
        ):
            continue

        replacement = (
            z,
            final_image,
            content,
            new_hash
        )

        break

    if replacement is None:

        raise RuntimeError(
            f"Could not find replacement slice for "
            f"{target['patient_key']}"
        )

    new_z, new_image, new_content, new_hash = (
        replacement
    )

    # -------------------------------------------------------------------------
    # SAVE REPLACEMENT
    # -------------------------------------------------------------------------

    output_path = Path(
        target["path"]
    )

    np.save(
        output_path,
        new_image
    )

    # -------------------------------------------------------------------------
    # UPDATE MANIFEST
    # -------------------------------------------------------------------------

    mask = (
        df["path"] ==
        target["path"]
    )

    df.loc[
        mask,
        "source_z"
    ] = new_z

    df.loc[
        mask,
        "sha256"
    ] = new_hash

    df.loc[
        mask,
        "content"
    ] = new_content

    df.loc[
        mask,
        "min"
    ] = float(
        new_image.min()
    )

    df.loc[
        mask,
        "max"
    ] = float(
        new_image.max()
    )

    df.loc[
        mask,
        "mean"
    ] = float(
        new_image.mean()
    )

    print(
        "Replacement z:",
        new_z
    )

    print(
        "New hash:",
        new_hash
    )

    print(
        "✅ Replacement saved."
    )

    del volume

# =============================================================================
# SAVE MANIFEST
# =============================================================================

df.to_csv(
    MANIFEST,
    index=False
)

print("\n" + "=" * 80)
print("REVALIDATING")
print("=" * 80)

# =============================================================================
# RECHECK FILE COUNT
# =============================================================================

actual_files = list(
    PROCESSED.rglob(
        "*.npy"
    )
)

print(
    "\nManifest rows:",
    len(df)
)

print(
    "Actual files:",
    len(actual_files)
)

# =============================================================================
# PATIENT COUNTS
# =============================================================================

patient_counts = (
    df
    .groupby(
        "patient_key"
    )
    .size()
)

print(
    "\nPatients:",
    len(patient_counts)
)

print(
    "Patients with exactly 15 slices:",
    int(
        (
            patient_counts == 15
        ).sum()
    )
)

print(
    "Patients not having 15:",
    int(
        (
            patient_counts != 15
        ).sum()
    )
)

# =============================================================================
# DUPLICATES
# =============================================================================

duplicates = df[
    df["sha256"].duplicated(
        keep=False
    )
]

print(
    "\nDuplicate hash groups:",
    duplicates[
        "sha256"
    ].nunique()
)

cross_patient = (
    duplicates
    .groupby(
        "sha256"
    )[
        "patient_key"
    ]
    .nunique()
)

cross_patient = cross_patient[
    cross_patient > 1
]

print(
    "Cross-patient duplicate groups:",
    len(cross_patient)
)

# =============================================================================
# LEAKAGE
# =============================================================================

train = set(
    df[
        df["split"] == "train"
    ]["patient_key"]
)

val = set(
    df[
        df["split"] == "validation"
    ]["patient_key"]
)

test = set(
    df[
        df["split"] == "test"
    ]["patient_key"]
)

print(
    "\nTrain ∩ Validation:",
    len(train & val)
)

print(
    "Train ∩ Test:",
    len(train & test)
)

print(
    "Validation ∩ Test:",
    len(val & test)
)

# =============================================================================
# NUMERICAL
# =============================================================================

print(
    "\nGlobal minimum:",
    df["min"].min()
)

print(
    "Global maximum:",
    df["max"].max()
)

# =============================================================================
# FINAL
# =============================================================================

passed = (
    len(df) == 1860
    and
    len(actual_files) == 1860
    and
    len(patient_counts) == 124
    and
    (patient_counts == 15).all()
    and
    len(cross_patient) == 0
    and
    len(train & val) == 0
    and
    len(train & test) == 0
    and
    len(val & test) == 0
    and
    df["min"].min() >= 0
    and
    df["max"].max() <= 1
)

print("\n" + "=" * 80)

if passed:

    print(
        "✅ MODEL 3B DATASET PASSED FINAL VALIDATION"
    )

    print(
        "\nPatients      : 124"
    )

    print(
        "Slices        : 1860"
    )

    print(
        "Per patient   : 15"
    )

    print(
        "Files         : 1860"
    )

    print(
        "Duplicates    : 0 cross-patient"
    )

    print(
        "Leakage       : 0"
    )

    print(
        "Normalization : [0,1]"
    )

    print(
        "\nNEXT → STEP 8 FINAL AUDIT"
    )

else:

    print(
        "❌ VALIDATION STILL FAILED"
    )

    print(
        "DO NOT TRAIN."
    )

print("=" * 80)

MODEL 3B - STEP 7-FIX2
CROSS-PATIENT DUPLICATE REPAIR

Duplicate rows: 53
Duplicate hash groups: 1

Duplicate records:
      class_name patient_id                  patient_key      split  slice_number  source_z source_volume_id                                                             source_path                                                                                                              path                                                           sha256
Brain_Metastasis Patient_03 Brain_Metastasis__Patient_03      train             1         5     661a662806ae /content/model3b/raw/Brain_Metastasis/Patient_03/Patient_03_CT_1.nii.gz      /content/model3b/processed/3class/train/Brain_Metastasis/Patient_03/Brain_Metastasis_Patient_03_slice_01.npy d9713b8675f8e33d3edb74c668bafb4a241d736f33fd4e4a6994d2bb8684a6e9
Brain_Metastasis Patient_03 Brain_Metastasis__Patient_03      train             2        10     661a662806ae /content/model3b/raw/Brain_Metastasis/Patient_03/Pat

In [ ]:
# =============================================================================
# MODEL 3B - STEP 7-FIX3
# REGENERATE SLICES WITH CORRECT PER-SLICE HASHING
# =============================================================================

import numpy as np
import pandas as pd
import nibabel as nib
import hashlib
from pathlib import Path
from scipy.ndimage import zoom
import shutil

ROOT = Path("/content/model3b")

OLD_PROCESSED = ROOT / "processed" / "3class"
NEW_PROCESSED = ROOT / "processed" / "3class_clean"

MANIFEST = ROOT / "manifests" / "model3b_step7fix_3class_slice_manifest.csv"

NEW_MANIFEST = ROOT / "manifests" / "model3b_step7fix3_3class_slice_manifest.csv"

df = pd.read_csv(MANIFEST)

print("=" * 80)
print("MODEL 3B - STEP 7-FIX3")
print("REGENERATING PROCESSED CT SLICES")
print("=" * 80)

# -----------------------------------------------------------------------------
# CLEAN OUTPUT
# -----------------------------------------------------------------------------

if NEW_PROCESSED.exists():
    shutil.rmtree(NEW_PROCESSED)

NEW_PROCESSED.mkdir(
    parents=True,
    exist_ok=True
)

# -----------------------------------------------------------------------------
# REBUILD EACH SLICE
# -----------------------------------------------------------------------------

rows = []

for i, row in enumerate(
    df.itertuples(index=False),
    start=1
):

    source_path = Path(row.source_path)

    print(
        f"[{i}/{len(df)}] "
        f"{row.class_name} / "
        f"{row.patient_key} / "
        f"slice {row.slice_number}"
    )

    img = nib.load(
        str(source_path)
    )

    volume = img.get_fdata(
        dtype=np.float32
    )

    volume = np.nan_to_num(
        volume,
        nan=0.0,
        posinf=2000.0,
        neginf=-1000.0
    )

    volume = np.clip(
        volume,
        -1000.0,
        2000.0
    )

    volume = (
        volume + 1000.0
    ) / 3000.0

    z = int(row.source_z)

    image = volume[:, :, z]

    # Resize while preserving actual slice information
    zoom_y = 224.0 / image.shape[0]
    zoom_x = 224.0 / image.shape[1]

    image = zoom(
        image,
        (
            zoom_y,
            zoom_x
        ),
        order=1
    )

    # Center crop/pad to exactly 224x224
    output = np.zeros(
        (224, 224),
        dtype=np.float32
    )

    h = min(
        224,
        image.shape[0]
    )

    w = min(
        224,
        image.shape[1]
    )

    src_y = max(
        0,
        (image.shape[0] - 224) // 2
    )

    src_x = max(
        0,
        (image.shape[1] - 224) // 2
    )

    dst_y = max(
        0,
        (224 - image.shape[0]) // 2
    )

    dst_x = max(
        0,
        (224 - image.shape[1]) // 2
    )

    output[
        dst_y:dst_y+h,
        dst_x:dst_x+w
    ] = image[
        src_y:src_y+h,
        src_x:src_x+w
    ]

    output = np.clip(
        output,
        0.0,
        1.0
    ).astype(
        np.float32
    )

    # -------------------------------------------------------------------------
    # HASH THE ACTUAL SAVED ARRAY
    # -------------------------------------------------------------------------

    output = np.ascontiguousarray(
        output
    )

    sha256 = hashlib.sha256(
        output.tobytes()
    ).hexdigest()

    # -------------------------------------------------------------------------
    # SAVE
    # -------------------------------------------------------------------------

    split_dir = (
        NEW_PROCESSED /
        row.split /
        row.class_name /
        row.patient_id
    )

    split_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    filename = (
        f"{row.class_name}_"
        f"{row.patient_id}_"
        f"slice_{int(row.slice_number):02d}.npy"
    )

    output_path = (
        split_dir /
        filename
    )

    np.save(
        output_path,
        output
    )

    rows.append({
        "class_name": row.class_name,
        "class_id": row.class_id,
        "patient_id": row.patient_id,
        "patient_key": row.patient_key,
        "split": row.split,
        "slice_number": row.slice_number,
        "source_z": z,
        "source_volume_id": row.source_volume_id,
        "source_path": row.source_path,
        "path": str(output_path),
        "sha256": sha256,
        "min": float(output.min()),
        "max": float(output.max()),
        "mean": float(output.mean()),
        "shape": "224x224"
    })

    del volume
    del image
    del output

# -----------------------------------------------------------------------------
# SAVE NEW MANIFEST
# -----------------------------------------------------------------------------

new_df = pd.DataFrame(rows)

new_df.to_csv(
    NEW_MANIFEST,
    index=False
)

print("\n" + "=" * 80)
print("REGENERATED")
print("=" * 80)

print(
    "Rows:",
    len(new_df)
)

print(
    "Files:",
    len(list(
        NEW_PROCESSED.rglob("*.npy")
    ))
)

# -----------------------------------------------------------------------------
# DUPLICATE CHECK
# -----------------------------------------------------------------------------

duplicate_rows = new_df[
    new_df["sha256"].duplicated(
        keep=False
    )
]

print(
    "\nDuplicate rows:",
    len(duplicate_rows)
)

print(
    "Duplicate hash groups:",
    duplicate_rows["sha256"].nunique()
)

cross = (
    duplicate_rows
    .groupby("sha256")["patient_key"]
    .nunique()
)

cross = cross[
    cross > 1
]

print(
    "Cross-patient duplicate groups:",
    len(cross)
)

# -----------------------------------------------------------------------------
# PATIENT CHECK
# -----------------------------------------------------------------------------

patient_counts = (
    new_df
    .groupby("patient_key")
    .size()
)

print(
    "\nPatients:",
    len(patient_counts)
)

print(
    "Patients with 15 slices:",
    int(
        (patient_counts == 15).sum()
    )
)

print(
    "Patients not having 15:",
    int(
        (patient_counts != 15).sum()
    )
)

# -----------------------------------------------------------------------------
# SPLIT CHECK
# -----------------------------------------------------------------------------

train = set(
    new_df[
        new_df.split == "train"
    ].patient_key
)

val = set(
    new_df[
        new_df.split == "validation"
    ].patient_key
)

test = set(
    new_df[
        new_df.split == "test"
    ].patient_key
)

print(
    "\nTrain ∩ Validation:",
    len(train & val)
)

print(
    "Train ∩ Test:",
    len(train & test)
)

print(
    "Validation ∩ Test:",
    len(val & test)
)

# -----------------------------------------------------------------------------
# RANGE CHECK
# -----------------------------------------------------------------------------

print(
    "\nGlobal minimum:",
    new_df["min"].min()
)

print(
    "Global maximum:",
    new_df["max"].max()
)

# -----------------------------------------------------------------------------
# FINAL CHECK
# -----------------------------------------------------------------------------

actual_files = list(
    NEW_PROCESSED.rglob("*.npy")
)

passed = (
    len(new_df) == 1860
    and
    len(actual_files) == 1860
    and
    len(patient_counts) == 124
    and
    (patient_counts == 15).all()
    and
    len(cross) == 0
    and
    len(train & val) == 0
    and
    len(train & test) == 0
    and
    len(val & test) == 0
    and
    new_df["min"].min() >= 0
    and
    new_df["max"].max() <= 1
)

print("\n" + "=" * 80)

if passed:

    print(
        "✅ STEP 7-FIX3 PASSED"
    )

    print(
        "1860 slices"
    )

    print(
        "124 patients"
    )

    print(
        "15 slices/patient"
    )

    print(
        "0 cross-patient duplicates"
    )

    print(
        "0 patient leakage"
    )

    print(
        "Normalized [0,1]"
    )

    print(
        "\nCLEAN DATASET:"
    )

    print(
        NEW_PROCESSED
    )

    print(
        "\nMANIFEST:"
    )

    print(
        NEW_MANIFEST
    )

else:

    print(
        "❌ STEP 7-FIX3 FAILED"
    )

print("=" * 80)

MODEL 3B - STEP 7-FIX3
REGENERATING PROCESSED CT SLICES
[1/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 1
[2/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 2
[3/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 3
[4/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 4
[5/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 5
[6/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 6
[7/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 7
[8/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 8
[9/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 9
[10/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 10
[11/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 11
[12/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 12
[13/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice 13
[14/1860] Brain_Metastasis / Brain_Metastasis__Patient_01 / slice

In [ ]:
# ================================================================================
# MODEL 3B - STEP 8
# FINAL PROCESSED-DATA VERIFICATION
# ================================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import hashlib
import json

PROCESSED_DIR = Path("/content/model3b/processed/3class")
MANIFEST_PATH = Path(
    "/content/model3b/manifests/model3b_step7_fix3_3class_slice_manifest.csv"
)

print("=" * 80)
print("MODEL 3B - STEP 8")
print("FINAL PROCESSED-DATA VERIFICATION")
print("=" * 80)

# ------------------------------------------------------------------------------
# 1. CHECK DIRECTORIES
# ------------------------------------------------------------------------------

if not PROCESSED_DIR.exists():
    raise RuntimeError(f"Processed directory not found: {PROCESSED_DIR}")

if not MANIFEST_PATH.exists():
    # fallback: locate the latest 3-class slice manifest
    candidates = list(
        Path("/content/model3b/manifests").glob("*3class*slice*manifest*.csv")
    )

    if not candidates:
        raise RuntimeError("3-class slice manifest not found.")

    MANIFEST_PATH = max(candidates, key=lambda p: p.stat().st_mtime)

print("\nProcessed directory:")
print(PROCESSED_DIR)

print("\nManifest:")
print(MANIFEST_PATH)

# ------------------------------------------------------------------------------
# 2. LOAD MANIFEST
# ------------------------------------------------------------------------------

manifest = pd.read_csv(MANIFEST_PATH)

print("\nManifest rows:", len(manifest))
print("Manifest columns:")
print(list(manifest.columns))

# ------------------------------------------------------------------------------
# 3. FIND NPY FILES
# ------------------------------------------------------------------------------

npy_files = sorted(PROCESSED_DIR.rglob("*.npy"))

print("\nActual .npy files:", len(npy_files))

if len(npy_files) == 0:
    raise RuntimeError("No processed .npy files found.")

# ------------------------------------------------------------------------------
# 4. VERIFY EACH FILE
# ------------------------------------------------------------------------------

errors = []
shapes = {}
mins = []
maxs = []

print("\nChecking processed files...")

for i, path in enumerate(npy_files, 1):

    try:
        arr = np.load(path, mmap_mode="r")

        shape = tuple(arr.shape)
        dtype = str(arr.dtype)

        shapes[shape] = shapes.get(shape, 0) + 1

        if len(shape) != 2:
            errors.append((str(path), f"Invalid shape {shape}"))
            continue

        if shape != (224, 224):
            errors.append((str(path), f"Unexpected shape {shape}"))

        if not np.isfinite(arr).all():
            errors.append((str(path), "NaN/Inf detected"))
            continue

        mins.append(float(arr.min()))
        maxs.append(float(arr.max()))

        if i % 250 == 0 or i == len(npy_files):
            print(f"[{i}/{len(npy_files)}] verified")

    except Exception as e:
        errors.append((str(path), str(e)))

# ------------------------------------------------------------------------------
# 5. CLASS COUNTS FROM PATH
# ------------------------------------------------------------------------------

class_counts = {}

for path in npy_files:

    parts = path.parts

    if "Meningioma" in parts:
        cls = "Meningioma"
    elif "Pituitary" in parts:
        cls = "Pituitary"
    elif "Brain_Metastasis" in parts:
        cls = "Brain_Metastasis"
    else:
        cls = "UNKNOWN"

    class_counts[cls] = class_counts.get(cls, 0) + 1

# ------------------------------------------------------------------------------
# 6. CHECK REQUIRED CLASSES
# ------------------------------------------------------------------------------

required_classes = {
    "Meningioma",
    "Pituitary",
    "Brain_Metastasis"
}

found_classes = set(class_counts.keys())

missing_classes = required_classes - found_classes

if missing_classes:
    errors.append(
        ("CLASS_CHECK", f"Missing classes: {sorted(missing_classes)}")
    )

# ------------------------------------------------------------------------------
# 7. CHECK EXPECTED TOTAL
# ------------------------------------------------------------------------------

expected_total = 1860

if len(npy_files) != expected_total:
    errors.append(
        (
            "TOTAL_FILE_COUNT",
            f"Expected {expected_total}, found {len(npy_files)}"
        )
    )

# ------------------------------------------------------------------------------
# 8. CHECK DUPLICATE FILE HASHES
# ------------------------------------------------------------------------------

print("\nChecking duplicate processed files...")

hashes = {}
duplicate_groups = []

for i, path in enumerate(npy_files, 1):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)

    digest = h.hexdigest()

    if digest in hashes:
        duplicate_groups.append(
            [str(hashes[digest]), str(path)]
        )
    else:
        hashes[digest] = path

print("Duplicate groups:", len(duplicate_groups))

# ------------------------------------------------------------------------------
# 9. PRINT RESULTS
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 8 VALIDATION")
print("=" * 80)

print("\nTotal processed files :", len(npy_files))
print("Expected files        :", expected_total)

print("\nClass distribution:")

for cls in [
    "Meningioma",
    "Pituitary",
    "Brain_Metastasis",
    "UNKNOWN"
]:
    if cls in class_counts:
        print(f"{cls:20s}: {class_counts[cls]}")

print("\nShapes:")
for shape, count in sorted(shapes.items()):
    print(f"{shape}: {count}")

print("\nGlobal minimum:", min(mins))
print("Global maximum:", max(maxs))

print("\nDuplicate groups:", len(duplicate_groups))
print("Validation errors:", len(errors))

# ------------------------------------------------------------------------------
# 10. SAVE VERIFICATION REPORT
# ------------------------------------------------------------------------------

REPORT_DIR = Path("/content/model3b/audit/step8")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

report = {
    "step": "8",
    "processed_directory": str(PROCESSED_DIR),
    "manifest": str(MANIFEST_PATH),
    "expected_files": expected_total,
    "actual_files": len(npy_files),
    "class_distribution": class_counts,
    "shapes": {str(k): v for k, v in shapes.items()},
    "global_min": min(mins),
    "global_max": max(maxs),
    "duplicate_groups": len(duplicate_groups),
    "validation_errors": errors,
    "status": "PASS" if len(errors) == 0 else "FAIL"
}

report_path = REPORT_DIR / "model3b_step8_final_verification.json"

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

# ------------------------------------------------------------------------------
# 11. FINAL DECISION
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)

if errors:

    print("⚠️ STEP 8 FAILED")
    print("=" * 80)

    print("\nErrors:")
    for item in errors[:20]:
        print("-", item)

    raise RuntimeError(
        "Processed dataset verification failed. DO NOT TRAIN."
    )

else:

    print("MODEL 3B - STEP 8 COMPLETE")
    print("=" * 80)
    print("✅ ALL PROCESSED CT DATA VERIFIED")
    print("✅ 3 CLASSES PRESENT")
    print("✅ 1860 PROCESSED SLICES PRESENT")
    print("✅ 224 × 224 SHAPE VERIFIED")
    print("✅ FINITE VALUES VERIFIED")
    print("✅ DATASET READY FOR TRAINING")

    print("\nVerification report:")
    print(report_path)

    print("\nNEXT → STEP 9: TRAIN 3-CLASS CT CLASSIFIER")
    print("=" * 80)

MODEL 3B - STEP 8
FINAL PROCESSED-DATA VERIFICATION

Processed directory:
/content/model3b/processed/3class

Manifest:
/content/model3b/manifests/model3b_step7fix3_3class_slice_manifest.csv

Manifest rows: 1860
Manifest columns:
['class_name', 'class_id', 'patient_id', 'patient_key', 'split', 'slice_number', 'source_z', 'source_volume_id', 'source_path', 'path', 'sha256', 'min', 'max', 'mean', 'shape']

Actual .npy files: 1860

Checking processed files...
[250/1860] verified
[500/1860] verified
[750/1860] verified
[1000/1860] verified
[1250/1860] verified
[1500/1860] verified
[1750/1860] verified
[1860/1860] verified

Checking duplicate processed files...
Duplicate groups: 51

STEP 8 VALIDATION

Total processed files : 1860
Expected files        : 1860

Class distribution:
Meningioma          : 300
Pituitary           : 900
Brain_Metastasis    : 660

Shapes:
(224, 224): 1860

Global minimum: 0.0
Global maximum: 1.0

Duplicate groups: 51
Validation errors: 0

MODEL 3B - STEP 8 COMPLETE


In [ ]:
# =============================================================================
# MODEL 3B - SAVE FINAL PROCESSED DATASET TO GOOGLE DRIVE
# =============================================================================

from pathlib import Path
import shutil
import pandas as pd
import json

print("=" * 80)
print("MODEL 3B - SAVING FINAL PROCESSED DATASET TO GOOGLE DRIVE")
print("=" * 80)

# -------------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------------

LOCAL_DATASET = Path(
    "/content/model3b/processed/3class_clean"
)

LOCAL_MANIFEST = Path(
    "/content/model3b/manifests/"
    "model3b_step7fix3_3class_slice_manifest.csv"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

DRIVE_DATASET = (
    DRIVE_ROOT /
    "processed" /
    "3class"
)

DRIVE_MANIFEST_DIR = (
    DRIVE_ROOT /
    "manifests"
)

DRIVE_DATASET.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# -------------------------------------------------------------------------
# CHECK LOCAL DATASET
# -------------------------------------------------------------------------

if not LOCAL_DATASET.exists():

    raise RuntimeError(
        f"Processed dataset not found:\n{LOCAL_DATASET}"
    )

files = list(
    LOCAL_DATASET.rglob("*.npy")
)

print(
    "\nLocal processed files:",
    len(files)
)

if len(files) != 1860:

    raise RuntimeError(
        f"Expected 1860 files, found {len(files)}"
    )

# -------------------------------------------------------------------------
# LOAD MANIFEST
# -------------------------------------------------------------------------

if not LOCAL_MANIFEST.exists():

    candidates = list(
        Path("/content/model3b/manifests")
        .glob("*3class*manifest*.csv")
    )

    if not candidates:

        raise RuntimeError(
            "No 3-class manifest found."
        )

    LOCAL_MANIFEST = max(
        candidates,
        key=lambda p: p.stat().st_mtime
    )

df = pd.read_csv(
    LOCAL_MANIFEST
)

print(
    "Manifest rows:",
    len(df)
)

# -------------------------------------------------------------------------
# VERIFY SPLITS
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SPLIT INFORMATION")
print("=" * 80)

print(
    "\nSlices per split:"
)

print(
    df["split"].value_counts()
)

print(
    "\nPatients per split:"
)

print(
    df.groupby(
        "split"
    )["patient_key"]
    .nunique()
)

print(
    "\nClass × split:"
)

print(
    pd.crosstab(
        df["split"],
        df["class_name"]
    )
)

# -------------------------------------------------------------------------
# PATIENT LEAKAGE CHECK
# -------------------------------------------------------------------------

train_patients = set(
    df[
        df["split"] == "train"
    ]["patient_key"]
)

val_patients = set(
    df[
        df["split"] == "validation"
    ]["patient_key"]
)

test_patients = set(
    df[
        df["split"] == "test"
    ]["patient_key"]
)

print(
    "\nPatient leakage:"
)

print(
    "Train ∩ Validation:",
    len(train_patients & val_patients)
)

print(
    "Train ∩ Test:",
    len(train_patients & test_patients)
)

print(
    "Validation ∩ Test:",
    len(val_patients & test_patients)
)

if (
    train_patients & val_patients
    or
    train_patients & test_patients
    or
    val_patients & test_patients
):

    raise RuntimeError(
        "Patient leakage detected."
    )

print(
    "✅ No patient leakage"
)

# -------------------------------------------------------------------------
# COPY DATASET
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COPYING PROCESSED DATASET")
print("=" * 80)

# Remove previous copy if present
if DRIVE_DATASET.exists():

    print(
        "\nRemoving previous Drive copy..."
    )

    shutil.rmtree(
        DRIVE_DATASET
    )

DRIVE_DATASET.mkdir(
    parents=True,
    exist_ok=True
)

# Copy the complete processed dataset
shutil.copytree(
    LOCAL_DATASET,
    DRIVE_DATASET,
    dirs_exist_ok=True
)

print(
    "\n✅ Dataset copied to:"
)

print(
    DRIVE_DATASET
)

# -------------------------------------------------------------------------
# COPY MANIFEST
# -------------------------------------------------------------------------

drive_manifest = (
    DRIVE_MANIFEST_DIR /
    "model3b_final_3class_slice_manifest.csv"
)

shutil.copy2(
    LOCAL_MANIFEST,
    drive_manifest
)

print(
    "\n✅ Manifest copied to:"
)

print(
    drive_manifest
)

# -------------------------------------------------------------------------
# CREATE DATASET SUMMARY
# -------------------------------------------------------------------------

summary = {

    "dataset":
        "Model 3B Final 3-Class CT Dataset",

    "classes": [
        "Meningioma",
        "Pituitary",
        "Brain_Metastasis"
    ],

    "total_patients":
        int(df["patient_key"].nunique()),

    "total_slices":
        int(len(df)),

    "slices_per_patient":
        15,

    "train_patients":
        int(len(train_patients)),

    "validation_patients":
        int(len(val_patients)),

    "test_patients":
        int(len(test_patients)),

    "train_slices":
        int(
            (df["split"] == "train").sum()
        ),

    "validation_slices":
        int(
            (df["split"] == "validation").sum()
        ),

    "test_slices":
        int(
            (df["split"] == "test").sum()
        ),

    "class_distribution":
        df["class_name"]
        .value_counts()
        .to_dict(),

    "split_distribution":
        df["split"]
        .value_counts()
        .to_dict(),

    "patient_leakage":
        False,

    "image_size":
        "224x224",

    "normalization":
        "[0,1]",

    "format":
        "NumPy .npy",

    "drive_dataset":
        str(DRIVE_DATASET),

    "drive_manifest":
        str(drive_manifest)
}

summary_path = (
    DRIVE_ROOT /
    "manifests" /
    "model3b_final_dataset_summary.json"
)

with open(
    summary_path,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# -------------------------------------------------------------------------
# FINAL DRIVE COUNT
# -------------------------------------------------------------------------

drive_files = list(
    DRIVE_DATASET.rglob("*.npy")
)

print("\n" + "=" * 80)
print("FINAL DRIVE VERIFICATION")
print("=" * 80)

print(
    "\nDrive .npy files:",
    len(drive_files)
)

print(
    "Expected:",
    1860
)

print(
    "\nDrive dataset:"
)

print(
    DRIVE_DATASET
)

print(
    "\nDrive manifest:"
)

print(
    drive_manifest
)

print(
    "\nDrive summary:"
)

print(
    summary_path
)

# -------------------------------------------------------------------------
# FINAL
# -------------------------------------------------------------------------

if len(drive_files) != 1860:

    raise RuntimeError(
        "Drive copy verification failed."
    )

print("\n" + "=" * 80)
print("✅ MODEL 3B DATASET SAFELY SAVED TO DRIVE")
print("=" * 80)

print(
    "\nTRAIN      : 1290 slices / 86 patients"
)

print(
    "VALIDATION : 285 slices / 19 patients"
)

print(
    "TEST       : 285 slices / 19 patients"
)

print(
    "TOTAL      : 1860 slices / 124 patients"
)

print(
    "\nClasses:"
)

print(
    "Meningioma"
)

print(
    "Pituitary"
)

print(
    "Brain_Metastasis"
)

print(
    "\nYou can now safely proceed to STEP 9."
)

print("=" * 80)

MODEL 3B - SAVING FINAL PROCESSED DATASET TO GOOGLE DRIVE

Local processed files: 1860
Manifest rows: 1860

SPLIT INFORMATION

Slices per split:
split
train         1290
test           285
validation     285
Name: count, dtype: int64

Patients per split:
split
test          19
train         86
validation    19
Name: patient_key, dtype: int64

Class × split:
class_name  Brain_Metastasis  Meningioma  Pituitary
split                                              
test                     105          45        135
train                    450         210        630
validation               105          45        135

Patient leakage:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
✅ No patient leakage

COPYING PROCESSED DATASET

Removing previous Drive copy...

✅ Dataset copied to:
/content/drive/MyDrive/Model3B/processed/3class

✅ Manifest copied to:
/content/drive/MyDrive/Model3B/manifests/model3b_final_3class_slice_manifest.csv

FINAL DRIVE VERIFICATION

Drive .npy files: 186

In [ ]:
# =============================================================================
# MODEL 3B - STEP 9
# GPU TRAINING FROM GOOGLE DRIVE
# =============================================================================

import os
import json
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from google.colab import drive
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# =============================================================================
# 1. MOUNT GOOGLE DRIVE
# =============================================================================

print("=" * 80)
print("MODEL 3B - STEP 9")
print("DRIVE DATASET CHECK + GPU TRAINING")
print("=" * 80)

drive.mount(
    "/content/drive",
    force_remount=False
)

# =============================================================================
# 2. DRIVE PATHS
# =============================================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

DATASET_DIR = (
    DRIVE_ROOT /
    "processed" /
    "3class"
)

# Your saved final manifest
MANIFEST = (
    DRIVE_ROOT /
    "manifests" /
    "model3b_final_3class_slice_manifest.csv"
)

# Possible fallback manifest names
if not MANIFEST.exists():

    candidates = list(
        (
            DRIVE_ROOT /
            "manifests"
        ).glob(
            "*3class*manifest*.csv"
        )
    )

    if candidates:
        MANIFEST = max(
            candidates,
            key=lambda p: p.stat().st_mtime
        )

MODEL_DIR = (
    DRIVE_ROOT /
    "models"
)

RESULT_DIR = (
    DRIVE_ROOT /
    "results" /
    "step9"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nDrive root:")
print(DRIVE_ROOT)

print("\nDataset:")
print(DATASET_DIR)

print("\nManifest:")
print(MANIFEST)

# =============================================================================
# 3. VERIFY DATASET EXISTS BEFORE TRAINING
# =============================================================================

print("\n" + "=" * 80)
print("DATASET AVAILABILITY CHECK")
print("=" * 80)

if not DRIVE_ROOT.exists():
    raise RuntimeError(
        "Model3B folder does not exist in Google Drive."
    )

if not DATASET_DIR.exists():
    raise RuntimeError(
        f"Processed dataset NOT found:\n{DATASET_DIR}"
    )

if not MANIFEST.exists():
    raise RuntimeError(
        f"Manifest NOT found:\n{MANIFEST}"
    )

print("✅ Model3B folder found")
print("✅ Processed dataset found")
print("✅ Manifest found")

# =============================================================================
# 4. LOAD MANIFEST
# =============================================================================

df = pd.read_csv(
    MANIFEST
)

print(
    "\nManifest rows:",
    len(df)
)

if len(df) != 1860:

    raise RuntimeError(
        f"Expected 1860 manifest rows, found {len(df)}"
    )

required_columns = [
    "class_name",
    "patient_key",
    "patient_id",
    "split",
    "path",
    "slice_number"
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:

    raise RuntimeError(
        f"Manifest missing columns: {missing_columns}"
    )

print(
    "✅ Manifest structure valid"
)

# =============================================================================
# 5. FIX PATHS TO DRIVE
# =============================================================================
#
# The manifest was created inside Colab and therefore contains /content/ paths.
# We replace the old local root with the Drive dataset root.
#
# =============================================================================

OLD_ROOT = "/content/model3b/processed/3class_clean"

def convert_to_drive_path(p):

    p = str(p)

    # If path points to the old clean dataset
    if p.startswith(OLD_ROOT):

        relative = p[
            len(OLD_ROOT):
        ].lstrip("/")

        return str(
            DATASET_DIR /
            relative
        )

    # If path already points to Drive
    if p.startswith(
        str(DATASET_DIR)
    ):

        return p

    # Generic fallback:
    # extract train/validation/test/... portion
    parts = Path(p).parts

    split_names = {
        "train",
        "validation",
        "test"
    }

    for i, part in enumerate(parts):

        if part in split_names:

            relative = Path(
                *parts[i:]
            )

            return str(
                DATASET_DIR /
                relative
            )

    return p


df["drive_path"] = (
    df["path"]
    .apply(convert_to_drive_path)
)

# =============================================================================
# 6. VERIFY ALL 1860 FILES
# =============================================================================

print("\n" + "=" * 80)
print("VERIFYING ALL PROCESSED CT FILES")
print("=" * 80)

missing = []

for i, p in enumerate(
    df["drive_path"],
    start=1
):

    if not Path(p).exists():
        missing.append(p)

    if i % 250 == 0:
        print(
            f"[{i}/{len(df)}] checked"
        )

print(
    f"\nMissing files: {len(missing)}"
)

if missing:

    print("\nFirst missing files:")

    for p in missing[:10]:
        print(p)

    raise RuntimeError(
        f"{len(missing)} processed CT files are missing from Drive."
    )

print(
    "✅ ALL 1860 PROCESSED CT FILES FOUND"
)

# =============================================================================
# 7. VERIFY SAMPLE ARRAYS
# =============================================================================

print("\n" + "=" * 80)
print("VERIFYING SAMPLE ARRAYS")
print("=" * 80)

sample_paths = df[
    "drive_path"
].sample(
    min(
        20,
        len(df)
    ),
    random_state=42
)

for p in sample_paths:

    arr = np.load(
        p,
        mmap_mode="r"
    )

    if arr.shape != (
        224,
        224
    ):

        raise RuntimeError(
            f"Invalid shape {arr.shape}: {p}"
        )

    if not np.isfinite(
        arr
    ).all():

        raise RuntimeError(
            f"NaN/Inf found: {p}"
        )

print(
    "✅ Sample arrays valid"
)

# =============================================================================
# 8. CLASS CHECK
# =============================================================================

CLASS_NAMES = [
    "Meningioma",
    "Pituitary",
    "Brain_Metastasis"
]

CLASS_TO_ID = {
    name: i
    for i, name in enumerate(
        CLASS_NAMES
    )
}

print("\nClass distribution:")

print(
    df["class_name"]
    .value_counts()
)

for cls in CLASS_NAMES:

    if cls not in set(
        df["class_name"]
    ):

        raise RuntimeError(
            f"Missing class: {cls}"
        )

# =============================================================================
# 9. SPLIT CHECK
# =============================================================================

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

print("\n" + "=" * 80)
print("DATASET SPLIT")
print("=" * 80)

print(
    "Train slices      :",
    len(train_df)
)

print(
    "Validation slices :",
    len(val_df)
)

print(
    "Test slices       :",
    len(test_df)
)

if (
    len(train_df) != 1290
    or
    len(val_df) != 285
    or
    len(test_df) != 285
):

    raise RuntimeError(
        "Unexpected train/validation/test distribution."
    )

# =============================================================================
# 10. PATIENT LEAKAGE CHECK
# =============================================================================

train_patients = set(
    train_df["patient_key"]
)

val_patients = set(
    val_df["patient_key"]
)

test_patients = set(
    test_df["patient_key"]
)

print(
    "\nPatients:"
)

print(
    "Train:",
    len(train_patients)
)

print(
    "Validation:",
    len(val_patients)
)

print(
    "Test:",
    len(test_patients)
)

print(
    "\nTrain ∩ Validation:",
    len(
        train_patients &
        val_patients
    )
)

print(
    "Train ∩ Test:",
    len(
        train_patients &
        test_patients
    )
)

print(
    "Validation ∩ Test:",
    len(
        val_patients &
        test_patients
    )
)

if (
    train_patients &
    val_patients
):

    raise RuntimeError(
        "Patient leakage: Train/Validation"
    )

if (
    train_patients &
    test_patients
):

    raise RuntimeError(
        "Patient leakage: Train/Test"
    )

if (
    val_patients &
    test_patients
):

    raise RuntimeError(
        "Patient leakage: Validation/Test"
    )

print(
    "✅ NO PATIENT LEAKAGE"
)

# =============================================================================
# 11. GPU CHECK
# =============================================================================

print("\n" + "=" * 80)
print("GPU CHECK")
print("=" * 80)

gpus = tf.config.list_physical_devices(
    "GPU"
)

print(
    "GPU devices:",
    gpus
)

if not gpus:

    raise RuntimeError(
        "GPU NOT DETECTED. "
        "Switch Colab runtime to GPU before training."
    )

for gpu in gpus:

    try:

        tf.config.experimental.set_memory_growth(
            gpu,
            True
        )

    except:
        pass

print(
    "✅ GPU AVAILABLE"
)

# =============================================================================
# 12. CONFIG
# =============================================================================

SEED = 42

tf.random.set_seed(
    SEED
)

np.random.seed(
    SEED
)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15

NUM_CLASSES = 3

# =============================================================================
# 13. NUMPY LOADER
# =============================================================================

def load_npy(
    path,
    label
):

    path = path.decode(
        "utf-8"
    )

    image = np.load(
        path
    ).astype(
        np.float32
    )

    # grayscale → RGB-like 3 channels
    image = np.stack(
        [
            image,
            image,
            image
        ],
        axis=-1
    )

    return (
        image,
        np.int32(label)
    )


def tf_load(
    path,
    label
):

    image, label = tf.numpy_function(
        load_npy,
        [
            path,
            label
        ],
        [
            tf.float32,
            tf.int32
        ]
    )

    image.set_shape(
        (
            IMG_SIZE,
            IMG_SIZE,
            3
        )
    )

    label.set_shape(())

    return (
        image,
        label
    )

# =============================================================================
# 14. AUGMENTATION
# =============================================================================

augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            0.03
        ),

        tf.keras.layers.RandomZoom(
            0.08
        ),

        tf.keras.layers.RandomTranslation(
            0.03,
            0.03
        )
    ]
)


def augment(
    image,
    label
):

    image = augmentation(
        image,
        training=True
    )

    return (
        image,
        label
    )

# =============================================================================
# 15. DATASET BUILDER
# =============================================================================

def make_dataset(
    frame,
    training=False
):

    paths = frame[
        "drive_path"
    ].values

    labels = frame[
        "label"
    ].values

    ds = tf.data.Dataset.from_tensor_slices(
        (
            paths,
            labels
        )
    )

    if training:

        ds = ds.shuffle(
            len(frame),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        tf_load,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:

        ds = ds.map(
            augment,
            num_parallel_calls=tf.data.AUTOTUNE
        )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


# Add labels
df["label"] = (
    df["class_name"]
    .map(CLASS_TO_ID)
    .astype(int)
)

train_df["label"] = (
    train_df["class_name"]
    .map(CLASS_TO_ID)
    .astype(int)
)

val_df["label"] = (
    val_df["class_name"]
    .map(CLASS_TO_ID)
    .astype(int)
)

test_df["label"] = (
    test_df["class_name"]
    .map(CLASS_TO_ID)
    .astype(int)
)

train_ds = make_dataset(
    train_df,
    True
)

val_ds = make_dataset(
    val_df,
    False
)

test_ds = make_dataset(
    test_df,
    False
)

print(
    "\n✅ TensorFlow datasets ready"
)

# =============================================================================
# 16. CLASS WEIGHTS
# =============================================================================

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(
        NUM_CLASSES
    ),
    y=train_df["label"].values
)

class_weights = {
    i: float(w)
    for i, w in enumerate(
        weights
    )
}

print(
    "\nClass weights:",
    class_weights
)

# =============================================================================
# 17. BUILD MODEL
# =============================================================================

print("\n" + "=" * 80)
print("BUILDING MOBILE NET V2")
print("=" * 80)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    ),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = tf.keras.Input(
    shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

x = tf.keras.applications.mobilenet_v2.preprocess_input(
    inputs
)

x = base_model(
    x,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(
    x
)

x = tf.keras.layers.Dropout(
    0.30
)(
    x
)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(
    x
)

model = tf.keras.Model(
    inputs,
    outputs
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy"
    ]
)

model.summary()

# =============================================================================
# 18. CALLBACKS — SAVE DIRECTLY TO DRIVE
# =============================================================================

BEST_MODEL = (
    MODEL_DIR /
    "model3b_mobilenetv2_best.keras"
)

FINAL_MODEL = (
    MODEL_DIR /
    "model3b_mobilenetv2_final.keras"
)

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# =============================================================================
# 19. FINAL SAFETY GATE
# =============================================================================

print("\n" + "=" * 80)
print("FINAL TRAINING SAFETY CHECK")
print("=" * 80)

print(
    "Dataset files :",
    len(df)
)

print(
    "Train         :",
    len(train_df)
)

print(
    "Validation    :",
    len(val_df)
)

print(
    "Test          :",
    len(test_df)
)

print(
    "Patients      :",
    df["patient_key"].nunique()
)

print(
    "GPU           :",
    len(gpus)
)

print(
    "Classes       :",
    CLASS_NAMES
)

print(
    "Dataset       :",
    DATASET_DIR
)

if len(df) != 1860:
    raise RuntimeError("STOP: dataset count incorrect.")

if len(train_df) != 1290:
    raise RuntimeError("STOP: train count incorrect.")

if len(val_df) != 285:
    raise RuntimeError("STOP: validation count incorrect.")

if len(test_df) != 285:
    raise RuntimeError("STOP: test count incorrect.")

if df["patient_key"].nunique() != 124:
    raise RuntimeError("STOP: patient count incorrect.")

print(
    "\n✅ DATASET VERIFIED"
)

print(
    "✅ SPLITS VERIFIED"
)

print(
    "✅ PATIENT LEAKAGE CHECK PASSED"
)

print(
    "✅ GPU VERIFIED"
)

print(
    "\nSTARTING TRAINING..."
)

# =============================================================================
# 20. TRAIN
# =============================================================================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

# =============================================================================
# 21. SAVE FINAL MODEL
# =============================================================================

model.save(
    FINAL_MODEL
)

print(
    "\n✅ FINAL MODEL SAVED:"
)

print(
    FINAL_MODEL
)

# =============================================================================
# 22. TEST
# =============================================================================

print("\n" + "=" * 80)
print("TEST EVALUATION")
print("=" * 80)

test_loss, test_accuracy = (
    model.evaluate(
        test_ds,
        verbose=1
    )
)

print(
    "\nTest Loss:",
    test_loss
)

print(
    "Test Accuracy:",
    f"{test_accuracy * 100:.2f}%"
)

# =============================================================================
# 23. PREDICTIONS
# =============================================================================

y_true = []
y_pred = []

for images, labels in test_ds:

    predictions = model.predict(
        images,
        verbose=0
    )

    preds = np.argmax(
        predictions,
        axis=1
    )

    y_true.extend(
        labels.numpy().tolist()
    )

    y_pred.extend(
        preds.tolist()
    )

y_true = np.array(
    y_true
)

y_pred = np.array(
    y_pred
)

# =============================================================================
# 24. CLASSIFICATION REPORT
# =============================================================================

report = classification_report(
    y_true,
    y_pred,
    labels=[
        0,
        1,
        2
    ],
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report
).transpose()

print(
    "\nClassification report:"
)

print(
    report_df
)

report_path = (
    RESULT_DIR /
    "classification_report.csv"
)

report_df.to_csv(
    report_path
)

# =============================================================================
# 25. CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[
        0,
        1,
        2
    ]
)

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print(
    "\nConfusion matrix:"
)

print(
    cm_df
)

cm_path = (
    RESULT_DIR /
    "confusion_matrix.csv"
)

cm_df.to_csv(
    cm_path
)

# =============================================================================
# 26. TRAINING HISTORY
# =============================================================================

history_df = pd.DataFrame(
    history.history
)

history_path = (
    RESULT_DIR /
    "training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

# =============================================================================
# 27. SUMMARY
# =============================================================================

summary = {

    "model":
        "MobileNetV2",

    "classes":
        CLASS_NAMES,

    "total_patients":
        int(
            df["patient_key"].nunique()
        ),

    "total_slices":
        int(len(df)),

    "train_patients":
        len(train_patients),

    "validation_patients":
        len(val_patients),

    "test_patients":
        len(test_patients),

    "train_slices":
        len(train_df),

    "validation_slices":
        len(val_df),

    "test_slices":
        len(test_df),

    "test_accuracy":
        float(test_accuracy),

    "test_loss":
        float(test_loss),

    "classification_report":
        report,

    "confusion_matrix":
        cm.tolist()
}

summary_path = (
    RESULT_DIR /
    "model3b_step9_summary.json"
)

with open(
    summary_path,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# FINAL
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 3B - STEP 9 COMPLETE")
print("=" * 80)

print(
    f"\nFINAL TEST ACCURACY: "
    f"{test_accuracy * 100:.2f}%"
)

print(
    "\nModel saved to:"
)

print(
    FINAL_MODEL
)

print(
    "\nResults saved to:"
)

print(
    RESULT_DIR
)

print(
    "\nNEXT → STEP 10: PATIENT-LEVEL EVALUATION"
)

print("=" * 80)

MODEL 3B - STEP 9
DRIVE DATASET CHECK + GPU TRAINING
Mounted at /content/drive

Drive root:
/content/drive/MyDrive/Model3B

Dataset:
/content/drive/MyDrive/Model3B/processed/3class

Manifest:
/content/drive/MyDrive/Model3B/manifests/model3b_final_3class_slice_manifest.csv

DATASET AVAILABILITY CHECK
✅ Model3B folder found
✅ Processed dataset found
✅ Manifest found

Manifest rows: 1860
✅ Manifest structure valid

VERIFYING ALL PROCESSED CT FILES
[250/1860] checked
[500/1860] checked
[750/1860] checked
[1000/1860] checked
[1250/1860] checked
[1500/1860] checked
[1750/1860] checked

Missing files: 0
✅ ALL 1860 PROCESSED CT FILES FOUND

VERIFYING SAMPLE ARRAYS
✅ Sample arrays valid

Class distribution:
class_name
Pituitary           900
Brain_Metastasis    660
Meningioma          300
Name: count, dtype: int64

DATASET SPLIT
Train slices      : 1290
Validation slices : 285
Test slices       : 285

Patients:
Train: 86
Validation: 19
Test: 19

Train ∩ Validation: 0
Train ∩ Test: 0
Validation 

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


FINAL TRAINING SAFETY CHECK
Dataset files : 1860
Train         : 1290
Validation    : 285
Test          : 285
Patients      : 124
GPU           : 1
Classes       : ['Meningioma', 'Pituitary', 'Brain_Metastasis']
Dataset       : /content/drive/MyDrive/Model3B/processed/3class

✅ DATASET VERIFIED
✅ SPLITS VERIFIED
✅ PATIENT LEAKAGE CHECK PASSED
✅ GPU VERIFIED

STARTING TRAINING...
Epoch 1/15
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.3432 - loss: 1.2561
Epoch 1: val_accuracy improved from None to 0.19649, saving model to /content/drive/MyDrive/Model3B/models/model3b_mobilenetv2_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Model3B/models/model3b_mobilenetv2_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 392s 9s/step - accuracy: 0.3574 - loss: 1.1983 - val_accuracy: 0.1965 - val_loss: 1.0581 - learning_rate: 0.0010
Epoch 2/15
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - accuracy: 0.3482 - loss: 1.1363
Epoch 2: val_accuracy improved from 0.19649 to 0.52632, saving mod

In [ ]:
# =============================================================================
# MODEL 3B - STEP 10 RECOVERY
# RESTORE EVERYTHING FROM GOOGLE DRIVE
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import tensorflow as tf
import json
import os

from google.colab import drive
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("=" * 80)
print("MODEL 3B - STEP 10 RECOVERY")
print("RESTORING DATASET + MODEL FROM GOOGLE DRIVE")
print("=" * 80)

# =============================================================================
# 1. MOUNT DRIVE
# =============================================================================

drive.mount(
    "/content/drive",
    force_remount=False
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

DATASET_DIR = (
    DRIVE_ROOT /
    "processed" /
    "3class"
)

MANIFEST_DIR = (
    DRIVE_ROOT /
    "manifests"
)

MODEL_DIR = (
    DRIVE_ROOT /
    "models"
)

RESULT_DIR = (
    DRIVE_ROOT /
    "results" /
    "step10"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nDrive root:")
print(DRIVE_ROOT)

# =============================================================================
# 2. CHECK DATASET
# =============================================================================

print("\n" + "=" * 80)
print("CHECKING DRIVE DATASET")
print("=" * 80)

if not DATASET_DIR.exists():
    raise RuntimeError(
        f"Dataset not found:\n{DATASET_DIR}"
    )

all_npy = list(
    DATASET_DIR.rglob("*.npy")
)

print(
    "\nTotal .npy files:",
    len(all_npy)
)

if len(all_npy) != 1860:
    raise RuntimeError(
        f"Expected 1860 files, found {len(all_npy)}"
    )

print(
    "✅ 1860 processed slices found"
)

# =============================================================================
# 3. FIND MANIFEST
# =============================================================================

print("\n" + "=" * 80)
print("SEARCHING FOR MANIFEST")
print("=" * 80)

manifest_candidates = []

if MANIFEST_DIR.exists():

    manifest_candidates = list(
        MANIFEST_DIR.glob("*.csv")
    )

print(
    "\nCSV files found in manifests:"
)

for p in manifest_candidates:
    print(
        " -",
        p.name
    )

# Prefer 3-class slice manifests
preferred = [
    p for p in manifest_candidates
    if "3class" in p.name.lower()
    and "slice" in p.name.lower()
]

if preferred:

    MANIFEST = max(
        preferred,
        key=lambda p: p.stat().st_mtime
    )

else:

    MANIFEST = None

# =============================================================================
# 4. REBUILD MANIFEST IF NECESSARY
# =============================================================================

if MANIFEST is not None:

    print(
        "\n✅ Existing manifest found:"
    )

    print(
        MANIFEST
    )

    df = pd.read_csv(
        MANIFEST
    )

else:

    print(
        "\n⚠️ No suitable manifest found."
    )

    print(
        "Rebuilding manifest from saved Drive dataset..."
    )

    rows = []

    CLASS_NAMES = [
        "Meningioma",
        "Pituitary",
        "Brain_Metastasis"
    ]

    CLASS_TO_ID = {
        name: i
        for i, name in enumerate(
            CLASS_NAMES
        )
    }

    for path in all_npy:

        relative = path.relative_to(
            DATASET_DIR
        )

        parts = relative.parts

        # Expected:
        # train / class / patient / file.npy
        # OR
        # train / class / file.npy

        split = parts[0]

        if split not in {
            "train",
            "validation",
            "test"
        }:
            continue

        class_name = None

        for cls in CLASS_NAMES:

            if cls in parts:

                class_name = cls
                break

        if class_name is None:
            continue

        # Find patient directory
        patient_id = "unknown"

        for part in parts:

            if (
                part.startswith("sub-")
                or
                part.startswith("Patient_")
            ):
                patient_id = part
                break

        # Filename
        slice_number = (
            path.stem
        )

        rows.append(
            {
                "class_name":
                    class_name,

                "class_id":
                    CLASS_TO_ID[class_name],

                "patient_id":
                    patient_id,

                "patient_key":
                    f"{class_name}_{patient_id}",

                "split":
                    split,

                "slice_number":
                    slice_number,

                "path":
                    str(path)
            }
        )

    df = pd.DataFrame(
        rows
    )

    if len(df) != 1860:

        raise RuntimeError(
            f"Rebuilt manifest contains "
            f"{len(df)} rows instead of 1860."
        )

    MANIFEST = (
        MANIFEST_DIR /
        "model3b_recovered_3class_manifest.csv"
    )

    MANIFEST_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        MANIFEST,
        index=False
    )

    print(
        "\n✅ Manifest rebuilt:"
    )

    print(
        MANIFEST
    )

# =============================================================================
# 5. DATASET CHECK
# =============================================================================

print("\n" + "=" * 80)
print("DATASET VERIFICATION")
print("=" * 80)

print(
    "\nManifest rows:",
    len(df)
)

print(
    "\nClass distribution:"
)

print(
    df["class_name"].value_counts()
)

print(
    "\nSplit distribution:"
)

print(
    df["split"].value_counts()
)

# =============================================================================
# 6. FIX PATHS
# =============================================================================

def resolve_path(p):

    p = str(p)

    # Already valid
    if Path(p).exists():
        return p

    # Try filename relative to Drive dataset
    name = Path(p).name

    matches = list(
        DATASET_DIR.rglob(name)
    )

    if matches:

        return str(
            matches[0]
        )

    return p


df["drive_path"] = (
    df["path"]
    .apply(resolve_path)
)

# Check every file
missing = []

for p in df["drive_path"]:

    if not Path(p).exists():
        missing.append(p)

print(
    "\nMissing files:",
    len(missing)
)

if missing:

    print(
        "\nFirst missing:"
    )

    for p in missing[:10]:
        print(p)

    raise RuntimeError(
        "Some dataset files cannot be located on Drive."
    )

print(
    "✅ ALL DATASET FILES FOUND"
)

# =============================================================================
# 7. SPLITS
# =============================================================================

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

print("\n" + "=" * 80)
print("SPLITS")
print("=" * 80)

print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(val_df)
)

print(
    "Test:",
    len(test_df)
)

# =============================================================================
# 8. PATIENT LEAKAGE
# =============================================================================

train_patients = set(
    train_df["patient_key"]
)

val_patients = set(
    val_df["patient_key"]
)

test_patients = set(
    test_df["patient_key"]
)

print(
    "\nTrain patients:",
    len(train_patients)
)

print(
    "Validation patients:",
    len(val_patients)
)

print(
    "Test patients:",
    len(test_patients)
)

print(
    "\nTrain ∩ Validation:",
    len(
        train_patients &
        val_patients
    )
)

print(
    "Train ∩ Test:",
    len(
        train_patients &
        test_patients
    )
)

print(
    "Validation ∩ Test:",
    len(
        val_patients &
        test_patients
    )
)

# =============================================================================
# 9. FIND MODEL
# =============================================================================

print("\n" + "=" * 80)
print("SEARCHING FOR TRAINED MODEL")
print("=" * 80)

model_candidates = list(
    MODEL_DIR.glob("*.keras")
)

for p in model_candidates:

    print(
        " -",
        p.name
    )

if not model_candidates:

    raise RuntimeError(
        "No .keras model found in Drive."
    )

# Prefer final model
final_candidates = [
    p for p in model_candidates
    if "final" in p.name.lower()
]

if final_candidates:

    MODEL_PATH = max(
        final_candidates,
        key=lambda p: p.stat().st_mtime
    )

else:

    MODEL_PATH = max(
        model_candidates,
        key=lambda p: p.stat().st_mtime
    )

print(
    "\n✅ Model selected:"
)

print(
    MODEL_PATH
)

# =============================================================================
# 10. LOAD MODEL
# =============================================================================

print(
    "\nLoading model..."
)

model = tf.keras.models.load_model(
    MODEL_PATH
)

print(
    "✅ Model loaded"
)

# =============================================================================
# 11. CLASS DEFINITIONS
# =============================================================================

CLASS_NAMES = [
    "Meningioma",
    "Pituitary",
    "Brain_Metastasis"
]

CLASS_TO_ID = {
    name: i
    for i, name in enumerate(
        CLASS_NAMES
    )
}

df["label"] = (
    df["class_name"]
    .map(CLASS_TO_ID)
    .astype(int)
)

test_df["label"] = (
    test_df["class_name"]
    .map(CLASS_TO_ID)
    .astype(int)
)

# =============================================================================
# 12. PATIENT-LEVEL PREDICTION
# =============================================================================

print("\n" + "=" * 80)
print("GENERATING TEST PREDICTIONS")
print("=" * 80)

results = []

for i, row in enumerate(
    test_df.itertuples(),
    1
):

    image = np.load(
        row.drive_path
    ).astype(
        np.float32
    )

    image = np.stack(
        [
            image,
            image,
            image
        ],
        axis=-1
    )

    image = np.expand_dims(
        image,
        axis=0
    )

    prob = model.predict(
        image,
        verbose=0
    )[0]

    pred = int(
        np.argmax(prob)
    )

    results.append(
        {
            "patient_key":
                row.patient_key,

            "true_label":
                int(row.label),

            "pred_label":
                pred,

            "true_class":
                CLASS_NAMES[
                    int(row.label)
                ],

            "pred_class":
                CLASS_NAMES[
                    pred
                ],

            "max_probability":
                float(
                    np.max(prob)
                )
        }
    )

    if i % 50 == 0:
        print(
            f"[{i}/{len(test_df)}]"
        )

pred_df = pd.DataFrame(
    results
)

# =============================================================================
# 13. SLICE ACCURACY
# =============================================================================

slice_accuracy = accuracy_score(
    pred_df["true_label"],
    pred_df["pred_label"]
)

print(
    "\nSlice accuracy:",
    f"{slice_accuracy * 100:.2f}%"
)

# =============================================================================
# 14. PATIENT MAJORITY VOTE
# =============================================================================

patient_results = []

for patient_key, group in pred_df.groupby(
    "patient_key"
):

    true_label = int(
        group["true_label"].iloc[0]
    )

    prediction = int(
        group["pred_label"]
        .value_counts()
        .index[0]
    )

    patient_results.append(
        {
            "patient_key":
                patient_key,

            "true_label":
                true_label,

            "pred_label":
                prediction,

            "true_class":
                CLASS_NAMES[
                    true_label
                ],

            "pred_class":
                CLASS_NAMES[
                    prediction
                ],

            "total_slices":
                len(group),

            "correct_slices":
                int(
                    (
                        group["true_label"]
                        ==
                        group["pred_label"]
                    ).sum()
                )
        }
    )

patient_df = pd.DataFrame(
    patient_results
)

patient_accuracy = accuracy_score(
    patient_df["true_label"],
    patient_df["pred_label"]
)

print(
    "\n" + "=" * 80
)

print(
    "PATIENT-LEVEL RESULTS"
)

print(
    "=" * 80
)

print(
    "\nPatients:",
    len(patient_df)
)

print(
    "Patient accuracy:",
    f"{patient_accuracy * 100:.2f}%"
)

# =============================================================================
# 15. CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(
    patient_df["true_label"],
    patient_df["pred_label"],
    labels=[
        0,
        1,
        2
    ]
)

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print(
    "\nPatient confusion matrix:"
)

print(
    cm_df
)

# =============================================================================
# 16. CLASSIFICATION REPORT
# =============================================================================

print(
    "\nPatient classification report:"
)

print(
    classification_report(
        patient_df["true_label"],
        patient_df["pred_label"],
        labels=[
            0,
            1,
            2
        ],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)

# =============================================================================
# 17. SAVE RESULTS
# =============================================================================

pred_df.to_csv(
    RESULT_DIR /
    "step10_slice_predictions.csv",
    index=False
)

patient_df.to_csv(
    RESULT_DIR /
    "step10_patient_predictions.csv",
    index=False
)

cm_df.to_csv(
    RESULT_DIR /
    "step10_patient_confusion_matrix.csv"
)

summary = {
    "model": str(MODEL_PATH),
    "dataset": str(DATASET_DIR),
    "manifest": str(MANIFEST),
    "total_slices": int(len(df)),
    "test_slices": int(len(test_df)),
    "test_patients": int(
        len(patient_df)
    ),
    "slice_accuracy": float(
        slice_accuracy
    ),
    "patient_accuracy": float(
        patient_accuracy
    ),
    "classes": CLASS_NAMES
}

with open(
    RESULT_DIR /
    "step10_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# DONE
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 3B - STEP 10 COMPLETE")
print("=" * 80)

print(
    "\nDataset restored from Drive:"
)

print(
    DATASET_DIR
)

print(
    "\nModel restored from Drive:"
)

print(
    MODEL_PATH
)

print(
    f"\nSlice accuracy   : "
    f"{slice_accuracy * 100:.2f}%"
)

print(
    f"Patient accuracy : "
    f"{patient_accuracy * 100:.2f}%"
)

print(
    "\nResults:"
)

print(
    RESULT_DIR
)

print("=" * 80)

MODEL 3B - STEP 10 RECOVERY
RESTORING DATASET + MODEL FROM GOOGLE DRIVE
Mounted at /content/drive

Drive root:
/content/drive/MyDrive/Model3B

CHECKING DRIVE DATASET

Total .npy files: 1860
✅ 1860 processed slices found

SEARCHING FOR MANIFEST

CSV files found in manifests:
 - meningioma_acquisition_manifest.csv
 - model3b_step1_ct_acquisition_manifest.csv
 - pituitary_acquisition_manifest.csv
 - brain_metastasis_acquisition_manifest.csv
 - model3b_step7_patient_split.csv
 - model3b_step7_3class_slice_manifest.csv
 - model3b_step7fix_patient_split.csv
 - model3b_step7fix_3class_slice_manifest.csv
 - model3b_final_3class_slice_manifest.csv

✅ Existing manifest found:
/content/drive/MyDrive/Model3B/manifests/model3b_final_3class_slice_manifest.csv

DATASET VERIFICATION

Manifest rows: 1860

Class distribution:
class_name
Pituitary           900
Brain_Metastasis    660
Meningioma          300
Name: count, dtype: int64

Split distribution:
split
train         1290
test           285
valida

In [ ]:
# =============================================================================
# MODEL 3B - STEP 11
# IMPROVED 3-CLASS TRAINING
# =============================================================================

import numpy as np
import pandas as pd
import tensorflow as tf
import json

from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("=" * 80)
print("MODEL 3B - STEP 11")
print("IMPROVED 3-CLASS CT TRAINING")
print("=" * 80)

# =============================================================================
# DRIVE
# =============================================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

DATASET = (
    ROOT /
    "processed" /
    "3class"
)

MANIFEST = (
    ROOT /
    "manifests" /
    "model3b_final_3class_slice_manifest.csv"
)

MODEL_DIR = (
    ROOT /
    "models"
)

RESULT_DIR = (
    ROOT /
    "results" /
    "step11"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# GPU
# =============================================================================

gpus = tf.config.list_physical_devices(
    "GPU"
)

print("\nGPU:", gpus)

if not gpus:
    raise RuntimeError(
        "GPU not detected."
    )

# =============================================================================
# LOAD MANIFEST
# =============================================================================

df = pd.read_csv(
    MANIFEST
)

print(
    "\nManifest:",
    len(df)
)

if len(df) != 1860:
    raise RuntimeError(
        "Expected 1860 samples."
    )

# =============================================================================
# CLASSES
# =============================================================================

CLASS_NAMES = [
    "Meningioma",
    "Pituitary",
    "Brain_Metastasis"
]

CLASS_TO_ID = {
    c: i
    for i, c in enumerate(
        CLASS_NAMES
    )
}

df["label"] = (
    df["class_name"]
    .map(CLASS_TO_ID)
    .astype(int)
)

# =============================================================================
# RESOLVE DRIVE PATHS
# =============================================================================

def resolve_path(p):

    p = str(p)

    if Path(p).exists():
        return p

    old_root = (
        "/content/model3b/processed/3class_clean"
    )

    if p.startswith(old_root):

        rel = p[
            len(old_root):
        ].lstrip("/")

        return str(
            DATASET /
            rel
        )

    filename = Path(p).name

    matches = list(
        DATASET.rglob(filename)
    )

    if matches:
        return str(
            matches[0]
        )

    return p


df["drive_path"] = (
    df["path"]
    .apply(resolve_path)
)

# =============================================================================
# VERIFY
# =============================================================================

missing = [
    p
    for p in df["drive_path"]
    if not Path(p).exists()
]

print(
    "Missing:",
    len(missing)
)

if missing:
    raise RuntimeError(
        "Missing processed files."
    )

print(
    "✅ All 1860 files found"
)

# =============================================================================
# SPLITS
# =============================================================================

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

print("\nTrain:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# =============================================================================
# PATIENT LEAKAGE
# =============================================================================

train_patients = set(
    train_df["patient_key"]
)

val_patients = set(
    val_df["patient_key"]
)

test_patients = set(
    test_df["patient_key"]
)

assert not (
    train_patients &
    val_patients
)

assert not (
    train_patients &
    test_patients
)

assert not (
    val_patients &
    test_patients
)

print(
    "✅ Patient-level split verified"
)

# =============================================================================
# CLASS DISTRIBUTION
# =============================================================================

print("\nTraining classes:")

print(
    train_df[
        "class_name"
    ].value_counts()
)

# =============================================================================
# CLASS WEIGHTS
# =============================================================================

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(3),
    y=train_df["label"].values
)

class_weights = {
    i: float(w)
    for i, w in enumerate(weights)
}

print(
    "\nClass weights:",
    class_weights
)

# =============================================================================
# DATA LOADER
# =============================================================================

IMG_SIZE = 224
BATCH_SIZE = 32

def load_npy(
    path,
    label
):

    path = path.decode(
        "utf-8"
    )

    img = np.load(
        path
    ).astype(
        np.float32
    )

    img = np.stack(
        [
            img,
            img,
            img
        ],
        axis=-1
    )

    return (
        img,
        np.int32(label)
    )


def tf_load(
    path,
    label
):

    img, label = tf.numpy_function(
        load_npy,
        [
            path,
            label
        ],
        [
            tf.float32,
            tf.int32
        ]
    )

    img.set_shape(
        (
            224,
            224,
            3
        )
    )

    label.set_shape(())

    return (
        img,
        label
    )

# =============================================================================
# AUGMENTATION
# =============================================================================

augmentation = tf.keras.Sequential(
    [

        tf.keras.layers.RandomRotation(
            0.08
        ),

        tf.keras.layers.RandomZoom(
            0.15
        ),

        tf.keras.layers.RandomTranslation(
            0.08,
            0.08
        ),

        tf.keras.layers.RandomContrast(
            0.15
        )

    ]
)

def augment(
    img,
    label
):

    img = augmentation(
        img,
        training=True
    )

    return (
        img,
        label
    )

# =============================================================================
# DATASETS
# =============================================================================

def make_dataset(
    frame,
    training=False
):

    ds = tf.data.Dataset.from_tensor_slices(
        (
            frame["drive_path"].values,
            frame["label"].values
        )
    )

    if training:

        ds = ds.shuffle(
            len(frame),
            seed=42,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        tf_load,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:

        ds = ds.map(
            augment,
            num_parallel_calls=tf.data.AUTOTUNE
        )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


train_ds = make_dataset(
    train_df,
    True
)

val_ds = make_dataset(
    val_df,
    False
)

test_ds = make_dataset(
    test_df,
    False
)

print(
    "✅ TensorFlow datasets ready"
)

# =============================================================================
# MODEL
# =============================================================================

print("\n" + "=" * 80)
print("BUILDING MOBILE NET V2")
print("=" * 80)

base = tf.keras.applications.MobileNetV2(
    input_shape=(
        224,
        224,
        3
    ),
    include_top=False,
    weights="imagenet"
)

# Fine-tune upper layers
base.trainable = True

# Freeze early layers
for layer in base.layers[:-40]:

    layer.trainable = False

inputs = tf.keras.Input(
    shape=(
        224,
        224,
        3
    )
)

x = tf.keras.applications.mobilenet_v2.preprocess_input(
    inputs
)

x = base(
    x,
    training=True
)

x = tf.keras.layers.GlobalAveragePooling2D()(
    x
)

x = tf.keras.layers.BatchNormalization()(
    x
)

x = tf.keras.layers.Dropout(
    0.40
)(
    x
)

x = tf.keras.layers.Dense(
    128,
    activation="relu"
)(
    x
)

x = tf.keras.layers.Dropout(
    0.30
)(
    x
)

outputs = tf.keras.layers.Dense(
    3,
    activation="softmax"
)(
    x
)

model = tf.keras.Model(
    inputs,
    outputs
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy"
    ]
)

model.summary()

# =============================================================================
# CALLBACKS
# =============================================================================

BEST_MODEL = (
    MODEL_DIR /
    "model3b_step11_best.keras"
)

FINAL_MODEL = (
    MODEL_DIR /
    "model3b_step11_final.keras"
)

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

# =============================================================================
# TRAIN
# =============================================================================

print("\n" + "=" * 80)
print("STARTING IMPROVED TRAINING")
print("=" * 80)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

# =============================================================================
# SAVE
# =============================================================================

model.save(
    FINAL_MODEL
)

print(
    "\n✅ Model saved:"
)

print(
    FINAL_MODEL
)

# =============================================================================
# TEST
# =============================================================================

loss, accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print(
    "\nTEST ACCURACY:",
    f"{accuracy * 100:.2f}%"
)

# =============================================================================
# PREDICTIONS
# =============================================================================

y_true = []
y_pred = []

for images, labels in test_ds:

    probabilities = model.predict(
        images,
        verbose=0
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    y_true.extend(
        labels.numpy()
    )

    y_pred.extend(
        predictions
    )

y_true = np.array(
    y_true
)

y_pred = np.array(
    y_pred
)

# =============================================================================
# REPORT
# =============================================================================

report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

print(
    "\nClassification report:"
)

print(
    pd.DataFrame(
        report
    ).transpose()
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print(
    "\nConfusion matrix:"
)

print(
    pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    )
)

# =============================================================================
# SAVE RESULTS
# =============================================================================

pd.DataFrame(
    report
).transpose().to_csv(
    RESULT_DIR /
    "classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    RESULT_DIR /
    "confusion_matrix.csv"
)

pd.DataFrame(
    history.history
).to_csv(
    RESULT_DIR /
    "training_history.csv",
    index=False
)

summary = {

    "model":
        "MobileNetV2 fine-tuned",

    "accuracy":
        float(accuracy),

    "classes":
        CLASS_NAMES,

    "train_slices":
        len(train_df),

    "validation_slices":
        len(val_df),

    "test_slices":
        len(test_df)
}

with open(
    RESULT_DIR /
    "step11_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print("\n" + "=" * 80)
print("MODEL 3B - STEP 11 COMPLETE")
print("=" * 80)

print(
    f"\nFINAL TEST ACCURACY: "
    f"{accuracy * 100:.2f}%"
)

print(
    "\nModel:"
)

print(
    FINAL_MODEL
)

print(
    "\nResults:"
)

print(
    RESULT_DIR
)

print("=" * 80)

MODEL 3B - STEP 11
IMPROVED 3-CLASS CT TRAINING
Mounted at /content/drive

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Manifest: 1860
Missing: 0
✅ All 1860 files found

Train: 1290
Validation: 285
Test: 285
✅ Patient-level split verified

Training classes:
class_name
Pituitary           630
Brain_Metastasis    450
Meningioma          210
Name: count, dtype: int64

Class weights: {0: 2.0476190476190474, 1: 0.6825396825396826, 2: 0.9555555555555556}
✅ TensorFlow datasets ready

BUILDING MOBILE NET V2
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,427,459 (9.26 MB)

 Trainable params: 1,848,451 (7.05 MB)

 Non-trainable params: 579,008 (2.21 MB)


STARTING IMPROVED TRAINING
Epoch 1/15
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.6943 - loss: 0.9255
Epoch 1: val_accuracy improved from None to 0.36842, saving model to /content/drive/MyDrive/Model3B/models/model3b_step11_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Model3B/models/model3b_step11_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 433s 10s/step - accuracy: 0.7116 - loss: 0.8013 - val_accuracy: 0.3684 - val_loss: 1.2550 - learning_rate: 1.0000e-04
Epoch 2/15
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.7626 - loss: 0.6460
Epoch 2: val_accuracy did not improve from 0.36842
41/41 ━━━━━━━━━━━━━━━━━━━━ 20s 483ms/step - accuracy: 0.7535 - loss: 0.6428 - val_accuracy: 0.3684 - val_loss: 1.1395 - learning_rate: 1.0000e-04
Epoch 3/15
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 452ms/step - accuracy: 0.7745 - loss: 0.5001
Epoch 3: val_accuracy did not improve from 0.36842
41/41 ━━━━━━━━━━━━━━━━━━━━ 19s 470ms/step - accuracy: 0.7519 - loss: 0.5287 - val_accuracy: 

In [ ]:
!pip -q install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 780.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 142.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 108.7 MB/s eta 0:00:00


In [ ]:
# =============================================================================
# MODEL 3B - STEP 12
# DATA DISTRIBUTION DIAGNOSTIC
# =============================================================================

import numpy as np
import pandas as pd
from pathlib import Path

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

MANIFEST = (
    ROOT /
    "manifests" /
    "model3b_final_3class_slice_manifest.csv"
)

DATASET = (
    ROOT /
    "processed" /
    "3class"
)

df = pd.read_csv(
    MANIFEST
)

# Resolve paths
def resolve_path(p):

    p = Path(str(p))

    if p.exists():
        return p

    filename = p.name

    matches = list(
        DATASET.rglob(filename)
    )

    if matches:
        return matches[0]

    return p


df["drive_path"] = (
    df["path"]
    .apply(resolve_path)
)

print("=" * 80)
print("MODEL 3B - STEP 12")
print("DATA DISTRIBUTION DIAGNOSTIC")
print("=" * 80)

# =============================================================================
# BASIC DISTRIBUTION
# =============================================================================

print("\nCLASS DISTRIBUTION")
print(
    df["class_name"].value_counts()
)

print("\nCLASS × SPLIT")
print(
    pd.crosstab(
        df["split"],
        df["class_name"]
    )
)

# =============================================================================
# SAMPLE STATISTICS
# =============================================================================

rows = []

for idx, row in df.iterrows():

    if idx % 200 == 0:
        print(
            f"[{idx}/{len(df)}]"
        )

    img = np.load(
        row["drive_path"]
    ).astype(
        np.float32
    )

    rows.append(
        {
            "class_name":
                row["class_name"],

            "split":
                row["split"],

            "mean":
                float(img.mean()),

            "std":
                float(img.std()),

            "min":
                float(img.min()),

            "max":
                float(img.max()),

            "p01":
                float(np.percentile(img, 1)),

            "p50":
                float(np.percentile(img, 50)),

            "p99":
                float(np.percentile(img, 99))
        }
    )

stats = pd.DataFrame(
    rows
)

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("STATISTICS BY CLASS")
print("=" * 80)

print(
    stats.groupby(
        "class_name"
    )[
        [
            "mean",
            "std",
            "min",
            "max",
            "p01",
            "p50",
            "p99"
        ]
    ].mean()
)

print("\n" + "=" * 80)
print("STATISTICS BY SPLIT")
print("=" * 80)

print(
    stats.groupby(
        "split"
    )[
        [
            "mean",
            "std",
            "min",
            "max",
            "p01",
            "p50",
            "p99"
        ]
    ].mean()
)

print("\n" + "=" * 80)
print("CLASS × SPLIT MEANS")
print("=" * 80)

print(
    stats.groupby(
        [
            "split",
            "class_name"
        ]
    )[
        [
            "mean",
            "std",
            "p01",
            "p50",
            "p99"
        ]
    ].mean()
)

# =============================================================================
# SAVE
# =============================================================================

OUT = (
    ROOT /
    "results" /
    "step12"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

stats.to_csv(
    OUT /
    "slice_statistics.csv",
    index=False
)

print("\nSaved:")
print(
    OUT /
    "slice_statistics.csv"
)

print("\n" + "=" * 80)
print("STEP 12 COMPLETE")
print("=" * 80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MODEL 3B - STEP 12
DATA DISTRIBUTION DIAGNOSTIC

CLASS DISTRIBUTION
class_name
Pituitary           900
Brain_Metastasis    660
Meningioma          300
Name: count, dtype: int64

CLASS × SPLIT
class_name  Brain_Metastasis  Meningioma  Pituitary
split                                              
test                     105          45        135
train                    450         210        630
validation               105          45        135
[0/1860]
[200/1860]
[400/1860]
[600/1860]
[800/1860]
[1000/1860]
[1200/1860]
[1400/1860]
[1600/1860]
[1800/1860]

STATISTICS BY CLASS
                      mean       std  min       max       p01       p50  \
class_name                                                                
Brain_Metastasis  0.332157  0.032037  0.0  0.357899  0.333333  0.333333   
Meningioma        0.197562  0.213700  0.0  0.916504  0.00000

In [3]:
# =============================================================================
# MODEL 3B - STEP 13
# FAST NORMALIZED 3-CLASS CT TRAINING
# =============================================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from google.colab import drive
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print("=" * 80)
print("MODEL 3B - STEP 13")
print("FAST NORMALIZED 3-CLASS CT TRAINING")
print("=" * 80)

# =============================================================================
# 1. DRIVE
# =============================================================================

drive.mount(
    "/content/drive",
    force_remount=False
)

ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

DATASET = (
    ROOT / "processed" / "3class"
)

MANIFEST = (
    ROOT / "manifests" /
    "model3b_final_3class_slice_manifest.csv"
)

MODEL_DIR = (
    ROOT / "models"
)

RESULT_DIR = (
    ROOT / "results" / "step13"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# 2. GPU
# =============================================================================

gpus = tf.config.list_physical_devices("GPU")

print("\nGPU:", gpus)

if not gpus:
    raise RuntimeError(
        "GPU NOT DETECTED. Switch Colab runtime to GPU."
    )

print("✅ GPU AVAILABLE")

# =============================================================================
# 3. LOAD MANIFEST
# =============================================================================

df = pd.read_csv(
    MANIFEST
)

print(
    "\nManifest rows:",
    len(df)
)

if len(df) != 1860:
    raise RuntimeError(
        f"Expected 1860 rows, found {len(df)}"
    )

# =============================================================================
# 4. CLASSES
# =============================================================================

CLASS_NAMES = [
    "Meningioma",
    "Pituitary",
    "Brain_Metastasis"
]

CLASS_TO_ID = {
    name: i
    for i, name in enumerate(
        CLASS_NAMES
    )
}

df["label"] = (
    df["class_name"]
    .map(CLASS_TO_ID)
    .astype(np.int32)
)

# =============================================================================
# 5. RESOLVE DRIVE PATHS
# =============================================================================

def resolve_path(p):

    p = Path(str(p))

    if p.exists():
        return str(p)

    filename = p.name

    matches = list(
        DATASET.rglob(filename)
    )

    if matches:
        return str(matches[0])

    return str(p)


df["drive_path"] = (
    df["path"]
    .apply(resolve_path)
)

missing = [
    p
    for p in df["drive_path"]
    if not Path(p).exists()
]

print(
    "\nMissing files:",
    len(missing)
)

if missing:
    print(
        "\nFirst missing files:"
    )

    for p in missing[:10]:
        print(p)

    raise RuntimeError(
        "Dataset is incomplete."
    )

print(
    "✅ ALL 1860 FILES FOUND"
)

# =============================================================================
# 6. SPLITS
# =============================================================================

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

print("\n" + "=" * 80)
print("DATASET SPLIT")
print("=" * 80)

print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(val_df)
)

print(
    "Test:",
    len(test_df)
)

# =============================================================================
# 7. PATIENT LEAKAGE CHECK
# =============================================================================

train_patients = set(
    train_df["patient_key"]
)

val_patients = set(
    val_df["patient_key"]
)

test_patients = set(
    test_df["patient_key"]
)

assert not (
    train_patients &
    val_patients
)

assert not (
    train_patients &
    test_patients
)

assert not (
    val_patients &
    test_patients
)

print(
    "✅ NO PATIENT LEAKAGE"
)

# =============================================================================
# 8. CLASS DISTRIBUTION
# =============================================================================

print("\nClass distribution:")

print(
    train_df["class_name"]
    .value_counts()
)

# =============================================================================
# 9. CLASS WEIGHTS
#
# Mild weighting instead of aggressive full balancing.
# =============================================================================

counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
    .values
)

total = counts.sum()

class_weights = {
    i: float(
        np.sqrt(
            total /
            (len(counts) * counts[i])
        )
    )
    for i in range(
        len(counts)
    )
}

print(
    "\nMild class weights:",
    class_weights
)

# =============================================================================
# 10. LOAD + PER-IMAGE NORMALIZATION
# =============================================================================

IMG_SIZE = 224
BATCH_SIZE = 32

def load_image(
    path,
    label
):

    path = path.decode(
        "utf-8"
    )

    image = np.load(
        path
    ).astype(
        np.float32
    )

    # ---------------------------------------------------------
    # Per-image normalization
    # ---------------------------------------------------------

    lo = np.percentile(
        image,
        1
    )

    hi = np.percentile(
        image,
        99
    )

    if hi > lo:

        image = (
            image - lo
        ) / (
            hi - lo
        )

    else:

        image = np.zeros_like(
            image
        )

    image = np.clip(
        image,
        0.0,
        1.0
    )

    # ---------------------------------------------------------
    # Convert grayscale → RGB
    # ---------------------------------------------------------

    image = np.stack(
        [
            image,
            image,
            image
        ],
        axis=-1
    )

    return (
        image.astype(
            np.float32
        ),
        np.int32(label)
    )


def tf_load(
    path,
    label
):

    image, label = tf.numpy_function(
        load_image,
        [
            path,
            label
        ],
        [
            tf.float32,
            tf.int32
        ]
    )

    image.set_shape(
        (
            IMG_SIZE,
            IMG_SIZE,
            3
        )
    )

    label.set_shape(())

    return (
        image,
        label
    )

# =============================================================================
# 11. DATASETS
# =============================================================================

def make_dataset(
    frame,
    training=False
):

    ds = tf.data.Dataset.from_tensor_slices(
        (
            frame["drive_path"].values,
            frame["label"].values
        )
    )

    if training:

        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=42,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        tf_load,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


train_ds = make_dataset(
    train_df,
    True
)

val_ds = make_dataset(
    val_df,
    False
)

test_ds = make_dataset(
    test_df,
    False
)

print(
    "\n✅ TensorFlow datasets ready"
)

# =============================================================================
# 12. MODEL
# =============================================================================

print("\n" + "=" * 80)
print("BUILDING MOBILE NET V2")
print("=" * 80)

base = tf.keras.applications.MobileNetV2(
    input_shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    ),
    include_top=False,
    weights="imagenet"
)

# IMPORTANT:
# Keep the backbone frozen initially.
base.trainable = False

inputs = tf.keras.Input(
    shape=(
        IMG_SIZE,
        IMG_SIZE,
        3
    )
)

x = tf.keras.applications.mobilenet_v2.preprocess_input(
    inputs
)

x = base(
    x,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(
    x
)

x = tf.keras.layers.Dropout(
    0.30
)(
    x
)

x = tf.keras.layers.Dense(
    128,
    activation="relu"
)(
    x
)

x = tf.keras.layers.Dropout(
    0.20
)(
    x
)

outputs = tf.keras.layers.Dense(
    3,
    activation="softmax"
)(
    x
)

model = tf.keras.Model(
    inputs,
    outputs
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy"
    ]
)

model.summary()

# =============================================================================
# 13. CALLBACKS
# =============================================================================

BEST_MODEL = (
    MODEL_DIR /
    "model3b_step13_best.keras"
)

FINAL_MODEL = (
    MODEL_DIR /
    "model3b_step13_final.keras"
)

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# =============================================================================
# 14. TRAIN
# =============================================================================

print("\n" + "=" * 80)
print("STARTING STEP 13 TRAINING")
print("=" * 80)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

# =============================================================================
# 15. SAVE
# =============================================================================

model.save(
    FINAL_MODEL
)

print(
    "\n✅ FINAL MODEL SAVED:"
)

print(
    FINAL_MODEL
)

# =============================================================================
# 16. TEST
# =============================================================================

print("\n" + "=" * 80)
print("TEST EVALUATION")
print("=" * 80)

test_loss, test_accuracy = (
    model.evaluate(
        test_ds,
        verbose=1
    )
)

print(
    f"\nTest Loss: {test_loss:.4f}"
)

print(
    f"Test Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

# =============================================================================
# 17. PREDICTIONS
# =============================================================================

y_true = []
y_pred = []

for images, labels in test_ds:

    probabilities = model.predict(
        images,
        verbose=0
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    y_true.extend(
        labels.numpy()
    )

    y_pred.extend(
        predictions
    )

y_true = np.array(
    y_true
)

y_pred = np.array(
    y_pred
)

# =============================================================================
# 18. REPORT
# =============================================================================

report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)

print(
    "\nClassification report:"
)

print(
    report
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print(
    "\nConfusion matrix:"
)

print(
    pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    )
)

# =============================================================================
# 19. SAVE RESULTS
# =============================================================================

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    RESULT_DIR /
    "confusion_matrix.csv"
)

with open(
    RESULT_DIR /
    "classification_report.txt",
    "w"
) as f:

    f.write(report)

pd.DataFrame(
    history.history
).to_csv(
    RESULT_DIR /
    "training_history.csv",
    index=False
)

summary = {
    "model": "MobileNetV2",
    "normalization": "per-image percentile 1-99",
    "train_slices": int(len(train_df)),
    "validation_slices": int(len(val_df)),
    "test_slices": int(len(test_df)),
    "test_accuracy": float(
        test_accuracy
    ),
    "classes": CLASS_NAMES,
    "dataset": str(DATASET)
}

with open(
    RESULT_DIR /
    "step13_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# 20. COMPLETE
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 3B - STEP 13 COMPLETE")
print("=" * 80)

print(
    f"\nFINAL TEST ACCURACY: "
    f"{test_accuracy * 100:.2f}%"
)

print(
    "\nBest model:"
)

print(
    BEST_MODEL
)

print(
    "\nFinal model:"
)

print(
    FINAL_MODEL
)

print(
    "\nResults:"
)

print(
    RESULT_DIR
)

print("=" * 80)

MODEL 3B - STEP 13
FAST NORMALIZED 3-CLASS CT TRAINING
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ GPU AVAILABLE

Manifest rows: 1860

Missing files: 0
✅ ALL 1860 FILES FOUND

DATASET SPLIT
Train: 1290
Validation: 285
Test: 285
✅ NO PATIENT LEAKAGE

Class distribution:
class_name
Pituitary           630
Brain_Metastasis    450
Meningioma          210
Name: count, dtype: int64

Mild class weights: {0: 1.4309504001254019, 1: 0.8261595987094035, 2: 0.9775252199076787}

✅ TensorFlow datasets ready

BUILDING MOBILE NET V2


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


STARTING STEP 13 TRAINING
Epoch 1/12
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step - accuracy: 0.4193 - loss: 1.2150
Epoch 1: val_accuracy improved from None to 0.36842, saving model to /content/drive/MyDrive/Model3B/models/model3b_step13_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Model3B/models/model3b_step13_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 28s 460ms/step - accuracy: 0.4535 - loss: 1.1235 - val_accuracy: 0.3684 - val_loss: 0.9741 - learning_rate: 0.0010
Epoch 2/12
40/41 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.5143 - loss: 0.9823
Epoch 2: val_accuracy improved from 0.36842 to 0.80702, saving model to /content/drive/MyDrive/Model3B/models/model3b_step13_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Model3B/models/model3b_step13_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 128ms/step - accuracy: 0.5636 - loss: 0.9478 - val_accuracy: 0.8070 - val_loss: 0.7683 - learning_rate: 0.0010
Epoch 3/12
40/41 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step -

In [4]:
# =============================================================================
# MODEL 3B - STEP 14
# TARGETED MENINGIOMA FINE-TUNING
# =============================================================================

import numpy as np
import pandas as pd
import tensorflow as tf
import json

from pathlib import Path
from google.colab import drive
from sklearn.metrics import classification_report, confusion_matrix

print("=" * 80)
print("MODEL 3B - STEP 14")
print("TARGETED MENINGIOMA FINE-TUNING")
print("=" * 80)

# =============================================================================
# 1. DRIVE + GPU
# =============================================================================

drive.mount(
    "/content/drive",
    force_remount=False
)

ROOT = Path("/content/drive/MyDrive/Model3B")

DATASET = ROOT / "processed" / "3class"

MANIFEST = (
    ROOT / "manifests" /
    "model3b_final_3class_slice_manifest.csv"
)

MODEL_PATH = (
    ROOT / "models" /
    "model3b_step13_best.keras"
)

MODEL_DIR = ROOT / "models"
RESULT_DIR = ROOT / "results" / "step14"

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

gpus = tf.config.list_physical_devices("GPU")

print("\nGPU:", gpus)

if not gpus:
    raise RuntimeError(
        "GPU NOT DETECTED"
    )

# =============================================================================
# 2. LOAD DATA
# =============================================================================

df = pd.read_csv(MANIFEST)

print(
    "\nManifest rows:",
    len(df)
)

CLASS_NAMES = [
    "Meningioma",
    "Pituitary",
    "Brain_Metastasis"
]

CLASS_TO_ID = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}

df["label"] = (
    df["class_name"]
    .map(CLASS_TO_ID)
    .astype(np.int32)
)

# =============================================================================
# 3. RESOLVE PATHS
# =============================================================================

def resolve_path(p):

    p = Path(str(p))

    if p.exists():
        return str(p)

    matches = list(
        DATASET.rglob(p.name)
    )

    if matches:
        return str(matches[0])

    return str(p)


df["drive_path"] = (
    df["path"]
    .apply(resolve_path)
)

missing = [
    p for p in df["drive_path"]
    if not Path(p).exists()
]

print(
    "Missing files:",
    len(missing)
)

if missing:
    raise RuntimeError(
        "Missing processed files."
    )

# =============================================================================
# 4. SPLITS
# =============================================================================

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

print(
    "\nTrain:",
    len(train_df)
)

print(
    "Validation:",
    len(val_df)
)

print(
    "Test:",
    len(test_df)
)

# =============================================================================
# 5. PATIENT LEAKAGE
# =============================================================================

train_patients = set(
    train_df["patient_key"]
)

val_patients = set(
    val_df["patient_key"]
)

test_patients = set(
    test_df["patient_key"]
)

assert not (
    train_patients & val_patients
)

assert not (
    train_patients & test_patients
)

assert not (
    val_patients & test_patients
)

print(
    "✅ Patient split verified"
)

# =============================================================================
# 6. LOAD + PER-IMAGE NORMALIZATION
# =============================================================================

def load_image(
    path,
    label
):

    path = path.decode(
        "utf-8"
    )

    image = np.load(
        path
    ).astype(
        np.float32
    )

    lo = np.percentile(
        image,
        1
    )

    hi = np.percentile(
        image,
        99
    )

    if hi > lo:

        image = (
            image - lo
        ) / (
            hi - lo
        )

    else:

        image = np.zeros_like(
            image
        )

    image = np.clip(
        image,
        0,
        1
    )

    image = np.stack(
        [
            image,
            image,
            image
        ],
        axis=-1
    )

    return (
        image.astype(
            np.float32
        ),
        np.int32(label)
    )


def tf_load(
    path,
    label
):

    image, label = tf.numpy_function(
        load_image,
        [path, label],
        [tf.float32, tf.int32]
    )

    image.set_shape(
        (224, 224, 3)
    )

    label.set_shape(())

    return image, label


# =============================================================================
# 7. DATASETS
# =============================================================================

BATCH_SIZE = 32


def make_dataset(
    frame,
    training=False
):

    ds = tf.data.Dataset.from_tensor_slices(
        (
            frame["drive_path"].values,
            frame["label"].values
        )
    )

    if training:

        ds = ds.shuffle(
            len(frame),
            seed=42,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        tf_load,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


train_ds = make_dataset(
    train_df,
    True
)

val_ds = make_dataset(
    val_df,
    False
)

test_ds = make_dataset(
    test_df,
    False
)

print(
    "✅ TensorFlow datasets ready"
)

# =============================================================================
# 8. LOAD STEP 13 BEST MODEL
# =============================================================================

print("\n" + "=" * 80)
print("LOADING STEP 13 BEST MODEL")
print("=" * 80)

if not MODEL_PATH.exists():

    raise RuntimeError(
        f"Step 13 model not found:\n{MODEL_PATH}"
    )

model = tf.keras.models.load_model(
    MODEL_PATH
)

print(
    "✅ Loaded:",
    MODEL_PATH
)

# =============================================================================
# 9. FIND MOBILE NET BACKBONE
# =============================================================================

base = None

for layer in model.layers:

    if isinstance(
        layer,
        tf.keras.Model
    ):

        if "mobilenet" in layer.name.lower():

            base = layer
            break

if base is None:

    raise RuntimeError(
        "MobileNetV2 backbone not found."
    )

print(
    "\nBackbone:",
    base.name
)

# =============================================================================
# 10. FINE-TUNE ONLY UPPER LAYERS
# =============================================================================

base.trainable = True

# Freeze everything except final ~20 layers
for layer in base.layers[:-20]:

    layer.trainable = False

for layer in base.layers[-20:]:

    layer.trainable = True

# Keep BatchNorm frozen for stability
for layer in base.layers:

    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):

        layer.trainable = False

trainable_count = sum(
    np.prod(v.shape)
    for v in model.trainable_weights
)

print(
    "\nTrainable parameters:",
    trainable_count
)

# =============================================================================
# 11. MENINGIOMA-TARGETED WEIGHTS
# =============================================================================

# Mildly increase Meningioma importance.
# Do NOT use extreme weighting.

class_weights = {
    0: 1.8,   # Meningioma
    1: 0.95,  # Pituitary
    2: 1.00   # Brain Metastasis
}

print(
    "\nFine-tuning weights:",
    class_weights
)

# =============================================================================
# 12. COMPILE
# =============================================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# =============================================================================
# 13. CALLBACKS
# =============================================================================

BEST_MODEL = (
    MODEL_DIR /
    "model3b_step14_best.keras"
)

FINAL_MODEL = (
    MODEL_DIR /
    "model3b_step14_final.keras"
)

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

# =============================================================================
# 14. TRAIN
# =============================================================================

print("\n" + "=" * 80)
print("STARTING STEP 14 FINE-TUNING")
print("=" * 80)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

# =============================================================================
# 15. SAVE
# =============================================================================

model.save(
    FINAL_MODEL
)

print(
    "\n✅ FINAL MODEL SAVED:"
)

print(
    FINAL_MODEL
)

# =============================================================================
# 16. TEST
# =============================================================================

print("\n" + "=" * 80)
print("STEP 14 TEST EVALUATION")
print("=" * 80)

loss, accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print(
    f"\nTest Accuracy: "
    f"{accuracy * 100:.2f}%"
)

# =============================================================================
# 17. PREDICTIONS
# =============================================================================

y_true = []
y_pred = []

for images, labels in test_ds:

    probabilities = model.predict(
        images,
        verbose=0
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    y_true.extend(
        labels.numpy()
    )

    y_pred.extend(
        predictions
    )

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# =============================================================================
# 18. REPORT
# =============================================================================

report_text = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print(
    "\nClassification report:"
)

print(
    report_text
)

print(
    "\nConfusion matrix:"
)

print(
    pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    )
)

# =============================================================================
# 19. SAVE RESULTS
# =============================================================================

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    RESULT_DIR /
    "confusion_matrix.csv"
)

with open(
    RESULT_DIR /
    "classification_report.txt",
    "w"
) as f:

    f.write(
        report_text
    )

pd.DataFrame(
    history.history
).to_csv(
    RESULT_DIR /
    "training_history.csv",
    index=False
)

summary = {

    "starting_model":
        "model3b_step13_best.keras",

    "final_model":
        str(FINAL_MODEL),

    "test_accuracy":
        float(accuracy),

    "classes":
        CLASS_NAMES,

    "meningioma_weight":
        1.8,

    "learning_rate":
        1e-5
}

with open(
    RESULT_DIR /
    "step14_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# COMPLETE
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 3B - STEP 14 COMPLETE")
print("=" * 80)

print(
    f"\nFINAL TEST ACCURACY: "
    f"{accuracy * 100:.2f}%"
)

print(
    "\nBest model:"
)

print(
    BEST_MODEL
)

print(
    "\nFinal model:"
)

print(
    FINAL_MODEL
)

print(
    "\nResults:"
)

print(
    RESULT_DIR
)

print("=" * 80)

MODEL 3B - STEP 14
TARGETED MENINGIOMA FINE-TUNING
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Manifest rows: 1860
Missing files: 0

Train: 1290
Validation: 285
Test: 285
✅ Patient split verified
✅ TensorFlow datasets ready

LOADING STEP 13 BEST MODEL
✅ Loaded: /content/drive/MyDrive/Model3B/models/model3b_step13_best.keras

Backbone: mobilenetv2_1.00_224

Trainable parameters: 1359235

Fine-tuning weights: {0: 1.8, 1: 0.95, 2: 1.0}

STARTING STEP 14 FINE-TUNING
Epoch 1/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - accuracy: 0.6478 - loss: 0.9569
Epoch 1: val_accuracy improved from None to 0.80702, saving model to /content/drive/MyDrive/Model3B/models/model3b_step14_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Model3B/models/model3b_step14_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 32s 493ms/step - accuracy: 0.

In [5]:
# =============================================================================
# MODEL 3B - STEP 15
# FINAL MENINGIOMA-FOCUSED TRAINING ATTEMPT
# =============================================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from google.colab import drive
from sklearn.metrics import classification_report, confusion_matrix

print("=" * 80)
print("MODEL 3B - STEP 15")
print("FINAL MENINGIOMA-FOCUSED TRAINING ATTEMPT")
print("=" * 80)

# =============================================================================
# 1. DRIVE
# =============================================================================

drive.mount(
    "/content/drive",
    force_remount=False
)

ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

DATASET = (
    ROOT / "processed" / "3class"
)

MANIFEST = (
    ROOT / "manifests" /
    "model3b_final_3class_slice_manifest.csv"
)

START_MODEL = (
    ROOT / "models" /
    "model3b_step14_best.keras"
)

MODEL_DIR = (
    ROOT / "models"
)

RESULT_DIR = (
    ROOT / "results" /
    "step15"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# 2. GPU
# =============================================================================

gpus = tf.config.list_physical_devices(
    "GPU"
)

print(
    "\nGPU:",
    gpus
)

if not gpus:
    raise RuntimeError(
        "GPU NOT DETECTED."
    )

# =============================================================================
# 3. LOAD MANIFEST
# =============================================================================

df = pd.read_csv(
    MANIFEST
)

print(
    "\nManifest rows:",
    len(df)
)

if len(df) != 1860:
    raise RuntimeError(
        "Expected 1860 rows."
    )

# =============================================================================
# 4. CLASSES
# =============================================================================

CLASS_NAMES = [
    "Meningioma",
    "Pituitary",
    "Brain_Metastasis"
]

CLASS_TO_ID = {
    name: i
    for i, name in enumerate(
        CLASS_NAMES
    )
}

df["label"] = (
    df["class_name"]
    .map(CLASS_TO_ID)
    .astype(np.int32)
)

# =============================================================================
# 5. RESOLVE FILES
# =============================================================================

def resolve_path(p):

    p = Path(
        str(p)
    )

    if p.exists():
        return str(p)

    matches = list(
        DATASET.rglob(
            p.name
        )
    )

    if matches:
        return str(
            matches[0]
        )

    return str(p)


df["drive_path"] = (
    df["path"]
    .apply(resolve_path)
)

missing = [
    p
    for p in df["drive_path"]
    if not Path(p).exists()
]

print(
    "Missing files:",
    len(missing)
)

if missing:
    raise RuntimeError(
        "Processed dataset incomplete."
    )

print(
    "✅ ALL FILES FOUND"
)

# =============================================================================
# 6. SPLITS
# =============================================================================

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

print("\nTrain:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# =============================================================================
# 7. PATIENT LEAKAGE
# =============================================================================

tr_patients = set(
    train_df["patient_key"]
)

va_patients = set(
    val_df["patient_key"]
)

te_patients = set(
    test_df["patient_key"]
)

assert not (
    tr_patients & va_patients
)

assert not (
    tr_patients & te_patients
)

assert not (
    va_patients & te_patients
)

print(
    "✅ NO PATIENT LEAKAGE"
)

# =============================================================================
# 8. NORMALIZATION
# =============================================================================

def load_image(
    path,
    label
):

    path = path.decode(
        "utf-8"
    )

    image = np.load(
        path
    ).astype(
        np.float32
    )

    # Per-slice percentile normalization
    lo = np.percentile(
        image,
        1
    )

    hi = np.percentile(
        image,
        99
    )

    if hi > lo:

        image = (
            image - lo
        ) / (
            hi - lo
        )

    else:

        image = np.zeros_like(
            image
        )

    image = np.clip(
        image,
        0,
        1
    )

    # grayscale -> RGB
    image = np.stack(
        [
            image,
            image,
            image
        ],
        axis=-1
    )

    return (
        image.astype(
            np.float32
        ),
        np.int32(label)
    )


def tf_load(
    path,
    label
):

    image, label = tf.numpy_function(
        load_image,
        [path, label],
        [tf.float32, tf.int32]
    )

    image.set_shape(
        (224, 224, 3)
    )

    label.set_shape(())

    return (
        image,
        label
    )

# =============================================================================
# 9. DATASET
# =============================================================================

BATCH_SIZE = 32


def make_dataset(
    frame,
    training=False
):

    ds = tf.data.Dataset.from_tensor_slices(
        (
            frame["drive_path"].values,
            frame["label"].values
        )
    )

    if training:

        ds = ds.shuffle(
            len(frame),
            seed=123,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        tf_load,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


train_ds = make_dataset(
    train_df,
    True
)

val_ds = make_dataset(
    val_df,
    False
)

test_ds = make_dataset(
    test_df,
    False
)

print(
    "✅ DATASETS READY"
)

# =============================================================================
# 10. LOAD STEP 14 BEST MODEL
# =============================================================================

print("\n" + "=" * 80)
print("LOADING STEP 14 BEST MODEL")
print("=" * 80)

if not START_MODEL.exists():

    raise RuntimeError(
        f"Model not found:\n{START_MODEL}"
    )

model = tf.keras.models.load_model(
    START_MODEL
)

print(
    "Loaded:",
    START_MODEL
)

# =============================================================================
# 11. FIND MOBILE NET BACKBONE
# =============================================================================

base = None

for layer in model.layers:

    if isinstance(
        layer,
        tf.keras.Model
    ):

        if "mobilenet" in (
            layer.name.lower()
        ):

            base = layer
            break

if base is None:

    raise RuntimeError(
        "MobileNetV2 backbone not found."
    )

print(
    "Backbone:",
    base.name
)

# =============================================================================
# 12. VERY LIMITED FINE-TUNING
# =============================================================================

base.trainable = True

# Freeze almost everything.
#
# Only final 8 layers are trainable.
#
# This is deliberately conservative.

for layer in base.layers:

    layer.trainable = False

for layer in base.layers[-8:]:

    layer.trainable = True

# BatchNorm must remain frozen.
for layer in base.layers:

    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):

        layer.trainable = False

# =============================================================================
# 13. MENINGIOMA EMPHASIS
# =============================================================================

class_weights = {

    0: 3.0,   # Meningioma

    1: 0.85,  # Pituitary

    2: 1.0    # Brain Metastasis
}

print(
    "\nClass weights:"
)

print(
    class_weights
)

# =============================================================================
# 14. COMPILE
# =============================================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=3e-6
    ),

    loss=(
        "sparse_categorical_crossentropy"
    ),

    metrics=[
        "accuracy"
    ]
)

# =============================================================================
# 15. CALLBACKS
# =============================================================================

BEST_MODEL = (
    MODEL_DIR /
    "model3b_step15_best.keras"
)

FINAL_MODEL = (
    MODEL_DIR /
    "model3b_step15_final.keras"
)

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

# =============================================================================
# 16. TRAIN
# =============================================================================

print("\n" + "=" * 80)
print("STARTING FINAL TRAINING ATTEMPT")
print("=" * 80)

history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=8,

    class_weight=class_weights,

    callbacks=callbacks,

    verbose=1
)

# =============================================================================
# 17. SAVE
# =============================================================================

model.save(
    FINAL_MODEL
)

print(
    "\n✅ MODEL SAVED:"
)

print(
    FINAL_MODEL
)

# =============================================================================
# 18. TEST
# =============================================================================

print("\n" + "=" * 80)
print("STEP 15 TEST EVALUATION")
print("=" * 80)

test_loss, test_accuracy = (
    model.evaluate(
        test_ds,
        verbose=1
    )
)

print(
    f"\nTEST ACCURACY: "
    f"{test_accuracy * 100:.2f}%"
)

# =============================================================================
# 19. PREDICTIONS
# =============================================================================

y_true = []
y_pred = []

for images, labels in test_ds:

    probabilities = model.predict(
        images,
        verbose=0
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    y_true.extend(
        labels.numpy()
    )

    y_pred.extend(
        predictions
    )

y_true = np.array(
    y_true
)

y_pred = np.array(
    y_pred
)

# =============================================================================
# 20. REPORT
# =============================================================================

report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred
)

print(
    "\nClassification report:"
)

print(
    report
)

print(
    "\nConfusion matrix:"
)

print(
    pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    )
)

# =============================================================================
# 21. SAVE RESULTS
# =============================================================================

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    RESULT_DIR /
    "confusion_matrix.csv"
)

with open(
    RESULT_DIR /
    "classification_report.txt",
    "w"
) as f:

    f.write(
        report
    )

pd.DataFrame(
    history.history
).to_csv(
    RESULT_DIR /
    "training_history.csv",
    index=False
)

summary = {

    "starting_model":
        str(START_MODEL),

    "test_accuracy":
        float(test_accuracy),

    "classes":
        CLASS_NAMES,

    "class_weights":
        class_weights,

    "learning_rate":
        3e-6,

    "trainable_backbone_layers":
        8

}

with open(
    RESULT_DIR /
    "step15_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# COMPLETE
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 3B - STEP 15 COMPLETE")
print("=" * 80)

print(
    f"\nFINAL TEST ACCURACY: "
    f"{test_accuracy * 100:.2f}%"
)

print(
    "\nBest model:"
)

print(
    BEST_MODEL
)

print(
    "\nFinal model:"
)

print(
    FINAL_MODEL
)

print(
    "\nResults:"
)

print(
    RESULT_DIR
)

print("=" * 80)

MODEL 3B - STEP 15
FINAL MENINGIOMA-FOCUSED TRAINING ATTEMPT
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Manifest rows: 1860
Missing files: 0
✅ ALL FILES FOUND

Train: 1290
Validation: 285
Test: 285
✅ NO PATIENT LEAKAGE
✅ DATASETS READY

LOADING STEP 14 BEST MODEL
Loaded: /content/drive/MyDrive/Model3B/models/model3b_step14_best.keras
Backbone: mobilenetv2_1.00_224

Class weights:
{0: 3.0, 1: 0.85, 2: 1.0}

STARTING FINAL TRAINING ATTEMPT
Epoch 1/8
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step - accuracy: 0.6872 - loss: 0.7973
Epoch 1: val_accuracy improved from None to 0.49123, saving model to /content/drive/MyDrive/Model3B/models/model3b_step15_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Model3B/models/model3b_step15_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 29s 447ms/step - accuracy: 0.6713 - loss: 0.7976 - val

In [6]:
# =============================================================================
# MODEL 3B - STEP 16
# FINAL INTERNAL TEST + MODEL EXPORT
# =============================================================================

import os
import json
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from google.colab import drive, files
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    accuracy_score
)

print("=" * 80)
print("MODEL 3B - STEP 16")
print("FINAL INTERNAL TEST + MODEL EXPORT")
print("=" * 80)

# =============================================================================
# 1. DRIVE
# =============================================================================

drive.mount(
    "/content/drive",
    force_remount=False
)

ROOT = Path(
    "/content/drive/MyDrive/Model3B"
)

DATASET = (
    ROOT / "processed" / "3class"
)

MANIFEST = (
    ROOT / "manifests" /
    "model3b_final_3class_slice_manifest.csv"
)

MODEL_PATH = (
    ROOT / "models" /
    "model3b_step14_best.keras"
)

RESULT_DIR = (
    ROOT / "results" /
    "step16_final_internal"
)

EXPORT_DIR = (
    ROOT / "exports"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# 2. VERIFY MODEL
# =============================================================================

print("\n" + "=" * 80)
print("MODEL CHECK")
print("=" * 80)

print(
    "Model:",
    MODEL_PATH
)

if not MODEL_PATH.exists():

    raise FileNotFoundError(
        MODEL_PATH
    )

model = tf.keras.models.load_model(
    MODEL_PATH
)

print(
    "✅ Model loaded"
)

# =============================================================================
# 3. LOAD MANIFEST
# =============================================================================

df = pd.read_csv(
    MANIFEST
)

print(
    "\nManifest rows:",
    len(df)
)

if len(df) != 1860:

    raise RuntimeError(
        "Expected 1860 manifest rows."
    )

CLASS_NAMES = [
    "Meningioma",
    "Pituitary",
    "Brain_Metastasis"
]

CLASS_TO_ID = {
    x: i
    for i, x in enumerate(
        CLASS_NAMES
    )
}

df["label"] = (
    df["class_name"]
    .map(CLASS_TO_ID)
    .astype(np.int32)
)

# =============================================================================
# 4. TEST SET
# =============================================================================

test_df = df[
    df["split"] == "test"
].copy()

print(
    "\nTest slices:",
    len(test_df)
)

print(
    "Test patients:",
    test_df["patient_key"]
    .nunique()
)

# =============================================================================
# 5. VERIFY PATIENT SPLIT
# =============================================================================

train_patients = set(
    df[
        df["split"] == "train"
    ]["patient_key"]
)

val_patients = set(
    df[
        df["split"] == "validation"
    ]["patient_key"]
)

test_patients = set(
    test_df["patient_key"]
)

assert not (
    train_patients &
    test_patients
)

assert not (
    val_patients &
    test_patients
)

print(
    "✅ No patient leakage"
)

# =============================================================================
# 6. RESOLVE FILES
# =============================================================================

def resolve_path(p):

    p = Path(
        str(p)
    )

    if p.exists():

        return str(p)

    matches = list(
        DATASET.rglob(
            p.name
        )
    )

    if matches:

        return str(
            matches[0]
        )

    return str(p)


test_df["drive_path"] = (
    test_df["path"]
    .apply(resolve_path)
)

missing = [
    p
    for p in test_df["drive_path"]
    if not Path(p).exists()
]

print(
    "Missing test files:",
    len(missing)
)

if missing:

    raise RuntimeError(
        "Test dataset incomplete."
    )

# =============================================================================
# 7. NORMALIZATION
# =============================================================================

def preprocess_image(path):

    image = np.load(
        path
    ).astype(
        np.float32
    )

    lo = np.percentile(
        image,
        1
    )

    hi = np.percentile(
        image,
        99
    )

    if hi > lo:

        image = (
            image - lo
        ) / (
            hi - lo
        )

    else:

        image = np.zeros_like(
            image
        )

    image = np.clip(
        image,
        0,
        1
    )

    image = np.stack(
        [
            image,
            image,
            image
        ],
        axis=-1
    )

    return image.astype(
        np.float32
    )

# =============================================================================
# 8. PREDICT
# =============================================================================

print("\n" + "=" * 80)
print("GENERATING INTERNAL TEST PREDICTIONS")
print("=" * 80)

y_true = []
y_pred = []
probabilities_all = []

for i, row in test_df.reset_index(
    drop=True
).iterrows():

    image = preprocess_image(
        row["drive_path"]
    )

    image = np.expand_dims(
        image,
        axis=0
    )

    probability = model.predict(
        image,
        verbose=0
    )[0]

    prediction = int(
        np.argmax(
            probability
        )
    )

    y_true.append(
        int(row["label"])
    )

    y_pred.append(
        prediction
    )

    probabilities_all.append(
        probability
    )

    if (
        (i + 1) % 50 == 0
        or
        (i + 1) == len(test_df)
    ):

        print(
            f"[{i + 1}/{len(test_df)}]"
        )

y_true = np.array(
    y_true
)

y_pred = np.array(
    y_pred
)

probabilities_all = np.array(
    probabilities_all
)

# =============================================================================
# 9. METRICS
# =============================================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)

balanced_acc = balanced_accuracy_score(
    y_true,
    y_pred
)

cm = confusion_matrix(
    y_true,
    y_pred
)

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_text = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)

# =============================================================================
# 10. RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("FINAL INTERNAL TEST RESULTS")
print("=" * 80)

print(
    f"\nSlice Accuracy: "
    f"{accuracy * 100:.2f}%"
)

print(
    f"Balanced Accuracy: "
    f"{balanced_acc * 100:.2f}%"
)

print(
    "\nClassification report:"
)

print(
    report_text
)

print(
    "\nConfusion matrix:"
)

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print(
    cm_df
)

# =============================================================================
# 11. PATIENT-LEVEL EVALUATION
# =============================================================================

print("\n" + "=" * 80)
print("PATIENT-LEVEL INTERNAL TEST")
print("=" * 80)

prediction_df = test_df.reset_index(
    drop=True
).copy()

prediction_df["true_label"] = y_true

prediction_df["pred_label"] = y_pred

prediction_df["true_class"] = (
    prediction_df["true_label"]
    .map(
        {
            i: name
            for i, name in enumerate(
                CLASS_NAMES
            )
        }
    )
)

prediction_df["pred_class"] = (
    prediction_df["pred_label"]
    .map(
        {
            i: name
            for i, name in enumerate(
                CLASS_NAMES
            )
        }
    )
)

patient_rows = []

for patient_key, group in (
    prediction_df
    .groupby("patient_key")
):

    true_label = int(
        group["true_label"]
        .iloc[0]
    )

    # Majority vote over slices
    pred_label = int(
        group["pred_label"]
        .value_counts()
        .idxmax()
    )

    patient_rows.append(
        {
            "patient_key":
                patient_key,

            "true_label":
                true_label,

            "pred_label":
                pred_label,

            "true_class":
                CLASS_NAMES[
                    true_label
                ],

            "pred_class":
                CLASS_NAMES[
                    pred_label
                ],

            "correct":
                true_label == pred_label
        }
    )

patient_df = pd.DataFrame(
    patient_rows
)

patient_accuracy = (
    patient_df["correct"]
    .mean()
)

patient_cm = confusion_matrix(
    patient_df["true_label"],
    patient_df["pred_label"]
)

print(
    "Patients:",
    len(patient_df)
)

print(
    f"Patient Accuracy: "
    f"{patient_accuracy * 100:.2f}%"
)

print(
    "\nPatient confusion matrix:"
)

print(
    pd.DataFrame(
        patient_cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    )
)

# =============================================================================
# 12. SAVE RESULTS
# =============================================================================

cm_df.to_csv(
    RESULT_DIR /
    "slice_confusion_matrix.csv"
)

patient_df.to_csv(
    RESULT_DIR /
    "patient_predictions.csv",
    index=False
)

prediction_df.to_csv(
    RESULT_DIR /
    "slice_predictions.csv",
    index=False
)

with open(
    RESULT_DIR /
    "classification_report.txt",
    "w"
) as f:

    f.write(
        report_text
    )

summary = {

    "model":
        "model3b_step14_best.keras",

    "dataset":
        "Model3B 3-class CT",

    "classes":
        CLASS_NAMES,

    "total_slices":
        int(len(df)),

    "test_slices":
        int(len(test_df)),

    "test_patients":
        int(test_df["patient_key"].nunique()),

    "slice_accuracy":
        float(accuracy),

    "balanced_accuracy":
        float(balanced_acc),

    "patient_accuracy":
        float(patient_accuracy),

    "patient_count":
        int(len(patient_df)),

    "external_test":
        "NOT PERFORMED - separate external dataset required"

}

with open(
    RESULT_DIR /
    "final_internal_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

# =============================================================================
# 13. COPY MODEL TO EXPORT DIRECTORY
# =============================================================================

LOCAL_EXPORT = (
    EXPORT_DIR /
    "Model3B_CT_Classifier_Final.keras"
)

shutil.copy2(
    MODEL_PATH,
    LOCAL_EXPORT
)

print("\n" + "=" * 80)
print("MODEL EXPORT")
print("=" * 80)

print(
    "\nExported model:"
)

print(
    LOCAL_EXPORT
)

print(
    f"\nModel size: "
    f"{LOCAL_EXPORT.stat().st_size / (1024**2):.2f} MB"
)

# =============================================================================
# 14. COMPLETE
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 3B - STEP 16 COMPLETE")
print("=" * 80)

print(
    f"\nInternal slice accuracy: "
    f"{accuracy * 100:.2f}%"
)

print(
    f"Internal balanced accuracy: "
    f"{balanced_acc * 100:.2f}%"
)

print(
    f"Internal patient accuracy: "
    f"{patient_accuracy * 100:.2f}%"
)

print(
    "\nModel ready for download:"
)

print(
    LOCAL_EXPORT
)

print("=" * 80)

MODEL 3B - STEP 16
FINAL INTERNAL TEST + MODEL EXPORT
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

MODEL CHECK
Model: /content/drive/MyDrive/Model3B/models/model3b_step14_best.keras
✅ Model loaded

Manifest rows: 1860

Test slices: 285
Test patients: 19
✅ No patient leakage
Missing test files: 0

GENERATING INTERNAL TEST PREDICTIONS
[50/285]
[100/285]
[150/285]
[200/285]
[250/285]
[285/285]

FINAL INTERNAL TEST RESULTS

Slice Accuracy: 82.46%
Balanced Accuracy: 65.43%

Classification report:
                  precision    recall  f1-score   support

      Meningioma       0.00      0.00      0.00        45
       Pituitary       0.74      0.96      0.84       135
Brain_Metastasis       0.99      1.00      1.00       105

        accuracy                           0.82       285
       macro avg       0.58      0.65      0.61       285
    weighted avg       0.72      0.82      0.76       285


Confusio

In [7]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/Model3B/exports/Model3B_CT_Classifier_Final.keras"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>